In [5]:
# CELL 0 - GPU VERIFICATION FOR SELECTED MODELS
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("GPU VERIFICATION - XGBoost, LightGBM, RandomForest, TabNet")
print("="*60)

# Create test data
np.random.seed(42)
X_test = np.random.rand(100, 5)
y_test = np.random.randint(0, 2, 100)

print(f"Test data: {X_test.shape[0]} samples, {X_test.shape[1]} features")

# Test XGBoost GPU
print("\n" + "-"*40)
print("XGBoost GPU Test")
print("-"*40)
try:
    xgb_gpu = xgb.XGBClassifier(tree_method='hist', device='cuda:0', n_estimators=10, verbosity=0)
    xgb_gpu.fit(X_test, y_test)
    print("✅ XGBoost: GPU WORKING")
    xgb_gpu_ready = True
except Exception as e:
    print(f"⚠️ XGBoost: CPU fallback - {str(e)[:50]}")
    xgb_gpu_ready = False

# Test LightGBM GPU
print("\n" + "-"*40)
print("LightGBM GPU Test")
print("-"*40)
try:
    train_data = lgb.Dataset(X_test, label=y_test)
    params = {'objective': 'binary', 'device': 'gpu', 'verbose': -1}
    lgb_gpu = lgb.train(params, train_data, num_boost_round=10)
    print("✅ LightGBM: GPU WORKING")
    lgb_gpu_ready = True
except Exception as e:
    print(f"⚠️ LightGBM: CPU fallback - {str(e)[:50]}")
    lgb_gpu_ready = False

# Test Random Forest (CPU only)
print("\n" + "-"*40)
print("Random Forest Test")
print("-"*40)
try:
    rf = RandomForestClassifier(n_estimators=10, random_state=42)
    rf.fit(X_test, y_test)
    print("✅ RandomForest: READY (CPU only)")
    rf_ready = True
except Exception as e:
    print(f"❌ RandomForest: Failed - {str(e)[:50]}")
    rf_ready = False

# Test TabNet (if available)
print("\n" + "-"*40)
print("TabNet Test")
print("-"*40)
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    tabnet = TabNetClassifier(n_d=8, n_a=8, n_steps=3, verbose=0)
    tabnet.fit(X_test, y_test, max_epochs=5, patience=3)
    print("✅ TabNet: READY (GPU if available)")
    tabnet_ready = True
except ImportError:
    print("⚠️ TabNet: Not installed - skipping")
    tabnet_ready = False
except Exception as e:
    print(f"⚠️ TabNet: Error - {str(e)[:50]}")
    tabnet_ready = False

# Final Summary
print("\n" + "="*60)
print("FINAL STATUS")
print("="*60)
print(f"XGBoost:      {'✅ GPU' if xgb_gpu_ready else '⚠️ CPU'}")
print(f"LightGBM:     {'✅ GPU' if lgb_gpu_ready else '⚠️ CPU'}")
print(f"RandomForest: ✅ CPU")
print(f"TabNet:       {'✅ READY' if tabnet_ready else '⚠️ SKIPPED'}")
print(f"VotingEnsemble: Will combine all available models")

ready_count = sum([xgb_gpu_ready, lgb_gpu_ready, rf_ready, tabnet_ready])
print(f"\nModels ready: {ready_count}/4")

print("\n✅ Ready for thesis pipeline")

GPU VERIFICATION - XGBoost, LightGBM, RandomForest, TabNet
Test data: 100 samples, 5 features

----------------------------------------
XGBoost GPU Test
----------------------------------------
✅ XGBoost: GPU WORKING

----------------------------------------
LightGBM GPU Test
----------------------------------------
✅ LightGBM: GPU WORKING

----------------------------------------
Random Forest Test
----------------------------------------
✅ RandomForest: READY (CPU only)

----------------------------------------
TabNet Test
----------------------------------------
✅ TabNet: READY (GPU if available)

FINAL STATUS
XGBoost:      ✅ GPU
LightGBM:     ✅ GPU
RandomForest: ✅ CPU
TabNet:       ✅ READY
VotingEnsemble: Will combine all available models

Models ready: 4/4

✅ Ready for thesis pipeline


In [6]:
import pandas as pd
import os

# Data path
DATA_PATH = r"D:\Thesis\Data"

# List of files to examine (keep split_chrono, remove split_random)
files = [
    'meds.csv', 
    'pmh.csv',
    'split_chrono_test.csv',
    'split_chrono_train.csv',
    'split_chrono_val.csv',
    'visits.csv'    
]

print("="*80)
print("DATA FILE PREVIEW - FIRST 30 ROWS, ALL COLUMNS")
print("="*80)

for file in files:
    file_path = os.path.join(DATA_PATH, file)
    
    if os.path.exists(file_path):
        print(f"\n{'='*80}")
        print(f"FILE: {file}")
        print(f"{'='*80}")
        
        # Load CSV file
        df = pd.read_csv(file_path, nrows=30)
        
        print(f"\nShape: {df.shape}")
        print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")
        print(f"\nFirst 30 rows (ALL COLUMNS):")
        
        # Print all columns, all 30 rows
        pd.set_option('display.max_columns', None)
        pd.set_option('display.max_rows', 30)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)
        
        print(df)
        
        print(f"\nData types:")
        print(df.dtypes)
        
        print(f"\nBasic statistics for numeric columns:")
        print(df.describe())
        
        print(f"\nMissing values count:")
        print(df.isnull().sum())
        
    else:
        print(f"\n{'='*80}")
        print(f"FILE NOT FOUND: {file}")
        print(f"{'='*80}")

print("\n" + "="*80)
print("PREVIEW COMPLETE")
print("="*80)

DATA FILE PREVIEW - FIRST 30 ROWS, ALL COLUMNS

FILE: meds.csv

Shape: (30, 11)

Columns (11): ['MRN', 'Med_ID', 'NDC', 'Name', 'Generic_name', 'Med_class', 'Med_subclass', 'Active', 'Entry_date', 'Start_date', 'End_date']

First 30 rows (ALL COLUMNS):
         MRN  Med_ID           NDC  \
0   98816154    3774  63739-486-10   
1   98816154    4572  63739-499-10   
2   98816154    8901           NaN   
3   98816154  116051  65862-662-30   
4   98816154  198710  0904-6935-61   
5   98816154  222671           NaN   
6   98816154  236624           NaN   
7   98816163     324  51079-788-01   
8   98816163   10434  63323-290-30   
9   98816163   70536  49884-398-77   
10  98816163   80840  0173-0447-02   
11  98816163  168488           NaN   
12  98816163  198857  0093-7684-32   
13  98816201   78035  50419-423-01   
14  98816201  226144           NaN   
15  98816238     662           NaN   
16  98816238    6494  0054-0017-25   
17  98816238    7172           NaN   
18  98816238    7857    0

In [7]:
# CELL 1 - LOAD ALL MC-MED DATASET COMPONENTS
import os
import pandas as pd
import logging
import numpy as np

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Configure paths
DATA_PATH = r"D:\Thesis\Data"
OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
os.makedirs(OUTPUT_PATH, exist_ok=True)

logger.info("="*80)
logger.info("📥 LOADING ALL MC-MED DATASET COMPONENTS")
logger.info("="*80)

# =============================================================================
# 1. LOAD CHRONOLOGICAL SPLITS (CSN lists)
# =============================================================================
split_files = {
    'train': 'split_chrono_train.csv',
    'val': 'split_chrono_val.csv',
    'test': 'split_chrono_test.csv'
}

splits = {}
for split_name, filename in split_files.items():
    filepath = os.path.join(DATA_PATH, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Split file missing: {filepath}")
    
    # CSNs are in first column without header
    csns = pd.read_csv(filepath, header=None)[0].astype(str).tolist()
    splits[split_name] = set(csns)
    logger.info(f"✅ Loaded {split_name} split: {len(csns):,} unique CSNs")

# =============================================================================
# 2. LOAD VISITS DATA (ALL columns)
# =============================================================================
logger.info("\n📋 Loading visits.csv (ALL columns)...")
visits_df = pd.read_csv(
    os.path.join(DATA_PATH, 'visits.csv'),
    dtype={'MRN': str, 'CSN': str},
    low_memory=False
)
logger.info(f"✅ Loaded visits: {len(visits_df):,} records, {len(visits_df.columns)} columns")

# Filter to adult population (≥18 years)
if 'Age' in visits_df.columns:
    original_count = len(visits_df)
    visits_df = visits_df[visits_df['Age'] >= 18].copy()
    logger.info(f"✅ Filtered to adult population: {len(visits_df):,} records (removed {original_count - len(visits_df):,} pediatric)")

# =============================================================================
# 3. LOAD LABS DATA (ALL rows)
# =============================================================================
logger.info("\n🧪 Loading labs.csv (ALL laboratory results)...")
labs_df = pd.read_csv(
    os.path.join(DATA_PATH, 'labs.csv'),
    dtype={'CSN': str},
    low_memory=False
)
logger.info(f"✅ Loaded labs: {len(labs_df):,} lab component records")

# =============================================================================
# 4. LOAD MEDS DATA (ALL rows)
# =============================================================================
logger.info("\n💊 Loading meds.csv (ALL medication records)...")
meds_df = pd.read_csv(
    os.path.join(DATA_PATH, 'meds.csv'),
    dtype={'MRN': str},
    low_memory=False
)
logger.info(f"✅ Loaded meds: {len(meds_df):,} medication records")

# =============================================================================
# 5. LOAD NUMERICS DATA (vital signs - ALL rows)
# =============================================================================
logger.info("\n📊 Loading numerics.csv (ALL vital signs and measurements)...")
numerics_df = pd.read_csv(
    os.path.join(DATA_PATH, 'numerics.csv'),
    dtype={'CSN': str},
    low_memory=False
)
logger.info(f"✅ Loaded numerics: {len(numerics_df):,} vital sign records")

# =============================================================================
# 6. LOAD ORDERS DATA (ALL rows)
# =============================================================================
logger.info("\n📝 Loading orders.csv (ALL orders placed)...")
orders_df = pd.read_csv(
    os.path.join(DATA_PATH, 'orders.csv'),
    dtype={'CSN': str},
    low_memory=False
)
logger.info(f"✅ Loaded orders: {len(orders_df):,} order records")

# =============================================================================
# 7. LOAD PMH DATA (past medical history - ALL rows)
# =============================================================================
logger.info("\n🏥 Loading pmh.csv (ALL past medical history records)...")
pmh_df = pd.read_csv(
    os.path.join(DATA_PATH, 'pmh.csv'),
    dtype={'MRN': str},
    low_memory=False
)
logger.info(f"✅ Loaded pmh: {len(pmh_df):,} diagnosis records")

# =============================================================================
# 8. LOAD RADS DATA (radiology - ALL rows)
# =============================================================================
logger.info("\n🩻 Loading rads.csv (ALL radiology records)...")
rads_df = pd.read_csv(
    os.path.join(DATA_PATH, 'rads.csv'),
    dtype={'CSN': str},
    low_memory=False
)
logger.info(f"✅ Loaded rads: {len(rads_df):,} radiology records")

# =============================================================================
# 9. LOAD WAVEFORM SUMMARY DATA (ALL rows)
# =============================================================================
logger.info("\n📈 Loading waveform_summary.csv (ALL waveform records)...")
waveform_df = pd.read_csv(
    os.path.join(DATA_PATH, 'waveform_summary.csv'),
    dtype={'CSN': str},
    low_memory=False
)
logger.info(f"✅ Loaded waveform: {len(waveform_df):,} waveform records")

# =============================================================================
# 10. APPLY CHRONOLOGICAL SPLITS TO ALL DATA
# =============================================================================
logger.info("\n" + "="*80)
logger.info("📊 APPLYING CHRONOLOGICAL SPLITS TO ALL DATA")
logger.info("="*80)

# Initialize dictionaries for each data type
datasets = {
    'train': {},
    'val': {},
    'test': {}
}

# Split visits data
for split_name, csn_set in splits.items():
    datasets[split_name]['visits'] = visits_df[visits_df['CSN'].astype(str).isin(csn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} visits: {len(datasets[split_name]['visits']):,} records")

# Split labs data
for split_name, csn_set in splits.items():
    datasets[split_name]['labs'] = labs_df[labs_df['CSN'].astype(str).isin(csn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} labs: {len(datasets[split_name]['labs']):,} records")

# Split numerics data
for split_name, csn_set in splits.items():
    datasets[split_name]['numerics'] = numerics_df[numerics_df['CSN'].astype(str).isin(csn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} numerics: {len(datasets[split_name]['numerics']):,} records")

# Split orders data
for split_name, csn_set in splits.items():
    datasets[split_name]['orders'] = orders_df[orders_df['CSN'].astype(str).isin(csn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} orders: {len(datasets[split_name]['orders']):,} records")

# Split rads data
for split_name, csn_set in splits.items():
    datasets[split_name]['rads'] = rads_df[rads_df['CSN'].astype(str).isin(csn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} rads: {len(datasets[split_name]['rads']):,} records")

# Split waveform data
for split_name, csn_set in splits.items():
    datasets[split_name]['waveform'] = waveform_df[waveform_df['CSN'].astype(str).isin(csn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} waveform: {len(datasets[split_name]['waveform']):,} records")

# For meds and pmh, split by MRN (patient-level, not visit-level)
# First get MRN for each split from visits
split_mrns = {}
for split_name in splits.keys():
    mrns = datasets[split_name]['visits']['MRN'].dropna().unique()
    split_mrns[split_name] = set(mrns.astype(str))
    logger.info(f"✅ {split_name.capitalize()} unique MRNs: {len(split_mrns[split_name]):,}")

# Split meds data by MRN
for split_name, mrn_set in split_mrns.items():
    datasets[split_name]['meds'] = meds_df[meds_df['MRN'].astype(str).isin(mrn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} meds: {len(datasets[split_name]['meds']):,} records")

# Split pmh data by MRN
for split_name, mrn_set in split_mrns.items():
    datasets[split_name]['pmh'] = pmh_df[pmh_df['MRN'].astype(str).isin(mrn_set)].copy()
    logger.info(f"✅ {split_name.capitalize()} pmh: {len(datasets[split_name]['pmh']):,} records")

# =============================================================================
# 11. SAVE ALL SPLIT DATA FOR REUSE
# =============================================================================
logger.info("\n" + "="*80)
logger.info("💾 SAVING ALL SPLIT DATA")
logger.info("="*80)

for split_name in splits.keys():
    split_output_path = os.path.join(OUTPUT_PATH, f'{split_name}_data')
    os.makedirs(split_output_path, exist_ok=True)
    
    # Save each data type
    for data_type in ['visits', 'labs', 'numerics', 'orders', 'rads', 'waveform', 'meds', 'pmh']:
        if data_type in datasets[split_name]:
            filepath = os.path.join(split_output_path, f'{data_type}.pkl')
            datasets[split_name][data_type].to_pickle(filepath)
    
    logger.info(f"✅ Saved {split_name} data to: {split_output_path}")

# =============================================================================
# 12. PRINT FINAL SUMMARY
# =============================================================================
logger.info("\n" + "="*80)
logger.info("✅ ALL DATA LOADING COMPLETE")
logger.info("="*80)
logger.info(f"\n📊 DATA SIZE SUMMARY:")
logger.info(f"   Total visits: {len(visits_df):,}")
logger.info(f"   Total labs: {len(labs_df):,}")
logger.info(f"   Total meds: {len(meds_df):,}")
logger.info(f"   Total numerics: {len(numerics_df):,}")
logger.info(f"   Total orders: {len(orders_df):,}")
logger.info(f"   Total pmh: {len(pmh_df):,}")
logger.info(f"   Total rads: {len(rads_df):,}")
logger.info(f"   Total waveform: {len(waveform_df):,}")

logger.info(f"\n📊 SPLIT SUMMARY:")
for split_name in splits.keys():
    logger.info(f"   {split_name.capitalize()}:")
    logger.info(f"      Visits: {len(datasets[split_name]['visits']):,}")
    if 'labs' in datasets[split_name]:
        logger.info(f"      Labs: {len(datasets[split_name]['labs']):,}")
    if 'meds' in datasets[split_name]:
        logger.info(f"      Meds: {len(datasets[split_name]['meds']):,}")
    if 'orders' in datasets[split_name]:
        logger.info(f"      Orders: {len(datasets[split_name]['orders']):,}")

logger.info("\n" + "="*80)
logger.info("⚠️  IMPORTANT NOTES:")
logger.info("   - All data loaded using CHRONOLOGICAL splits (temporal integrity)")
logger.info(f"   - Data saved to: {OUTPUT_PATH}")
logger.info("   - For Hosp-LOS: Use visits ONLY (same 29 admission-available features as ED-LOS/Admission)")
logger.info("   - For ED-LOS/Admission: Use visits ONLY (admission-available features)")
logger.info("="*80)

# Return datasets for use in subsequent cells
datasets, splits

2026-07-24 14:17:41 | INFO | ================================================================================
2026-07-24 14:17:41 | INFO | 📥 LOADING ALL MC-MED DATASET COMPONENTS
2026-07-24 14:17:41 | INFO | ================================================================================
2026-07-24 14:17:41 | INFO | ✅ Loaded train split: 81,588 unique CSNs
2026-07-24 14:17:41 | INFO | ✅ Loaded val split: 11,770 unique CSNs
2026-07-24 14:17:41 | INFO | ✅ Loaded test split: 12,020 unique CSNs
2026-07-24 14:17:41 | INFO | 
📋 Loading visits.csv (ALL columns)...
2026-07-24 14:17:43 | INFO | ✅ Loaded visits: 118,385 records, 33 columns
2026-07-24 14:17:43 | INFO | ✅ Filtered to adult population: 118,385 records (removed 0 pediatric)
2026-07-24 14:17:43 | INFO | 
🧪 Loading labs.csv (ALL laboratory results)...
2026-07-24 14:17:55 | INFO | ✅ Loaded labs: 5,706,470 lab component records
2026-07-24 14:17:55 | INFO | 
💊 Loading meds.csv (ALL medication records)...
2026-07-24 14:17:56 | INFO | ✅ Lo

({'train': {'visits':              MRN       CSN  Visit_no  Visits  Age Gender  \
   0       98880961  98874959         1       1   82      F   
   1       99121566  99354408         1       2   59      F   
   2       99121566  99121037         2       2   63      F   
   3       99608739  99327434         1       2   90      F   
   4       99608739  99502864         2       2   89      F   
   ...          ...       ...       ...     ...  ...    ...   
   118380  99648524  99789968         3       5   41      F   
   118381  99312340  99391376         1       1   37      M   
   118382  99828357  99836129         1       1   22      M   
   118383  98964925  98979879         1       1   24      M   
   118384  99189786  99274174         1       1   41      M   
   
                                Race                Ethnicity Means_of_arrival  \
   0                           White  Non-Hispanic/Non-Latino              EMS   
   1                           Other  Non-Hispanic/Non-La

In [8]:
# CELL 2 - FEATURE ENGINEERING: 29 ADMISSION-AVAILABLE CLINICAL PREDICTORS
# WITH PUBLICATION-READY FEATURE NAMES (EXACTLY AS IN TABLE 2)

import pandas as pd
import numpy as np
import logging
import os

logger = logging.getLogger(__name__)

# Set output path to paper output directory
OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load raw split data from the subfolders where CELL 1 saved them
train_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'train_data', 'visits.pkl'))
val_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'val_data', 'visits.pkl'))
test_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'test_data', 'visits.pkl'))

# Load meds and pmh from original data path
meds_df = pd.read_csv(os.path.join(r"D:\Thesis\Data", 'meds.csv'), dtype={'MRN': str})
pmh_df = pd.read_csv(os.path.join(r"D:\Thesis\Data", 'pmh.csv'), dtype={'MRN': str})

logger.info("="*80)
logger.info("🔬 FEATURE ENGINEERING: 29 Admission-Available Clinical Predictors")
logger.info("="*80)
logger.info(f"Train visits: {len(train_df):,}")
logger.info(f"Val visits: {len(val_df):,}")
logger.info(f"Test visits: {len(test_df):,}")

# ============================================================================
# 1. TEMPORAL FEATURES (Arrival Context)
# ============================================================================
def extract_temporal_features(df):
    """Derive arrival context features available at triage moment"""
    df = df.copy()
    df['Arrival_time'] = pd.to_datetime(df['Arrival_time'], errors='coerce')
    
    # Convert to timezone-naive if timezone-aware
    if df['Arrival_time'].dt.tz is not None:
        df['Arrival_time'] = df['Arrival_time'].dt.tz_localize(None)
    
    # Handle missing arrival times
    df['Arrival_time'] = df['Arrival_time'].fillna(pd.Timestamp('2024-01-01 12:00:00'))
    df['Arrival_time'] = pd.to_datetime(df['Arrival_time'])
    
    # EXACT NAME: Arrival Hour
    df['Arrival Hour'] = df['Arrival_time'].dt.hour.astype('int8')
    
    # EXACT NAME: Weekend Arrival
    df['Weekend Arrival'] = (df['Arrival_time'].dt.dayofweek >= 5).astype('int8')
    
    # EXACT NAME: EMS Arrival
    df['EMS Arrival'] = (df['Means_of_arrival'] == 'EMS').astype('int8')
    
    return df

train_df = extract_temporal_features(train_df)
val_df = extract_temporal_features(val_df)
test_df = extract_temporal_features(test_df)
logger.info("✅ Temporal features derived: Arrival Hour, Weekend Arrival, EMS Arrival")

# ============================================================================
# 2. DEMOGRAPHIC FEATURES
# ============================================================================
def encode_demographics(df):
    """Encode demographic variables with clinical validity"""
    df = df.copy()
    
    # EXACT NAME: Age
    df['Age'] = df['Age'].astype('float32')
    
    # EXACT NAME: Sex (Male)
    df['Sex (Male)'] = (df['Gender'] == 'M').astype('int8')
    
    # EXACT NAME: Race (White)
    df['Race (White)'] = (df['Race'] == 'White').astype('int8')
    
    # EXACT NAME: Race (Black)
    df['Race (Black)'] = (df['Race'] == 'Black or African American').astype('int8')
    
    # EXACT NAME: Hispanic Ethnicity
    df['Hispanic Ethnicity'] = (df['Ethnicity'] == 'Hispanic/Latino').astype('int8')
    
    # EXACT NAME: Medicare Insurance
    df['Medicare Insurance'] = (df['Payor_class'] == 'Medicare').astype('int8')
    
    # EXACT NAME: Medicaid Insurance
    df['Medicaid Insurance'] = (df['Payor_class'] == 'Medicaid').astype('int8')
    
    return df

train_df = encode_demographics(train_df)
val_df = encode_demographics(val_df)
test_df = encode_demographics(test_df)
logger.info("✅ Demographic features encoded")

# ============================================================================
# 3. TRIAGE VITAL SIGNS (EXACT NAMES FROM TABLE 2)
# ============================================================================
# EXACT NAMES: Triage Temperature, Triage Heart Rate, Triage Respiratory Rate
#              Triage SpO2, Triage Systolic BP, Triage Diastolic BP
VITAL_COLS = ['Triage_Temp', 'Triage_HR', 'Triage_RR', 'Triage_SpO2', 'Triage_SBP', 'Triage_DBP']
for col in VITAL_COLS:
    if col in train_df.columns:
        train_df[col] = train_df[col].astype('float32')
        val_df[col] = val_df[col].astype('float32')
        test_df[col] = test_df[col].astype('float32')

# Rename to publication names
train_df = train_df.rename(columns={
    'Triage_Temp': 'Triage Temperature',
    'Triage_HR': 'Triage Heart Rate',
    'Triage_RR': 'Triage Respiratory Rate',
    'Triage_SpO2': 'Triage SpO2',
    'Triage_SBP': 'Triage Systolic BP',
    'Triage_DBP': 'Triage Diastolic BP'
})
val_df = val_df.rename(columns={
    'Triage_Temp': 'Triage Temperature',
    'Triage_HR': 'Triage Heart Rate',
    'Triage_RR': 'Triage Respiratory Rate',
    'Triage_SpO2': 'Triage SpO2',
    'Triage_SBP': 'Triage Systolic BP',
    'Triage_DBP': 'Triage Diastolic BP'
})
test_df = test_df.rename(columns={
    'Triage_Temp': 'Triage Temperature',
    'Triage_HR': 'Triage Heart Rate',
    'Triage_RR': 'Triage Respiratory Rate',
    'Triage_SpO2': 'Triage SpO2',
    'Triage_SBP': 'Triage Systolic BP',
    'Triage_DBP': 'Triage Diastolic BP'
})
logger.info("✅ Triage vital signs renamed to publication format")

# ============================================================================
# 4. TRIAGE ACUITY LEVEL
# ============================================================================
def extract_acuity_level(acuity_str):
    if pd.isna(acuity_str) or acuity_str == '':
        return np.nan
    return int(str(acuity_str)[0])

# EXACT NAME: Triage Acuity Level
train_df['Triage Acuity Level'] = train_df['Triage_acuity'].apply(extract_acuity_level).astype('float32')
val_df['Triage Acuity Level'] = val_df['Triage_acuity'].apply(extract_acuity_level).astype('float32')
test_df['Triage Acuity Level'] = test_df['Triage_acuity'].apply(extract_acuity_level).astype('float32')

# Impute with mode from train set
triage_mode = train_df['Triage Acuity Level'].mode()[0]
train_df['Triage Acuity Level'] = train_df['Triage Acuity Level'].fillna(triage_mode).astype('int8')
val_df['Triage Acuity Level'] = val_df['Triage Acuity Level'].fillna(triage_mode).astype('int8')
test_df['Triage Acuity Level'] = test_df['Triage Acuity Level'].fillna(triage_mode).astype('int8')
logger.info("✅ Triage acuity encoded")

# ============================================================================
# 5. CHIEF COMPLAINT FEATURES (EXACT NAMES FROM TABLE 2)
# ============================================================================
def extract_cc_features(df):
    df = df.copy()
    
    # EXACT NAME: Complaint Count
    df['Complaint Count'] = df['CC'].fillna('').apply(
        lambda x: len([c for c in str(x).split(',') if c.strip() != '']) if pd.notna(x) else 0
    ).astype('int8')
    
    # EXACT NAME: CC: Abdominal Pain
    df['CC: Abdominal Pain'] = df['CC'].str.contains('ABDOMINAL PAIN', case=False, na=False).astype('int8')
    
    # EXACT NAME: CC: Chest Pain
    df['CC: Chest Pain'] = df['CC'].str.contains('CHEST PAIN', case=False, na=False).astype('int8')
    
    # EXACT NAME: CC: Shortness of Breath
    df['CC: Shortness of Breath'] = df['CC'].str.contains('SHORTNESS OF BREATH', case=False, na=False).astype('int8')
    
    # EXACT NAME: CC: Fall (no space after colon per table)
    df['CC:Fall'] = df['CC'].str.contains('FALL', case=False, na=False).astype('int8')
    
    # EXACT NAME: CC: Fever
    df['CC: Fever'] = df['CC'].str.contains('FEVER', case=False, na=False).astype('int8')
    
    # EXACT NAME: CC: Critical (no space after colon per table)
    critical_terms = ['SUICID', 'OVERDOSE', 'TRAUMA', 'BLEEDING', 'SEIZURE', 'STROKE', 'CHEST PAIN']
    df['CC:Critical'] = df['CC'].fillna('').apply(
        lambda x: int(any(term in str(x).upper() for term in critical_terms)) if pd.notna(x) else 0
    ).astype('int8')
    
    return df

train_df = extract_cc_features(train_df)
val_df = extract_cc_features(val_df)
test_df = extract_cc_features(test_df)
logger.info("✅ Chief complaint features derived")

# ============================================================================
# 6. DISEASE CONDITION COUNT (from PMH)
# ============================================================================
# EXACT NAME: Disease Condition Count
pmh_ccs = pmh_df.groupby('MRN')['CCS'].nunique().reset_index(name='Disease Condition Count')
train_df = train_df.merge(pmh_ccs, on='MRN', how='left').fillna({'Disease Condition Count': 0})
val_df = val_df.merge(pmh_ccs, on='MRN', how='left').fillna({'Disease Condition Count': 0})
test_df = test_df.merge(pmh_ccs, on='MRN', how='left').fillna({'Disease Condition Count': 0})
train_df['Disease Condition Count'] = train_df['Disease Condition Count'].astype('int16')
val_df['Disease Condition Count'] = val_df['Disease Condition Count'].astype('int16')
test_df['Disease Condition Count'] = test_df['Disease Condition Count'].astype('int16')
logger.info("✅ Disease condition count derived")

# ============================================================================
# 7. HOME MEDICATION FEATURES (EXACT NAMES FROM TABLE 2)
# ============================================================================
# EXACT NAME: Home Medication Count
med_counts = meds_df.groupby('MRN').size().reset_index(name='Home Medication Count')

# EXACT NAME: Unique Medication Classes Count
med_class_counts = meds_df.groupby('MRN')['Med_class'].nunique().reset_index(name='Unique Medication Classes Count')

train_df = train_df.merge(med_counts, on='MRN', how='left').fillna({'Home Medication Count': 0})
val_df = val_df.merge(med_counts, on='MRN', how='left').fillna({'Home Medication Count': 0})
test_df = test_df.merge(med_counts, on='MRN', how='left').fillna({'Home Medication Count': 0})

train_df = train_df.merge(med_class_counts, on='MRN', how='left').fillna({'Unique Medication Classes Count': 0})
val_df = val_df.merge(med_class_counts, on='MRN', how='left').fillna({'Unique Medication Classes Count': 0})
test_df = test_df.merge(med_class_counts, on='MRN', how='left').fillna({'Unique Medication Classes Count': 0})

# EXACT NAME: Polypharmacy Flag (5 or more medications)
train_df['Polypharmacy Flag'] = (train_df['Home Medication Count'] >= 5).astype('int8')
val_df['Polypharmacy Flag'] = (val_df['Home Medication Count'] >= 5).astype('int8')
test_df['Polypharmacy Flag'] = (test_df['Home Medication Count'] >= 5).astype('int8')

train_df['Home Medication Count'] = train_df['Home Medication Count'].astype('int16')
val_df['Home Medication Count'] = val_df['Home Medication Count'].astype('int16')
test_df['Home Medication Count'] = test_df['Home Medication Count'].astype('int16')

train_df['Unique Medication Classes Count'] = train_df['Unique Medication Classes Count'].astype('int8')
val_df['Unique Medication Classes Count'] = val_df['Unique Medication Classes Count'].astype('int8')
test_df['Unique Medication Classes Count'] = test_df['Unique Medication Classes Count'].astype('int8')
logger.info("✅ Home medication features derived")

# ============================================================================
# 8. CHARLSON COMORBIDITY INDEX (placeholder - will be populated from ICD codes)
# ============================================================================
# EXACT NAME: Charlson Comorbidity Index
train_df['Charlson Comorbidity Index'] = 0
val_df['Charlson Comorbidity Index'] = 0
test_df['Charlson Comorbidity Index'] = 0

# ============================================================================
# 9. COMPILE FINAL 29-FEATURE SET (EXACT NAMES FROM TABLE 2)
# ============================================================================
FEATURE_COLUMNS = [
    # Demographics (7)
    'Age',
    'Sex (Male)',
    'Race (White)',
    'Race (Black)',
    'Hispanic Ethnicity',
    'Medicare Insurance',
    'Medicaid Insurance',
    
    # Triage Vitals & Acuity (7)
    'Triage Temperature',
    'Triage Heart Rate',
    'Triage Respiratory Rate',
    'Triage SpO2',
    'Triage Systolic BP',
    'Triage Diastolic BP',
    'Triage Acuity Level',
    
    # Arrival Context (3)
    'Arrival Hour',
    'Weekend Arrival',
    'EMS Arrival',
    
    # Chief Complaint (7)
    'Complaint Count',
    'CC: Abdominal Pain',
    'CC: Chest Pain',
    'CC: Shortness of Breath',
    'CC:Fall',
    'CC: Fever',
    'CC:Critical',
    
    # Comorbidity Burden (2)
    'Charlson Comorbidity Index',
    'Disease Condition Count',
    
    # Home Medications (3)
    'Home Medication Count',
    'Unique Medication Classes Count',
    'Polypharmacy Flag'
]

assert len(FEATURE_COLUMNS) == 29, f"Feature count: {len(FEATURE_COLUMNS)}"

# Create feature matrices with publication-ready names
X_train = train_df[FEATURE_COLUMNS].copy()
X_val = val_df[FEATURE_COLUMNS].copy()
X_test = test_df[FEATURE_COLUMNS].copy()

logger.info("\n" + "="*80)
logger.info("✅ FEATURE ENGINEERING COMPLETE")
logger.info("="*80)
logger.info(f"Final feature count: {len(FEATURE_COLUMNS)}")
logger.info(f"Train: {X_train.shape}")
logger.info(f"Val: {X_val.shape}")
logger.info(f"Test: {X_test.shape}")

# Display feature names for verification (matching Table 2)
logger.info("\n📋 FEATURE NAMES (EXACTLY AS IN TABLE 2):")
for i, feature in enumerate(FEATURE_COLUMNS, 1):
    logger.info(f"   {i:2d}. {feature}")

# Save feature matrices
X_train.to_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val.to_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test.to_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))
logger.info(f"\n💾 Feature matrices saved to {OUTPUT_PATH}")

2026-07-24 14:21:15 | INFO | ================================================================================
2026-07-24 14:21:15 | INFO | 🔬 FEATURE ENGINEERING: 29 Admission-Available Clinical Predictors
2026-07-24 14:21:15 | INFO | ================================================================================
2026-07-24 14:21:15 | INFO | Train visits: 81,588
2026-07-24 14:21:15 | INFO | Val visits: 11,770
2026-07-24 14:21:15 | INFO | Test visits: 12,020
2026-07-24 14:21:17 | INFO | ✅ Temporal features derived: Arrival Hour, Weekend Arrival, EMS Arrival
2026-07-24 14:21:18 | INFO | ✅ Demographic features encoded
2026-07-24 14:21:18 | INFO | ✅ Triage vital signs renamed to publication format
2026-07-24 14:21:18 | INFO | ✅ Triage acuity encoded
2026-07-24 14:21:19 | INFO | ✅ Chief complaint features derived
2026-07-24 14:21:19 | INFO | ✅ Disease condition count derived
2026-07-24 14:21:20 | INFO | ✅ Home medication features derived
2026-07-24 14:21:21 | INFO | 
2026-07-24 14:21:21 | I

In [9]:
# CELL 3 - BINARY CLASSIFICATION TARGETS WITH ISSUES HANDLED
import pandas as pd
import numpy as np
import logging
import os
import json

logger = logging.getLogger(__name__)

# Set output path
OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load raw splits (saved from CELL 1)
train_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'train_data', 'visits.pkl'))
val_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'val_data', 'visits.pkl'))
test_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'test_data', 'visits.pkl'))

logger.info("="*80)
logger.info("🎯 BINARY CLASSIFICATION TARGETS WITH ISSUES HANDLED")
logger.info("="*80)
logger.info(f"Train visits: {len(train_raw):,}")
logger.info(f"Val visits: {len(val_raw):,}")
logger.info(f"Test visits: {len(test_raw):,}")

# ============================================================================
# ISSUE 4 FIX: ED-LOS >8h AS BINARY (Triage prediction - uses admission-available features)
# ============================================================================
ED_LOS_THRESHOLD = 8  # hours

def define_ed_los_binary(df):
    """Binary ED-LOS target: ≥8 hours (prolonged ED stay)"""
    df = df.copy()
    df['TARGET_ED_LOS_over8h'] = (df['ED_LOS'] >= ED_LOS_THRESHOLD).astype('int8')
    return df

train_raw = define_ed_los_binary(train_raw)
val_raw = define_ed_los_binary(val_raw)
test_raw = define_ed_los_binary(test_raw)

# Calculate imbalance ratio for ED-LOS
train_pos = train_raw['TARGET_ED_LOS_over8h'].sum()
train_neg = len(train_raw) - train_pos
ed_los_imbalance_ratio = train_neg / train_pos

logger.info("\n✅ TASK 1: ED-LOS ≥8 hours (Binary - Triage Prediction)")
logger.info(f"   Threshold: ≥{ED_LOS_THRESHOLD} hours = Prolonged ED stay")
logger.info(f"   Train prevalence: {train_raw['TARGET_ED_LOS_over8h'].mean():.1%}")
logger.info(f"   Val prevalence: {val_raw['TARGET_ED_LOS_over8h'].mean():.1%}")
logger.info(f"   Test prevalence: {test_raw['TARGET_ED_LOS_over8h'].mean():.1%}")
logger.info(f"   Imbalance ratio (neg/pos): {ed_los_imbalance_ratio:.2f}")

# Detect distribution shift
ed_los_shift = test_raw['TARGET_ED_LOS_over8h'].mean() - train_raw['TARGET_ED_LOS_over8h'].mean()
if abs(ed_los_shift) > 0.02:
    logger.warning(f"   ⚠️ Distribution shift detected: Test prevalence +{ed_los_shift:.1%} higher than train")

# ============================================================================
# TASK 2: HOSPITAL ADMISSION (Binary)
# ============================================================================
def define_admission_binary(df):
    """Binary admission target: Observation, Inpatient, ICU = Admitted"""
    df = df.copy()
    admitted_categories = ['Observation', 'Inpatient', 'ICU']
    df['TARGET_Admitted'] = df['ED_dispo'].isin(admitted_categories).astype('int8')
    return df

train_raw = define_admission_binary(train_raw)
val_raw = define_admission_binary(val_raw)
test_raw = define_admission_binary(test_raw)

# Calculate imbalance ratio for Admission
train_pos_admit = train_raw['TARGET_Admitted'].sum()
train_neg_admit = len(train_raw) - train_pos_admit
admit_imbalance_ratio = train_neg_admit / train_pos_admit

logger.info("\n✅ TASK 2: Hospital Admission (Binary)")
logger.info(f"   Admitted categories: Observation, Inpatient, ICU")
logger.info(f"   Train prevalence: {train_raw['TARGET_Admitted'].mean():.1%}")
logger.info(f"   Val prevalence: {val_raw['TARGET_Admitted'].mean():.1%}")
logger.info(f"   Test prevalence: {test_raw['TARGET_Admitted'].mean():.1%}")
logger.info(f"   Imbalance ratio (neg/pos): {admit_imbalance_ratio:.2f}")

# ============================================================================
# ISSUE 3 FIX: Hosp-LOS WITH PROPER FILTERING (Admitted patients only)
# ============================================================================
HOSP_LOS_THRESHOLD = 7  # days

def define_hosp_los_binary(df):
    """
    Binary Hosp-LOS target: >7 days = Prolonged hospital stay (POSITIVE class = 1)
    ≤7 days = Short hospital stay (NEGATIVE class = 0)
    Only defined for admitted patients (TARGET_Admitted == 1)
    """
    df = df.copy()
    
    # Initialize with NA (will be filtered out later)
    df['TARGET_Hosp_LOS_over7d'] = pd.NA
    
    # Only calculate for admitted patients
    admitted_mask = df['TARGET_Admitted'] == 1
    
    # >7 days = 1 (prolonged stay), ≤7 days = 0 (short stay)
    df.loc[admitted_mask, 'TARGET_Hosp_LOS_over7d'] = (df.loc[admitted_mask, 'Hosp_LOS'] > HOSP_LOS_THRESHOLD).astype('int8')
    
    return df

train_raw = define_hosp_los_binary(train_raw)
val_raw = define_hosp_los_binary(val_raw)
test_raw = define_hosp_los_binary(test_raw)

# Extract admitted patients only for Hosp-LOS analysis
train_admitted = train_raw[train_raw['TARGET_Admitted'] == 1].copy()
val_admitted = val_raw[val_raw['TARGET_Admitted'] == 1].copy()
test_admitted = test_raw[test_raw['TARGET_Admitted'] == 1].copy()

# Log Hosp_LOS distribution for transparency
logger.info("\n📊 HOSP_LOS DISTRIBUTION (Admitted Patients Only):")
logger.info(f"   Train - Mean LOS: {train_admitted['Hosp_LOS'].mean():.2f} days")
logger.info(f"   Train - Median LOS: {train_admitted['Hosp_LOS'].median():.2f} days")
logger.info(f"   Train - LOS >{HOSP_LOS_THRESHOLD} days: {(train_admitted['Hosp_LOS'] > HOSP_LOS_THRESHOLD).mean():.1%}")
logger.info(f"   Val - LOS >{HOSP_LOS_THRESHOLD} days: {(val_admitted['Hosp_LOS'] > HOSP_LOS_THRESHOLD).mean():.1%}")
logger.info(f"   Test - LOS >{HOSP_LOS_THRESHOLD} days: {(test_admitted['Hosp_LOS'] > HOSP_LOS_THRESHOLD).mean():.1%}")

# Calculate imbalance ratio for Hosp-LOS (on admitted patients only)
hosp_pos = train_admitted['TARGET_Hosp_LOS_over7d'].sum()
hosp_neg = len(train_admitted) - hosp_pos
hosp_imbalance_ratio = hosp_neg / hosp_pos if hosp_pos > 0 else 0

logger.info("\n✅ TASK 3: Hosp-LOS >7 days (Binary - Admitted patients only)")
logger.info(f"   Threshold: >{HOSP_LOS_THRESHOLD} days = Prolonged hospital stay (POSITIVE class)")
logger.info(f"   Train admitted patients: {len(train_admitted):,} (prevalence: {train_admitted['TARGET_Hosp_LOS_over7d'].mean():.1%})")
logger.info(f"   Val admitted patients: {len(val_admitted):,} (prevalence: {val_admitted['TARGET_Hosp_LOS_over7d'].mean():.1%})")
logger.info(f"   Test admitted patients: {len(test_admitted):,} (prevalence: {test_admitted['TARGET_Hosp_LOS_over7d'].mean():.1%})")
logger.info(f"   Imbalance ratio (neg/pos): {hosp_imbalance_ratio:.2f}")

# Detect distribution shift for Hosp-LOS
hosp_shift = test_admitted['TARGET_Hosp_LOS_over7d'].mean() - train_admitted['TARGET_Hosp_LOS_over7d'].mean()
if abs(hosp_shift) > 0.02:
    logger.warning(f"   ⚠️ Distribution shift detected: Test prevalence +{hosp_shift:.1%} higher than train")

# ============================================================================
# ISSUE 5 FIX: Save separate datasets for different feature sets
# ============================================================================

# Save full datasets for later feature extraction
logger.info("\n" + "="*80)
logger.info("📁 SAVING DATASETS FOR DIFFERENT FEATURE SETS")
logger.info("="*80)

# For ED-LOS and Admission (use admission-available features only)
train_admission_available = train_raw.copy()
val_admission_available = val_raw.copy()
test_admission_available = test_raw.copy()

# For Hosp-LOS (will use same 29 admission-available features - NO expanded features)
train_hosp_los = train_admitted.copy()
val_hosp_los = val_admitted.copy()
test_hosp_los = test_admitted.copy()

# Save Hosp-LOS specific datasets (admitted patients only)
train_hosp_los.to_pickle(os.path.join(OUTPUT_PATH, 'train_hosp_los.pkl'))
val_hosp_los.to_pickle(os.path.join(OUTPUT_PATH, 'val_hosp_los.pkl'))
test_hosp_los.to_pickle(os.path.join(OUTPUT_PATH, 'test_hosp_los.pkl'))

logger.info("✅ Saved Hosp-LOS datasets (admitted patients only):")
logger.info(f"   train_hosp_los.pkl: {len(train_hosp_los):,} patients")
logger.info(f"   val_hosp_los.pkl: {len(val_hosp_los):,} patients")
logger.info(f"   test_hosp_los.pkl: {len(test_hosp_los):,} patients")

# ============================================================================
# STRICT LEAKAGE PREVENTION: REMOVE OUTCOME COLUMNS
# ============================================================================
OUTCOME_COLUMNS = [
    'ED_LOS', 'Hosp_LOS', 
    'Admit_time', 'Dispo_time', 'Departure_time', 'Roomed_time',
    'Arrival_time', 
    'ED_dispo', 'DC_dispo', 
    'CSN', 'MRN', 'CC',
    'Age', 'Gender', 'Race', 'Ethnicity', 'Means_of_arrival', 'Payor_class',
    'Triage_Temp', 'Triage_HR', 'Triage_RR', 'Triage_SpO2', 'Triage_SBP', 'Triage_DBP',
    'Triage_acuity', 'Dx_ICD9', 'Dx_ICD10', 'Dx_name', 'Admit_service'
]

# Target columns (binary only)
TARGET_COLUMNS = [
    'TARGET_ED_LOS_over8h',
    'TARGET_Admitted',
    'TARGET_Hosp_LOS_over7d'
]

# Create target-only dataframes for admission-available tasks
y_train_admit = train_raw[TARGET_COLUMNS].copy()
y_val_admit = val_raw[TARGET_COLUMNS].copy()
y_test_admit = test_raw[TARGET_COLUMNS].copy()

# Create target-only dataframes for Hosp-LOS (admitted only, no TARGET_Admitted needed)
HOSP_TARGET_COLUMNS = ['TARGET_Hosp_LOS_over7d']
y_train_hosp = train_hosp_los[HOSP_TARGET_COLUMNS].copy()
y_val_hosp = val_hosp_los[HOSP_TARGET_COLUMNS].copy()
y_test_hosp = test_hosp_los[HOSP_TARGET_COLUMNS].copy()

# Remove outcome columns
for col in OUTCOME_COLUMNS:
    if col in y_train_admit.columns:
        y_train_admit.drop(columns=[col], inplace=True)
    if col in y_val_admit.columns:
        y_val_admit.drop(columns=[col], inplace=True)
    if col in y_test_admit.columns:
        y_test_admit.drop(columns=[col], inplace=True)

logger.info("\n" + "="*80)
logger.info("✅ LEAKAGE PREVENTION ENFORCED")
logger.info("="*80)
logger.info(f"Admission-available targets shape: {y_train_admit.shape}")
logger.info(f"Hosp-LOS targets shape: {y_train_hosp.shape}")

# ============================================================================
# ISSUE 1 FIX: Save class weights for imbalance handling
# ============================================================================
class_weights = {
    'ED_LOS_over8h': {
        'threshold_hours': ED_LOS_THRESHOLD,
        'scale_pos_weight': float(ed_los_imbalance_ratio),
        'positive_class_weight': 1.0,
        'negative_class_weight': float(ed_los_imbalance_ratio),
        'prevalence': float(train_raw['TARGET_ED_LOS_over8h'].mean())
    },
    'Admission': {
        'scale_pos_weight': float(admit_imbalance_ratio),
        'positive_class_weight': 1.0,
        'negative_class_weight': float(admit_imbalance_ratio),
        'prevalence': float(train_raw['TARGET_Admitted'].mean())
    },
    'Hosp_LOS_over7d': {
        'threshold_days': HOSP_LOS_THRESHOLD,
        'positive_class_definition': f'>{HOSP_LOS_THRESHOLD} days (prolonged stay)',
        'negative_class_definition': f'≤{HOSP_LOS_THRESHOLD} days (short stay)',
        'scale_pos_weight': float(hosp_imbalance_ratio),
        'positive_class_weight': 1.0,
        'negative_class_weight': float(hosp_imbalance_ratio),
        'prevalence': float(train_admitted['TARGET_Hosp_LOS_over7d'].mean())
    }
}

with open(os.path.join(OUTPUT_PATH, 'class_weights.json'), 'w') as f:
    json.dump(class_weights, f, indent=2)

logger.info("\n📊 CLASS WEIGHTS FOR IMBALANCE HANDLING:")
logger.info(f"   ED-LOS: scale_pos_weight={ed_los_imbalance_ratio:.2f}")
logger.info(f"   Admission: scale_pos_weight={admit_imbalance_ratio:.2f}")
logger.info(f"   Hosp-LOS: scale_pos_weight={hosp_imbalance_ratio:.2f}")

# ============================================================================
# SAVE TARGETS
# ============================================================================
y_train_admit.to_pickle(os.path.join(OUTPUT_PATH, 'y_train_admit.pkl'))
y_val_admit.to_pickle(os.path.join(OUTPUT_PATH, 'y_val_admit.pkl'))
y_test_admit.to_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

y_train_hosp.to_pickle(os.path.join(OUTPUT_PATH, 'y_train_hosp.pkl'))
y_val_hosp.to_pickle(os.path.join(OUTPUT_PATH, 'y_val_hosp.pkl'))
y_test_hosp.to_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))

# ============================================================================
# SAVE TARGET DISTRIBUTIONS
# ============================================================================
def convert_to_serializable(d):
    if isinstance(d, dict):
        return {str(k): convert_to_serializable(v) for k, v in d.items()}
    elif isinstance(d, (np.integer, np.int8, np.int16, np.int32, np.int64)):
        return int(d)
    elif isinstance(d, (np.floating, np.float32, np.float64)):
        return float(d)
    elif isinstance(d, np.ndarray):
        return d.tolist()
    else:
        return d

target_distributions = {
    'admission_available_tasks': {
        'train': {
            'ED_LOS_over8h': convert_to_serializable(y_train_admit['TARGET_ED_LOS_over8h'].value_counts().sort_index().to_dict()),
            'Admitted': convert_to_serializable(y_train_admit['TARGET_Admitted'].value_counts().sort_index().to_dict()),
            'ED_LOS_over8h_prevalence': float(y_train_admit['TARGET_ED_LOS_over8h'].mean()),
            'Admitted_prevalence': float(y_train_admit['TARGET_Admitted'].mean())
        },
        'val': {
            'ED_LOS_over8h': convert_to_serializable(y_val_admit['TARGET_ED_LOS_over8h'].value_counts().sort_index().to_dict()),
            'Admitted': convert_to_serializable(y_val_admit['TARGET_Admitted'].value_counts().sort_index().to_dict())
        },
        'test': {
            'ED_LOS_over8h': convert_to_serializable(y_test_admit['TARGET_ED_LOS_over8h'].value_counts().sort_index().to_dict()),
            'Admitted': convert_to_serializable(y_test_admit['TARGET_Admitted'].value_counts().sort_index().to_dict())
        }
    },
    'hosp_los_task': {
        'threshold_days': HOSP_LOS_THRESHOLD,
        'positive_class': f'>{HOSP_LOS_THRESHOLD} days (prolonged stay)',
        'negative_class': f'≤{HOSP_LOS_THRESHOLD} days (short stay)',
        'train': {
            'Hosp_LOS_over7d': convert_to_serializable(y_train_hosp['TARGET_Hosp_LOS_over7d'].value_counts().sort_index().to_dict()),
            'prevalence': float(y_train_hosp['TARGET_Hosp_LOS_over7d'].mean()),
            'n_patients': len(y_train_hosp)
        },
        'val': {
            'Hosp_LOS_over7d': convert_to_serializable(y_val_hosp['TARGET_Hosp_LOS_over7d'].value_counts().sort_index().to_dict()),
            'n_patients': len(y_val_hosp)
        },
        'test': {
            'Hosp_LOS_over7d': convert_to_serializable(y_test_hosp['TARGET_Hosp_LOS_over7d'].value_counts().sort_index().to_dict()),
            'n_patients': len(y_test_hosp)
        }
    },
    'class_weights': class_weights,
    'distribution_shifts': {
        'ED_LOS_over8h': float(test_raw['TARGET_ED_LOS_over8h'].mean() - train_raw['TARGET_ED_LOS_over8h'].mean()),
        'Hosp_LOS_over7d': float(test_admitted['TARGET_Hosp_LOS_over7d'].mean() - train_admitted['TARGET_Hosp_LOS_over7d'].mean())
    }
}

with open(os.path.join(OUTPUT_PATH, 'target_distributions.json'), 'w') as f:
    json.dump(target_distributions, f, indent=2)

logger.info(f"\n💾 Targets saved to {OUTPUT_PATH}")
logger.info("   • y_train_admit.pkl, y_val_admit.pkl, y_test_admit.pkl (ED-LOS + Admission)")
logger.info("   • y_train_hosp.pkl, y_val_hosp.pkl, y_test_hosp.pkl (Hosp-LOS only)")
logger.info("   • class_weights.json (for imbalance handling)")
logger.info("   • target_distributions.json (full documentation)")

# ============================================================================
# SUMMARY OF FIXED ISSUES
# ============================================================================
logger.info("\n" + "="*80)
logger.info("✅ ALL ISSUES HANDLED")
logger.info("="*80)
logger.info("ISSUE 1 (Class Imbalance):")
logger.info(f"   → scale_pos_weight calculated for each task")
logger.info(f"   → Saved to class_weights.json")
logger.info("\nISSUE 2 (Distribution Shift):")
logger.info(f"   → ED-LOS shift: {target_distributions['distribution_shifts']['ED_LOS_over8h']:.3f}")
logger.info(f"   → Hosp-LOS shift: {target_distributions['distribution_shifts']['Hosp_LOS_over7d']:.3f}")
logger.info(f"   → Will require isotonic calibration during training")
logger.info("\nISSUE 3 (Hosp-LOS Sample Size):")
logger.info(f"   → Filtered to admitted patients only")
logger.info(f"   → Train: {len(train_hosp_los):,} | Val: {len(val_hosp_los):,} | Test: {len(test_hosp_los):,}")
logger.info("\nISSUE 4 (ED-LOS >8h as Binary):")
logger.info(f"   → Binary target created (not multiclass)")
logger.info(f"   → Prevalence: {train_raw['TARGET_ED_LOS_over8h'].mean():.1%}")
logger.info("\nISSUE 5 (Different Feature Sets):")
logger.info(f"   → Separate datasets saved for Hosp-LOS")
logger.info(f"   → Will use SAME 29 admission-available features as ED-LOS/Admission (NO expanded features)")

logger.info("\n" + "="*80)
logger.info("✅ READY FOR MODEL TRAINING")
logger.info("="*80)
logger.info("Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble")
logger.info("Tasks: (1) ED-LOS ≥8h, (2) Admission, (3) Hosp-LOS >7d")
logger.info("""
Next Steps:
1. CELL 4: Extract admission-available features (29 features for ALL THREE tasks - NO expanded features)
2. CELL 5: Train models with class weights and calibration
""")
logger.info("="*80)

2026-07-24 14:21:22 | INFO | ================================================================================
2026-07-24 14:21:22 | INFO | 🎯 BINARY CLASSIFICATION TARGETS WITH ISSUES HANDLED
2026-07-24 14:21:22 | INFO | ================================================================================
2026-07-24 14:21:22 | INFO | Train visits: 81,588
2026-07-24 14:21:22 | INFO | Val visits: 11,770
2026-07-24 14:21:22 | INFO | Test visits: 12,020
2026-07-24 14:21:22 | INFO | 
✅ TASK 1: ED-LOS ≥8 hours (Binary - Triage Prediction)
2026-07-24 14:21:22 | INFO |    Threshold: ≥8 hours = Prolonged ED stay
2026-07-24 14:21:22 | INFO |    Train prevalence: 19.0%
2026-07-24 14:21:22 | INFO |    Val prevalence: 19.3%
2026-07-24 14:21:22 | INFO |    Test prevalence: 22.3%
2026-07-24 14:21:22 | INFO |    Imbalance ratio (neg/pos): 4.26
2026-07-24 14:21:22 | WARNING |    ⚠️ Distribution shift detected: Test prevalence +3.3% higher than train
2026-07-24 14:21:22 | INFO | 
✅ TASK 2: Hospital Admission 

In [10]:
# CELL 4 - SIMPLER ALTERNATIVE (Using direct filtering with CSN)

import pandas as pd
import numpy as np
import logging
import os
import json

logger = logging.getLogger(__name__)

OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
os.makedirs(OUTPUT_PATH, exist_ok=True)

logger.info("="*80)
logger.info("🔬 CELL 4: HOSP-LOS USING SAME 29 ADMISSION-AVAILABLE FEATURES")
logger.info("="*80)

# ============================================================================
# 1. LOAD DATA
# ============================================================================
# Load Hosp-LOS datasets (admitted patients only)
train_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'train_hosp_los.pkl'))
val_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'val_hosp_los.pkl'))
test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'test_hosp_los.pkl'))

# Load feature matrices (29 features)
X_train_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))

# Load raw data to get CSN mapping
train_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'train_data', 'visits.pkl'))
val_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'val_data', 'visits.pkl'))
test_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'test_data', 'visits.pkl'))

logger.info(f"Train admitted: {len(train_hosp):,}, Features: {X_train_admit.shape}")

# ============================================================================
# 2. CREATE CSN TO FEATURE INDEX MAPPING
# ============================================================================
# Create mapping from CSN to index in feature matrix
train_csn_to_idx = {csn: idx for idx, csn in enumerate(train_raw['CSN'].astype(str))}
val_csn_to_idx = {csn: idx for idx, csn in enumerate(val_raw['CSN'].astype(str))}
test_csn_to_idx = {csn: idx for idx, csn in enumerate(test_raw['CSN'].astype(str))}

# Get indices for admitted patients
train_indices = [train_csn_to_idx[csn] for csn in train_hosp['CSN'].astype(str) if csn in train_csn_to_idx]
val_indices = [val_csn_to_idx[csn] for csn in val_hosp['CSN'].astype(str) if csn in val_csn_to_idx]
test_indices = [test_csn_to_idx[csn] for csn in test_hosp['CSN'].astype(str) if csn in test_csn_to_idx]

logger.info(f"Matched indices - Train: {len(train_indices)}, Val: {len(val_indices)}, Test: {len(test_indices)}")

# ============================================================================
# 3. FILTER FEATURE MATRICES
# ============================================================================
X_train_hosp = X_train_admit.iloc[train_indices].reset_index(drop=True)
X_val_hosp = X_val_admit.iloc[val_indices].reset_index(drop=True)
X_test_hosp = X_test_admit.iloc[test_indices].reset_index(drop=True)

# Load and verify targets
y_train_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_hosp.pkl'))
y_val_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_hosp.pkl'))
y_test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))

assert len(X_train_hosp) == len(y_train_hosp), "Train mismatch!"
assert len(X_val_hosp) == len(y_val_hosp), "Val mismatch!"
assert len(X_test_hosp) == len(y_test_hosp), "Test mismatch!"

logger.info(f"\n✅ Final shapes:")
logger.info(f"   X_train_hosp: {X_train_hosp.shape}")
logger.info(f"   X_val_hosp: {X_val_hosp.shape}")
logger.info(f"   X_test_hosp: {X_test_hosp.shape}")

# ============================================================================
# 4. SAVE
# ============================================================================
X_train_hosp.to_pickle(os.path.join(OUTPUT_PATH, 'X_train_hosp.pkl'))
X_val_hosp.to_pickle(os.path.join(OUTPUT_PATH, 'X_val_hosp.pkl'))
X_test_hosp.to_pickle(os.path.join(OUTPUT_PATH, 'X_test_hosp.pkl'))

logger.info(f"\n💾 Saved Hosp-LOS feature matrices (29 features)")
logger.info("="*80)

2026-07-24 14:21:23 | INFO | ================================================================================
2026-07-24 14:21:23 | INFO | 🔬 CELL 4: HOSP-LOS USING SAME 29 ADMISSION-AVAILABLE FEATURES
2026-07-24 14:21:23 | INFO | ================================================================================
2026-07-24 14:21:23 | INFO | Train admitted: 32,733, Features: (81588, 29)
2026-07-24 14:21:23 | INFO | Matched indices - Train: 32733, Val: 4820, Test: 4765
2026-07-24 14:21:23 | INFO | 
✅ Final shapes:
2026-07-24 14:21:23 | INFO |    X_train_hosp: (32733, 29)
2026-07-24 14:21:23 | INFO |    X_val_hosp: (4820, 29)
2026-07-24 14:21:23 | INFO |    X_test_hosp: (4765, 29)
2026-07-24 14:21:23 | INFO | 
💾 Saved Hosp-LOS feature matrices (29 features)
2026-07-24 14:21:23 | INFO | ================================================================================


In [11]:
# CELL 5 - TASK 1: ED-LOS ≥8 HOURS PREDICTION (COMPLETE - SHAP for ALL Models)
# Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble
# SHAP analysis for ALL 5 models (each gets FULL SHAP plots)
# TabNet uses KernelExplainer approximation
# Voting Ensemble uses surrogate RandomForest + TreeExplainer

import os
import numpy as np
import pandas as pd
import logging
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    f1_score, brier_score_loss, roc_curve, precision_recall_curve,
    matthews_corrcoef
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.utils import resample, class_weight
import xgboost as xgb
import lightgbm as lgb
from pytorch_tabnet.tab_model import TabNetClassifier
import shap
import torch

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
TASK_OUTPUT = os.path.join(OUTPUT_PATH, 'task1_ed_los')
os.makedirs(TASK_OUTPUT, exist_ok=True)

logger.info("="*80)
logger.info("🎯 TASK 1: ED-LOS ≥8 HOURS PREDICTION")
logger.info("Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble")
logger.info("SHAP analysis for ALL 5 models (each gets FULL SHAP plots)")
logger.info("TabNet: Using KernelExplainer approximation")
logger.info("Voting Ensemble: Surrogate RandomForest + TreeExplainer")
logger.info("="*80)

# ============================================================================
# 1. LOAD DATA (29 admission-available features)
# ============================================================================
X_train = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))

y_train_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_admit.pkl'))
y_val_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_admit.pkl'))
y_test_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

# Extract ED-LOS target
y_train = y_train_full['TARGET_ED_LOS_over8h'].values.astype(int)
y_val = y_val_full['TARGET_ED_LOS_over8h'].values.astype(int)
y_test = y_test_full['TARGET_ED_LOS_over8h'].values.astype(int)

# Load class weights
with open(os.path.join(OUTPUT_PATH, 'class_weights.json'), 'r') as f:
    class_weights = json.load(f)

scale_pos_weight = class_weights['ED_LOS_over8h']['scale_pos_weight']
prevalence = class_weights['ED_LOS_over8h']['prevalence']

logger.info("\n📊 DATA LOADED")
logger.info(f"   Train: {X_train.shape}, Prevalence: {y_train.mean():.1%}")
logger.info(f"   Val: {X_val.shape}, Prevalence: {y_val.mean():.1%}")
logger.info(f"   Test: {X_test.shape}, Prevalence: {y_test.mean():.1%}")
logger.info(f"   Scale pos weight: {scale_pos_weight:.2f}")

# ============================================================================
# 2. PREPROCESSING
# ============================================================================
logger.info("\n📈 PREPROCESSING")

# Convert to numeric and handle any remaining non-numeric columns
for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_val[col] = pd.to_numeric(X_val[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Keep as DataFrame for SHAP compatibility
X_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_df = pd.DataFrame(X_val_scaled, columns=X_train.columns)
X_test_df = pd.DataFrame(X_test_scaled, columns=X_train.columns)

feature_names = X_train.columns.tolist()

logger.info(f"   Features: {len(feature_names)}")
logger.info(f"   Train shape: {X_train_scaled.shape}")

# ============================================================================
# 3. BOOTSTRAP CONFIDENCE INTERVALS FUNCTION
# ============================================================================
def bootstrap_metrics(y_true, y_pred_proba, n_bootstrap=1000, ci=95):
    """Calculate bootstrap confidence intervals for metrics"""
    
    np.random.seed(42)
    n_samples = len(y_true)
    
    thresholds = np.arange(0.05, 0.95, 0.05)
    best_youden = 0
    optimal_thresh = 0.5
    for thresh in thresholds:
        y_pred_class = (y_pred_proba >= thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        youden = sens + spec - 1
        if youden > best_youden:
            best_youden = youden
            optimal_thresh = thresh
    
    boot_metrics = {name: [] for name in ['auc_roc', 'auc_pr', 'sensitivity', 
                                           'specificity', 'f1', 'brier', 'mcc']}
    
    for _ in range(n_bootstrap):
        indices = resample(range(n_samples), n_samples=n_samples)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred_proba[indices]
        
        if len(np.unique(y_true_boot)) == 2:
            auc_roc = roc_auc_score(y_true_boot, y_pred_boot)
            auc_pr = average_precision_score(y_true_boot, y_pred_boot)
            brier = brier_score_loss(y_true_boot, y_pred_boot)
            
            y_pred_class = (y_pred_boot >= optimal_thresh).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true_boot, y_pred_class).ravel()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            f1 = f1_score(y_true_boot, y_pred_class) if (tp + fp + fn) > 0 else 0
            mcc = matthews_corrcoef(y_true_boot, y_pred_class)
            
            boot_metrics['auc_roc'].append(auc_roc)
            boot_metrics['auc_pr'].append(auc_pr)
            boot_metrics['sensitivity'].append(sens)
            boot_metrics['specificity'].append(spec)
            boot_metrics['f1'].append(f1)
            boot_metrics['brier'].append(brier)
            boot_metrics['mcc'].append(mcc)
    
    ci_lower = (100 - ci) / 2
    ci_upper = 100 - ci_lower
    
    results = {}
    for name, values in boot_metrics.items():
        if values:
            results[name] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'ci_lower': np.percentile(values, ci_lower),
                'ci_upper': np.percentile(values, ci_upper)
            }
    
    return results, optimal_thresh

# ============================================================================
# 4. SHAP ANALYSIS FOR TREE MODELS (XGBoost, LightGBM, RandomForest) - FIXED
# ============================================================================
def shap_analysis_tree(model, model_name, X_test_data, feature_names, output_folder, n_samples=300):
    """Comprehensive SHAP analysis for tree-based models (XGBoost, LightGBM, RandomForest)
    
    This function handles both single-output models and multi-output models (RandomForest)
    by detecting if shap_values is a list or 3D array and extracting class 1.
    """
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name}...")
        
        n_use = min(n_samples, X_test_data.shape[0])
        X_subset = X_test_data[:n_use]
        
        # Create TreeExplainer
        explainer = shap.TreeExplainer(model)
        shap_values_raw = explainer.shap_values(X_subset)
        expected_value_raw = explainer.expected_value
        
        logger.info(f"      Raw SHAP type: {type(shap_values_raw)}")
        
        # Handle different output formats
        if isinstance(shap_values_raw, list):
            logger.info(f"      SHAP is list with {len(shap_values_raw)} elements")
            # For binary classification, take class 1 (positive class)
            if len(shap_values_raw) == 2:
                shap_values = shap_values_raw[1]
                logger.info(f"      Using class 1 (positive) SHAP values")
            else:
                shap_values = shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0]
        else:
            shap_values = shap_values_raw
        
        # Handle expected value
        if isinstance(expected_value_raw, list):
            if len(expected_value_raw) == 2:
                expected_value = expected_value_raw[1]
                logger.info(f"      Using class 1 expected value: {expected_value:.4f}")
            else:
                expected_value = expected_value_raw[1] if len(expected_value_raw) > 1 else expected_value_raw[0]
        else:
            expected_value = expected_value_raw
        
        # Ensure shap_values is 2D (n_samples, n_features)
        if hasattr(shap_values, 'shape'):
            logger.info(f"      SHAP values shape before processing: {shap_values.shape}")
        
        # If shap_values is 3D (n_samples, n_features, n_classes), squeeze to 2D
        if len(shap_values.shape) == 3:
            logger.info(f"      SHAP is 3D with shape {shap_values.shape}. Taking class 1 (index 1)...")
            shap_values = shap_values[:, :, 1]
            logger.info(f"      New shape: {shap_values.shape}")
        
        # If shap_values is still 3D with different ordering, try alternative
        if len(shap_values.shape) == 3:
            if shap_values.shape[0] == 2:
                shap_values = shap_values[1, :, :]
                logger.info(f"      Alternative: took first dimension index 1, new shape: {shap_values.shape}")
            else:
                shap_values = shap_values[:, :, 1]
                logger.info(f"      Alternative: took last dimension index 1, new shape: {shap_values.shape}")
        
        # Final check: ensure 2D
        if len(shap_values.shape) != 2:
            logger.error(f"      SHAP values has {len(shap_values.shape)} dimensions. Attempting to flatten...")
            if len(shap_values.shape) == 1:
                shap_values = shap_values.reshape(1, -1)
            else:
                shap_values = shap_values.reshape(shap_values.shape[0], -1)
            logger.info(f"      Reshaped to: {shap_values.shape}")
        
        # Calculate feature importance (mean absolute SHAP values per feature)
        shap_importance = np.abs(shap_values).mean(axis=0)
        
        # Ensure shap_importance is 1D
        if len(shap_importance.shape) > 1:
            logger.info(f"      Importance shape is {shap_importance.shape}, flattening...")
            shap_importance = shap_importance.flatten()
        
        # Ensure lengths match
        if len(shap_importance) != len(feature_names):
            logger.warning(f"      Mismatch: importance length {len(shap_importance)} vs features {len(feature_names)}")
            if len(shap_importance) < len(feature_names):
                # Pad with zeros
                shap_importance = np.pad(shap_importance, (0, len(feature_names) - len(shap_importance)))
            else:
                # Truncate
                shap_importance = shap_importance[:len(feature_names)]
        
        # Create DataFrame
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # SHAP PLOT 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # SHAP PLOT 2: Summary Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # SHAP PLOT 3: Waterfall Plot (first test sample)
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # SHAP PLOT 4: Force Plot
        try:
            shap.initjs()
            shap.force_plot(
                expected_value, shap_values[0], X_subset.iloc[0],
                feature_names=feature_names, matplotlib=True, show=False
            )
            plt.title(f'{model_name} - SHAP Force Plot - ED-LOS', fontsize=14)
            plt.tight_layout()
            plt.savefig(os.path.join(model_folder, 'shap_force_plot.png'), dpi=300, bbox_inches='tight')
            plt.close()
            logger.info(f"      ✅ Saved: shap_force_plot.png")
        except Exception as e:
            logger.warning(f"      Force plot skipped: {str(e)[:50]}")
        
        # SHAP PLOT 5: Dependence Plots for Top 5 Features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} - ED-LOS', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        # Log top features
        logger.info(f"\n   📊 {model_name} Top 10 Features (SHAP):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 5. SHAP ANALYSIS FOR TABNET (using KernelExplainer)
# ============================================================================
def shap_analysis_tabnet(model, model_name, X_train_data, X_test_data, feature_names, output_folder, n_samples=200, n_background=100):
    """SHAP analysis for TabNet using KernelExplainer to produce same plots as tree models"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name} (KernelExplainer)...")
        
        n_background = min(n_background, X_train_data.shape[0])
        n_use = min(n_samples, X_test_data.shape[0])
        
        X_background = X_train_data[:n_background]
        X_subset = X_test_data[:n_use]
        
        # Define prediction function for TabNet
        def predict_proba_fn(x):
            return model.predict_proba(x)[:, 1]
        
        # Create KernelExplainer
        logger.info(f"      Creating KernelExplainer with {n_background} background samples...")
        explainer = shap.KernelExplainer(predict_proba_fn, X_background)
        
        # Compute SHAP values
        logger.info(f"      Computing SHAP values for {n_use} samples (this may take a few minutes)...")
        shap_values = explainer.shap_values(X_subset, nsamples=500)
        
        # Calculate feature importance
        shap_importance = np.abs(shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar - KernelExplainer) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm - KernelExplainer) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=explainer.expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0 - KernelExplainer) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (KernelExplainer) - ED-LOS', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features (KernelExplainer):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 6. MODEL-AGNOSTIC SHAP ANALYSIS FOR VOTING ENSEMBLE (Surrogate RandomForest)
# ============================================================================
def shap_analysis_ensemble(model_proba_fn, model_name, X_train_data, X_test_data, feature_names, output_folder, n_samples=200, n_background=100):
    """SHAP analysis for Voting Ensemble using surrogate RandomForest"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name} (surrogate RandomForest)...")
        
        n_train = min(n_background, X_train_data.shape[0])
        n_use = min(n_samples, X_test_data.shape[0])
        
        X_train_subset = X_train_data[:n_train]
        X_subset = X_test_data[:n_use]
        
        # Get ensemble predictions on training data
        logger.info(f"      Generating ensemble predictions on {n_train} training samples...")
        ensemble_train_preds = model_proba_fn(X_train_subset)
        
        # Train surrogate RandomForest
        logger.info(f"      Training surrogate RandomForest Regressor...")
        surrogate = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        surrogate.fit(X_train_subset, ensemble_train_preds)
        
        # Use TreeExplainer on surrogate
        logger.info(f"      Computing SHAP values for {n_use} test samples...")
        explainer = shap.TreeExplainer(surrogate)
        shap_values = explainer.shap_values(X_subset)
        expected_value = explainer.expected_value
        
        # Calculate feature importance
        shap_importance = np.abs(shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar - Surrogate RF) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm - Surrogate RF) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0 - Surrogate RF) - ED-LOS', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (Surrogate RF) - ED-LOS', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features (Surrogate RandomForest):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 7. TRAIN XGBOOST
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 1: XGBoost")
logger.info("="*60)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    device='cuda:0',
    random_state=42,
    verbosity=0,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=False)
xgb_pred = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_ci, xgb_thresh = bootstrap_metrics(y_test, xgb_pred)
logger.info(f"\n📊 XGBoost Results:")
logger.info(f"   AUC-ROC: {xgb_ci['auc_roc']['mean']:.4f} [{xgb_ci['auc_roc']['ci_lower']:.4f}-{xgb_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {xgb_ci['sensitivity']['mean']:.1%}")

xgb_shap = shap_analysis_tree(xgb_model, 'XGBoost', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 8. TRAIN LIGHTGBM
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 2: LightGBM")
logger.info("="*60)

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.01,
    reg_lambda=0.01,
    device='gpu',
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], 
              eval_metric='auc', callbacks=[lgb.early_stopping(50, verbose=False)])

lgb_pred = lgb_model.predict_proba(X_test_scaled)[:, 1]

lgb_ci, lgb_thresh = bootstrap_metrics(y_test, lgb_pred)
logger.info(f"\n📊 LightGBM Results:")
logger.info(f"   AUC-ROC: {lgb_ci['auc_roc']['mean']:.4f} [{lgb_ci['auc_roc']['ci_lower']:.4f}-{lgb_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {lgb_ci['sensitivity']['mean']:.1%}")

lgb_shap = shap_analysis_tree(lgb_model, 'LightGBM', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 9. TRAIN RANDOM FOREST
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 3: Random Forest")
logger.info("="*60)

rf_class_weight = class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train
)
rf_class_weight_dict = {0: rf_class_weight[0], 1: rf_class_weight[1]}

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight=rf_class_weight_dict,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_ci, rf_thresh = bootstrap_metrics(y_test, rf_pred)
logger.info(f"\n📊 Random Forest Results:")
logger.info(f"   AUC-ROC: {rf_ci['auc_roc']['mean']:.4f} [{rf_ci['auc_roc']['ci_lower']:.4f}-{rf_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {rf_ci['sensitivity']['mean']:.1%}")

# SHAP for Random Forest - NOW WORKS with fixed function
rf_shap = shap_analysis_tree(rf_model, 'RandomForest', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 10. TRAIN TABNET
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 4: TabNet")
logger.info("="*60)

tabnet_model = TabNetClassifier(
    n_d=64,
    n_a=64,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='sparsemax',
    verbose=0,
    device_name='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42
)

tabnet_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False,
    weights=np.where(y_train == 1, scale_pos_weight, 1.0)
)

tabnet_pred = tabnet_model.predict_proba(X_test_scaled)[:, 1]

tabnet_ci, tabnet_thresh = bootstrap_metrics(y_test, tabnet_pred)
logger.info(f"\n📊 TabNet Results:")
logger.info(f"   AUC-ROC: {tabnet_ci['auc_roc']['mean']:.4f} [{tabnet_ci['auc_roc']['ci_lower']:.4f}-{tabnet_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {tabnet_ci['sensitivity']['mean']:.1%}")

# SHAP for TabNet - Using KernelExplainer to get FULL SHAP plots
tabnet_shap = shap_analysis_tabnet(tabnet_model, 'TabNet', X_train_df, X_test_df, feature_names, TASK_OUTPUT, n_samples=200)

# ============================================================================
# 11. VOTING ENSEMBLE (4 models: XGB, LGB, RF, TabNet)
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 5: Voting Ensemble (Weighted - 4 Models)")
logger.info("="*60)

# Calculate validation AUCs for weights
xgb_val_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val_scaled)[:, 1])
lgb_val_auc = roc_auc_score(y_val, lgb_model.predict_proba(X_val_scaled)[:, 1])
rf_val_auc = roc_auc_score(y_val, rf_model.predict_proba(X_val_scaled)[:, 1])
tabnet_val_auc = roc_auc_score(y_val, tabnet_model.predict_proba(X_val_scaled)[:, 1])

total_auc = xgb_val_auc + lgb_val_auc + rf_val_auc + tabnet_val_auc
weights = {
    'xgb': xgb_val_auc / total_auc,
    'lgb': lgb_val_auc / total_auc,
    'rf': rf_val_auc / total_auc,
    'tabnet': tabnet_val_auc / total_auc
}

logger.info(f"\n   Validation AUCs:")
logger.info(f"      XGBoost: {xgb_val_auc:.4f}")
logger.info(f"      LightGBM: {lgb_val_auc:.4f}")
logger.info(f"      Random Forest: {rf_val_auc:.4f}")
logger.info(f"      TabNet: {tabnet_val_auc:.4f}")
logger.info(f"\n   Ensemble Weights:")
logger.info(f"      XGB: {weights['xgb']:.3f}, LGB: {weights['lgb']:.3f}, RF: {weights['rf']:.3f}, TabNet: {weights['tabnet']:.3f}")

# Weighted ensemble predictions
ensemble_pred = (
    weights['xgb'] * xgb_pred +
    weights['lgb'] * lgb_pred +
    weights['rf'] * rf_pred +
    weights['tabnet'] * tabnet_pred
)

ensemble_ci, ensemble_thresh = bootstrap_metrics(y_test, ensemble_pred)
logger.info(f"\n📊 Voting Ensemble Results:")
logger.info(f"   AUC-ROC: {ensemble_ci['auc_roc']['mean']:.4f} [{ensemble_ci['auc_roc']['ci_lower']:.4f}-{ensemble_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {ensemble_ci['sensitivity']['mean']:.1%}")

# Define prediction function for ensemble
def ensemble_predict_proba(X):
    """Returns ensemble predictions for a given input matrix"""
    xgb_proba = xgb_model.predict_proba(X)[:, 1]
    lgb_proba = lgb_model.predict_proba(X)[:, 1]
    rf_proba = rf_model.predict_proba(X)[:, 1]
    tabnet_proba = tabnet_model.predict_proba(X)[:, 1]
    
    return (weights['xgb'] * xgb_proba +
            weights['lgb'] * lgb_proba +
            weights['rf'] * rf_proba +
            weights['tabnet'] * tabnet_proba)

# SHAP for Voting Ensemble (using surrogate model)
ensemble_shap = shap_analysis_ensemble(
    ensemble_predict_proba, 'VotingEnsemble', X_train_df, X_test_df, feature_names, TASK_OUTPUT, n_samples=200
)

# ============================================================================
# 12. MODEL COMPARISON
# ============================================================================
logger.info("\n" + "="*80)
logger.info("📊 MODEL COMPARISON SUMMARY")
logger.info("="*80)

comparison_df = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Random Forest', 'TabNet', 'Voting Ensemble'],
    'AUC-ROC': [xgb_ci['auc_roc']['mean'], lgb_ci['auc_roc']['mean'], 
                rf_ci['auc_roc']['mean'], tabnet_ci['auc_roc']['mean'], 
                ensemble_ci['auc_roc']['mean']],
    'AUC-ROC_CI_lower': [xgb_ci['auc_roc']['ci_lower'], lgb_ci['auc_roc']['ci_lower'],
                         rf_ci['auc_roc']['ci_lower'], tabnet_ci['auc_roc']['ci_lower'],
                         ensemble_ci['auc_roc']['ci_lower']],
    'AUC-ROC_CI_upper': [xgb_ci['auc_roc']['ci_upper'], lgb_ci['auc_roc']['ci_upper'],
                         rf_ci['auc_roc']['ci_upper'], tabnet_ci['auc_roc']['ci_upper'],
                         ensemble_ci['auc_roc']['ci_upper']],
    'Sensitivity': [xgb_ci['sensitivity']['mean'], lgb_ci['sensitivity']['mean'],
                    rf_ci['sensitivity']['mean'], tabnet_ci['sensitivity']['mean'],
                    ensemble_ci['sensitivity']['mean']],
    'Specificity': [xgb_ci['specificity']['mean'], lgb_ci['specificity']['mean'],
                    rf_ci['specificity']['mean'], tabnet_ci['specificity']['mean'],
                    ensemble_ci['specificity']['mean']],
    'F1': [xgb_ci['f1']['mean'], lgb_ci['f1']['mean'],
           rf_ci['f1']['mean'], tabnet_ci['f1']['mean'],
           ensemble_ci['f1']['mean']]
})

comparison_df = comparison_df.sort_values('AUC-ROC', ascending=False)
comparison_df.to_csv(os.path.join(TASK_OUTPUT, 'model_comparison.csv'), index=False)

logger.info("\n" + comparison_df[['Model', 'AUC-ROC', 'Sensitivity', 'Specificity', 'F1']].to_string(index=False))

# ============================================================================
# 13. VISUALIZATIONS
# ============================================================================
logger.info("\n📈 Creating Performance Visualizations")

# ROC Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    fpr, tpr, _ = roc_curve(y_test, preds)
    auc = roc_auc_score(y_test, preds)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - ED-LOS ≥8 hours Prediction', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'roc_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: roc_curves.png")

# PR Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    precision, recall, _ = precision_recall_curve(y_test, preds)
    auprc = average_precision_score(y_test, preds)
    plt.plot(recall, precision, label=f'{name} (AUPRC={auprc:.4f})', linewidth=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves - ED-LOS ≥8 hours Prediction', fontsize=14)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'pr_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: pr_curves.png")

# Calibration Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    prob_true, prob_pred = calibration_curve(y_test, preds, n_bins=10)
    ece = np.mean(np.abs(prob_true - prob_pred))
    plt.plot(prob_pred, prob_true, marker='o', label=f'{name} (ECE={ece:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curves - ED-LOS ≥8 hours Prediction', fontsize=14)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'calibration_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: calibration_curves.png")

# ============================================================================
# 14. SAVE ALL MODEL PREDICTIONS
# ============================================================================
predictions_df = pd.DataFrame({
    'XGBoost': xgb_pred,
    'LightGBM': lgb_pred,
    'RandomForest': rf_pred,
    'TabNet': tabnet_pred,
    'VotingEnsemble': ensemble_pred,
    'y_true': y_test
})
predictions_df.to_csv(os.path.join(TASK_OUTPUT, 'all_predictions.csv'), index=False)
logger.info("   ✅ Saved: all_predictions.csv")

# ============================================================================
# 15. FINAL SUMMARY
# ============================================================================
logger.info("\n" + "="*80)
logger.info("✅ TASK 1 COMPLETE")
logger.info("="*80)
logger.info(f"Best Model: {comparison_df.iloc[0]['Model']} (AUC={comparison_df.iloc[0]['AUC-ROC']:.4f})")
logger.info(f"\n📁 SHAP Output Folders (ALL with COMPLETE plots):")
logger.info(f"   {TASK_OUTPUT}/")
logger.info(f"   ├── xgboost_shap/        (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── lightgbm_shap/       (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── randomforest_shap/   (SHAP bar, summary, waterfall, dependence) ✅ FIXED")
logger.info(f"   ├── tabnet_shap/         (SHAP bar, summary, waterfall, dependence - KernelExplainer) ✅ ADDED")
logger.info(f"   ├── votingensemble_shap/ (SHAP bar, summary, waterfall, dependence - Surrogate RF) ✅ FIXED")
logger.info(f"   ├── model_comparison.csv")
logger.info(f"   ├── roc_curves.png")
logger.info(f"   ├── pr_curves.png")
logger.info(f"   ├── calibration_curves.png")
logger.info(f"   └── all_predictions.csv")
logger.info("="*80)

2026-07-24 14:21:56 | INFO | ================================================================================
2026-07-24 14:21:56 | INFO | 🎯 TASK 1: ED-LOS ≥8 HOURS PREDICTION
2026-07-24 14:21:56 | INFO | Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble
2026-07-24 14:21:56 | INFO | SHAP analysis for ALL 5 models (each gets FULL SHAP plots)
2026-07-24 14:21:56 | INFO | TabNet: Using KernelExplainer approximation
2026-07-24 14:21:56 | INFO | Voting Ensemble: Surrogate RandomForest + TreeExplainer
2026-07-24 14:21:56 | INFO | ================================================================================
2026-07-24 14:21:56 | INFO | 
📊 DATA LOADED
2026-07-24 14:21:56 | INFO |    Train: (81588, 29), Prevalence: 19.0%
2026-07-24 14:21:56 | INFO |    Val: (11770, 29), Prevalence: 19.3%
2026-07-24 14:21:56 | INFO |    Test: (12020, 29), Prevalence: 22.3%
2026-07-24 14:21:56 | INFO |    Scale pos weight: 4.26
2026-07-24 14:21:56 | INFO | 
📈 PREPROCESSING
2026-07-24 14:21:56 | I

2026-07-24 14:22:35 | INFO |       ✅ Saved: shap_force_plot.png
2026-07-24 14:22:36 | INFO |       ✅ Saved: shap_dependence_EMS Arrival.png
2026-07-24 14:22:37 | INFO |       ✅ Saved: shap_dependence_Age.png
2026-07-24 14:22:37 | INFO |       ✅ Saved: shap_dependence_Triage Acuity Level.png
2026-07-24 14:22:38 | INFO |       ✅ Saved: shap_dependence_Disease Condition Count.png
2026-07-24 14:22:38 | INFO |       ✅ Saved: shap_dependence_Home Medication Count.png
2026-07-24 14:22:38 | INFO | 
   📊 XGBoost Top 10 Features (SHAP):
2026-07-24 14:22:38 | INFO |       17. EMS Arrival: 0.1975
2026-07-24 14:22:38 | INFO |       1. Age: 0.1431
2026-07-24 14:22:38 | INFO |       14. Triage Acuity Level: 0.1429
2026-07-24 14:22:38 | INFO |       26. Disease Condition Count: 0.1388
2026-07-24 14:22:38 | INFO |       27. Home Medication Count: 0.1342
2026-07-24 14:22:38 | INFO |       9. Triage Heart Rate: 0.1043
2026-07-24 14:22:38 | INFO |       19. CC: Abdominal Pain: 0.0959
2026-07-24 14:22:38 |

2026-07-24 14:23:09 | INFO |       ✅ Saved: shap_force_plot.png
2026-07-24 14:23:09 | INFO |       ✅ Saved: shap_dependence_EMS Arrival.png
2026-07-24 14:23:10 | INFO |       ✅ Saved: shap_dependence_Triage Acuity Level.png
2026-07-24 14:23:11 | INFO |       ✅ Saved: shap_dependence_Disease Condition Count.png
2026-07-24 14:23:11 | INFO |       ✅ Saved: shap_dependence_Age.png
2026-07-24 14:23:12 | INFO |       ✅ Saved: shap_dependence_Home Medication Count.png
2026-07-24 14:23:12 | INFO | 
   📊 LightGBM Top 10 Features (SHAP):
2026-07-24 14:23:12 | INFO |       17. EMS Arrival: 0.0438
2026-07-24 14:23:12 | INFO |       14. Triage Acuity Level: 0.0370
2026-07-24 14:23:12 | INFO |       26. Disease Condition Count: 0.0298
2026-07-24 14:23:12 | INFO |       1. Age: 0.0257
2026-07-24 14:23:12 | INFO |       27. Home Medication Count: 0.0220
2026-07-24 14:23:12 | INFO |       19. CC: Abdominal Pain: 0.0113
2026-07-24 14:23:12 | INFO |       9. Triage Heart Rate: 0.0056
2026-07-24 14:23:12 


Early stopping occurred at epoch 99 with best_epoch = 79 and best_val_auc = 0.66609


2026-07-24 14:54:23 | INFO | 
📊 TabNet Results:
2026-07-24 14:54:24 | INFO |    AUC-ROC: 0.6385 [0.6273-0.6489]
2026-07-24 14:54:24 | INFO |    Sensitivity: 73.3%
2026-07-24 14:54:24 | INFO | 
   🔍 Computing SHAP values for TabNet (KernelExplainer)...
2026-07-24 14:54:24 | INFO |       Creating KernelExplainer with 100 background samples...
2026-07-24 14:54:24 | INFO |       Computing SHAP values for 200 samples (this may take a few minutes)...


...

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [12]:
# CELL 6 - TASK 2: HOSPITAL ADMISSION PREDICTION (COMPLETE - SHAP for ALL Models)
# Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble
# SHAP analysis for ALL 5 models (Voting Ensemble uses surrogate model)
# Each model gets its own folder with all SHAP plots and results
# FIXED: RandomForest SHAP now works correctly with 3D array handling

import os
import numpy as np
import pandas as pd
import logging
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    f1_score, brier_score_loss, roc_curve, precision_recall_curve,
    matthews_corrcoef
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.utils import resample, class_weight
import xgboost as xgb
import lightgbm as lgb
from pytorch_tabnet.tab_model import TabNetClassifier
import shap
import torch

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
TASK_OUTPUT = os.path.join(OUTPUT_PATH, 'task2_admission')
os.makedirs(TASK_OUTPUT, exist_ok=True)

logger.info("="*80)
logger.info("🎯 TASK 2: HOSPITAL ADMISSION PREDICTION")
logger.info("Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble")
logger.info("SHAP analysis for ALL 5 models (each gets FULL SHAP plots)")
logger.info("TabNet: Using KernelExplainer approximation")
logger.info("Voting Ensemble: Surrogate RandomForest + TreeExplainer")
logger.info("="*80)

# ============================================================================
# 1. LOAD DATA (29 admission-available features)
# ============================================================================
X_train = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))

y_train_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_admit.pkl'))
y_val_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_admit.pkl'))
y_test_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

# Extract Admission target
y_train = y_train_full['TARGET_Admitted'].values.astype(int)
y_val = y_val_full['TARGET_Admitted'].values.astype(int)
y_test = y_test_full['TARGET_Admitted'].values.astype(int)

# Load class weights
with open(os.path.join(OUTPUT_PATH, 'class_weights.json'), 'r') as f:
    class_weights = json.load(f)

scale_pos_weight = class_weights['Admission']['scale_pos_weight']

logger.info("\n📊 DATA LOADED")
logger.info(f"   Train: {X_train.shape}, Prevalence: {y_train.mean():.1%}")
logger.info(f"   Val: {X_val.shape}, Prevalence: {y_val.mean():.1%}")
logger.info(f"   Test: {X_test.shape}, Prevalence: {y_test.mean():.1%}")
logger.info(f"   Scale pos weight: {scale_pos_weight:.2f}")

# ============================================================================
# 2. PREPROCESSING
# ============================================================================
logger.info("\n📈 PREPROCESSING")

# Convert to numeric and handle any remaining non-numeric columns
for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_val[col] = pd.to_numeric(X_val[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

# Keep as DataFrame for SHAP compatibility
X_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_df = pd.DataFrame(X_val_scaled, columns=X_train.columns)
X_test_df = pd.DataFrame(X_test_scaled, columns=X_train.columns)

feature_names = X_train.columns.tolist()

logger.info(f"   Features: {len(feature_names)}")
logger.info(f"   Train shape: {X_train_scaled.shape}")

# ============================================================================
# 3. BOOTSTRAP CONFIDENCE INTERVALS FUNCTION
# ============================================================================
def bootstrap_metrics(y_true, y_pred_proba, n_bootstrap=1000, ci=95):
    """Calculate bootstrap confidence intervals for metrics"""
    
    np.random.seed(42)
    n_samples = len(y_true)
    
    # For admission, use threshold 0.5 (clinical decision threshold)
    optimal_thresh = 0.5
    
    boot_metrics = {name: [] for name in ['auc_roc', 'auc_pr', 'sensitivity', 
                                           'specificity', 'f1', 'brier', 'mcc']}
    
    for _ in range(n_bootstrap):
        indices = resample(range(n_samples), n_samples=n_samples)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred_proba[indices]
        
        if len(np.unique(y_true_boot)) == 2:
            auc_roc = roc_auc_score(y_true_boot, y_pred_boot)
            auc_pr = average_precision_score(y_true_boot, y_pred_boot)
            brier = brier_score_loss(y_true_boot, y_pred_boot)
            
            y_pred_class = (y_pred_boot >= optimal_thresh).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true_boot, y_pred_class).ravel()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            f1 = f1_score(y_true_boot, y_pred_class) if (tp + fp + fn) > 0 else 0
            mcc = matthews_corrcoef(y_true_boot, y_pred_class)
            
            boot_metrics['auc_roc'].append(auc_roc)
            boot_metrics['auc_pr'].append(auc_pr)
            boot_metrics['sensitivity'].append(sens)
            boot_metrics['specificity'].append(spec)
            boot_metrics['f1'].append(f1)
            boot_metrics['brier'].append(brier)
            boot_metrics['mcc'].append(mcc)
    
    ci_lower = (100 - ci) / 2
    ci_upper = 100 - ci_lower
    
    results = {}
    for name, values in boot_metrics.items():
        if values:
            results[name] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'ci_lower': np.percentile(values, ci_lower),
                'ci_upper': np.percentile(values, ci_upper)
            }
    
    return results, optimal_thresh

# ============================================================================
# 4. SHAP ANALYSIS FOR TREE MODELS (XGBoost, LightGBM, RandomForest) - FIXED
# ============================================================================
def shap_analysis_tree(model, model_name, X_test_data, feature_names, output_folder, n_samples=300):
    """Comprehensive SHAP analysis for tree-based models (XGBoost, LightGBM, RandomForest)
    
    This function handles both single-output models and multi-output models (RandomForest)
    by detecting if shap_values is a list or 3D array and extracting class 1.
    """
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name}...")
        
        n_use = min(n_samples, X_test_data.shape[0])
        X_subset = X_test_data[:n_use]
        
        # Create TreeExplainer
        explainer = shap.TreeExplainer(model)
        shap_values_raw = explainer.shap_values(X_subset)
        expected_value_raw = explainer.expected_value
        
        logger.info(f"      Raw SHAP type: {type(shap_values_raw)}")
        
        # Handle different output formats
        if isinstance(shap_values_raw, list):
            logger.info(f"      SHAP is list with {len(shap_values_raw)} elements")
            # For binary classification, take class 1 (positive class)
            if len(shap_values_raw) == 2:
                shap_values = shap_values_raw[1]
                logger.info(f"      Using class 1 (positive) SHAP values")
            else:
                shap_values = shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0]
        else:
            shap_values = shap_values_raw
        
        # Handle expected value
        if isinstance(expected_value_raw, list):
            if len(expected_value_raw) == 2:
                expected_value = expected_value_raw[1]
                logger.info(f"      Using class 1 expected value: {expected_value:.4f}")
            else:
                expected_value = expected_value_raw[1] if len(expected_value_raw) > 1 else expected_value_raw[0]
        else:
            expected_value = expected_value_raw
        
        # Ensure shap_values is 2D (n_samples, n_features)
        if hasattr(shap_values, 'shape'):
            logger.info(f"      SHAP values shape before processing: {shap_values.shape}")
        
        # If shap_values is 3D (n_samples, n_features, n_classes), squeeze to 2D
        if len(shap_values.shape) == 3:
            logger.info(f"      SHAP is 3D with shape {shap_values.shape}. Taking class 1 (index 1)...")
            shap_values = shap_values[:, :, 1]
            logger.info(f"      New shape: {shap_values.shape}")
        
        # If shap_values is still 3D with different ordering, try alternative
        if len(shap_values.shape) == 3:
            if shap_values.shape[0] == 2:
                shap_values = shap_values[1, :, :]
                logger.info(f"      Alternative: took first dimension index 1, new shape: {shap_values.shape}")
            else:
                shap_values = shap_values[:, :, 1]
                logger.info(f"      Alternative: took last dimension index 1, new shape: {shap_values.shape}")
        
        # Final check: ensure 2D
        if len(shap_values.shape) != 2:
            logger.error(f"      SHAP values has {len(shap_values.shape)} dimensions. Attempting to flatten...")
            if len(shap_values.shape) == 1:
                shap_values = shap_values.reshape(1, -1)
            else:
                shap_values = shap_values.reshape(shap_values.shape[0], -1)
            logger.info(f"      Reshaped to: {shap_values.shape}")
        
        # Calculate feature importance (mean absolute SHAP values per feature)
        shap_importance = np.abs(shap_values).mean(axis=0)
        
        # Ensure shap_importance is 1D
        if len(shap_importance.shape) > 1:
            logger.info(f"      Importance shape is {shap_importance.shape}, flattening...")
            shap_importance = shap_importance.flatten()
        
        # Ensure lengths match
        if len(shap_importance) != len(feature_names):
            logger.warning(f"      Mismatch: importance length {len(shap_importance)} vs features {len(feature_names)}")
            if len(shap_importance) < len(feature_names):
                # Pad with zeros
                shap_importance = np.pad(shap_importance, (0, len(feature_names) - len(shap_importance)))
            else:
                # Truncate
                shap_importance = shap_importance[:len(feature_names)]
        
        # Create DataFrame
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # SHAP PLOT 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # SHAP PLOT 2: Summary Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # SHAP PLOT 3: Waterfall Plot (first test sample)
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # SHAP PLOT 4: Force Plot
        try:
            shap.initjs()
            shap.force_plot(
                expected_value, shap_values[0], X_subset.iloc[0],
                feature_names=feature_names, matplotlib=True, show=False
            )
            plt.title(f'{model_name} - SHAP Force Plot - Admission', fontsize=14)
            plt.tight_layout()
            plt.savefig(os.path.join(model_folder, 'shap_force_plot.png'), dpi=300, bbox_inches='tight')
            plt.close()
            logger.info(f"      ✅ Saved: shap_force_plot.png")
        except Exception as e:
            logger.warning(f"      Force plot skipped: {str(e)[:50]}")
        
        # SHAP PLOT 5: Dependence Plots for Top 5 Features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (Admission)', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        # Log top features
        logger.info(f"\n   📊 {model_name} Top 10 Features (SHAP):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 5. SHAP ANALYSIS FOR TABNET (using KernelExplainer)
# ============================================================================
def shap_analysis_tabnet(model, model_name, X_train_data, X_test_data, feature_names, output_folder, n_samples=200, n_background=100):
    """SHAP analysis for TabNet using KernelExplainer to produce same plots as tree models"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name} (KernelExplainer)...")
        
        n_background = min(n_background, X_train_data.shape[0])
        n_use = min(n_samples, X_test_data.shape[0])
        
        X_background = X_train_data[:n_background]
        X_subset = X_test_data[:n_use]
        
        # Define prediction function for TabNet
        def predict_proba_fn(x):
            return model.predict_proba(x)[:, 1]
        
        # Create KernelExplainer
        logger.info(f"      Creating KernelExplainer with {n_background} background samples...")
        explainer = shap.KernelExplainer(predict_proba_fn, X_background)
        
        # Compute SHAP values
        logger.info(f"      Computing SHAP values for {n_use} samples (this may take a few minutes)...")
        shap_values = explainer.shap_values(X_subset, nsamples=500)
        
        # Calculate feature importance
        shap_importance = np.abs(shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar - KernelExplainer) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm - KernelExplainer) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=explainer.expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0 - KernelExplainer) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (KernelExplainer) - Admission', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features (KernelExplainer):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 6. MODEL-AGNOSTIC SHAP ANALYSIS FOR VOTING ENSEMBLE
# ============================================================================
def shap_analysis_ensemble(model_proba_fn, model_name, X_train_data, X_test_data, feature_names, output_folder, n_samples=200, n_background=100):
    """SHAP analysis for Voting Ensemble using surrogate RandomForest"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name} (surrogate RandomForest)...")
        
        n_train = min(n_background, X_train_data.shape[0])
        n_use = min(n_samples, X_test_data.shape[0])
        
        X_train_subset = X_train_data[:n_train]
        X_subset = X_test_data[:n_use]
        
        # Get ensemble predictions on training data
        logger.info(f"      Generating ensemble predictions on {n_train} training samples...")
        ensemble_train_preds = model_proba_fn(X_train_subset)
        
        # Train surrogate RandomForest
        logger.info(f"      Training surrogate RandomForest Regressor...")
        surrogate = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        surrogate.fit(X_train_subset, ensemble_train_preds)
        
        # Use TreeExplainer on surrogate
        logger.info(f"      Computing SHAP values for {n_use} test samples...")
        explainer = shap.TreeExplainer(surrogate)
        shap_values = explainer.shap_values(X_subset)
        expected_value = explainer.expected_value
        
        # Calculate feature importance
        shap_importance = np.abs(shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar - Surrogate RF) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm - Surrogate RF) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0 - Surrogate RF) - Admission', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (Surrogate RF) - Admission', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features (Surrogate RandomForest):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 7. TRAIN XGBOOST
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 1: XGBoost")
logger.info("="*60)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    device='cuda:0',
    random_state=42,
    verbosity=0,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=False)
xgb_pred = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_ci, xgb_thresh = bootstrap_metrics(y_test, xgb_pred)
logger.info(f"\n📊 XGBoost Results:")
logger.info(f"   AUC-ROC: {xgb_ci['auc_roc']['mean']:.4f} [{xgb_ci['auc_roc']['ci_lower']:.4f}-{xgb_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {xgb_ci['sensitivity']['mean']:.1%}")

xgb_shap = shap_analysis_tree(xgb_model, 'XGBoost', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 8. TRAIN LIGHTGBM
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 2: LightGBM")
logger.info("="*60)

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.01,
    reg_lambda=0.01,
    device='gpu',
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], 
              eval_metric='auc', callbacks=[lgb.early_stopping(50, verbose=False)])

lgb_pred = lgb_model.predict_proba(X_test_scaled)[:, 1]

lgb_ci, lgb_thresh = bootstrap_metrics(y_test, lgb_pred)
logger.info(f"\n📊 LightGBM Results:")
logger.info(f"   AUC-ROC: {lgb_ci['auc_roc']['mean']:.4f} [{lgb_ci['auc_roc']['ci_lower']:.4f}-{lgb_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {lgb_ci['sensitivity']['mean']:.1%}")

lgb_shap = shap_analysis_tree(lgb_model, 'LightGBM', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 9. TRAIN RANDOM FOREST
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 3: Random Forest")
logger.info("="*60)

# Compute balanced class weights
rf_class_weight = class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train
)
rf_class_weight_dict = {0: rf_class_weight[0], 1: rf_class_weight[1]}

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight=rf_class_weight_dict,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_ci, rf_thresh = bootstrap_metrics(y_test, rf_pred)
logger.info(f"\n📊 Random Forest Results:")
logger.info(f"   AUC-ROC: {rf_ci['auc_roc']['mean']:.4f} [{rf_ci['auc_roc']['ci_lower']:.4f}-{rf_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {rf_ci['sensitivity']['mean']:.1%}")

# SHAP for Random Forest - NOW WORKS with fixed function
rf_shap = shap_analysis_tree(rf_model, 'RandomForest', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 10. TRAIN TABNET
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 4: TabNet")
logger.info("="*60)

tabnet_model = TabNetClassifier(
    n_d=64,
    n_a=64,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='sparsemax',
    verbose=0,
    device_name='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42
)

tabnet_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False,
    weights=np.where(y_train == 1, scale_pos_weight, 1.0)
)

tabnet_pred = tabnet_model.predict_proba(X_test_scaled)[:, 1]

tabnet_ci, tabnet_thresh = bootstrap_metrics(y_test, tabnet_pred)
logger.info(f"\n📊 TabNet Results:")
logger.info(f"   AUC-ROC: {tabnet_ci['auc_roc']['mean']:.4f} [{tabnet_ci['auc_roc']['ci_lower']:.4f}-{tabnet_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {tabnet_ci['sensitivity']['mean']:.1%}")

# SHAP for TabNet - Using KernelExplainer to get FULL SHAP plots (not just feature importance)
tabnet_shap = shap_analysis_tabnet(tabnet_model, 'TabNet', X_train_df, X_test_df, feature_names, TASK_OUTPUT, n_samples=200)

# ============================================================================
# 11. VOTING ENSEMBLE (4 models: XGB, LGB, RF, TabNet)
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 5: Voting Ensemble (Weighted - 4 Models)")
logger.info("="*60)

# Calculate validation AUCs for weights
xgb_val_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val_scaled)[:, 1])
lgb_val_auc = roc_auc_score(y_val, lgb_model.predict_proba(X_val_scaled)[:, 1])
rf_val_auc = roc_auc_score(y_val, rf_model.predict_proba(X_val_scaled)[:, 1])
tabnet_val_auc = roc_auc_score(y_val, tabnet_model.predict_proba(X_val_scaled)[:, 1])

total_auc = xgb_val_auc + lgb_val_auc + rf_val_auc + tabnet_val_auc
weights = {
    'xgb': xgb_val_auc / total_auc,
    'lgb': lgb_val_auc / total_auc,
    'rf': rf_val_auc / total_auc,
    'tabnet': tabnet_val_auc / total_auc
}

logger.info(f"\n   Validation AUCs:")
logger.info(f"      XGBoost: {xgb_val_auc:.4f}")
logger.info(f"      LightGBM: {lgb_val_auc:.4f}")
logger.info(f"      Random Forest: {rf_val_auc:.4f}")
logger.info(f"      TabNet: {tabnet_val_auc:.4f}")
logger.info(f"\n   Ensemble Weights:")
logger.info(f"      XGB: {weights['xgb']:.3f}, LGB: {weights['lgb']:.3f}, RF: {weights['rf']:.3f}, TabNet: {weights['tabnet']:.3f}")

# Weighted ensemble predictions
ensemble_pred = (
    weights['xgb'] * xgb_pred +
    weights['lgb'] * lgb_pred +
    weights['rf'] * rf_pred +
    weights['tabnet'] * tabnet_pred
)

ensemble_ci, ensemble_thresh = bootstrap_metrics(y_test, ensemble_pred)
logger.info(f"\n📊 Voting Ensemble Results:")
logger.info(f"   AUC-ROC: {ensemble_ci['auc_roc']['mean']:.4f} [{ensemble_ci['auc_roc']['ci_lower']:.4f}-{ensemble_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {ensemble_ci['sensitivity']['mean']:.1%}")

# Define prediction function for ensemble
def ensemble_predict_proba(X):
    """Returns ensemble predictions for a given input matrix"""
    xgb_proba = xgb_model.predict_proba(X)[:, 1]
    lgb_proba = lgb_model.predict_proba(X)[:, 1]
    rf_proba = rf_model.predict_proba(X)[:, 1]
    tabnet_proba = tabnet_model.predict_proba(X)[:, 1]
    
    return (weights['xgb'] * xgb_proba +
            weights['lgb'] * lgb_proba +
            weights['rf'] * rf_proba +
            weights['tabnet'] * tabnet_proba)

# SHAP for Voting Ensemble (using surrogate model)
ensemble_shap = shap_analysis_ensemble(
    ensemble_predict_proba, 'VotingEnsemble', X_train_df, X_test_df, feature_names, TASK_OUTPUT, n_samples=200
)

# ============================================================================
# 12. MODEL COMPARISON
# ============================================================================
logger.info("\n" + "="*80)
logger.info("📊 MODEL COMPARISON SUMMARY")
logger.info("="*80)

comparison_df = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Random Forest', 'TabNet', 'Voting Ensemble'],
    'AUC-ROC': [xgb_ci['auc_roc']['mean'], lgb_ci['auc_roc']['mean'], 
                rf_ci['auc_roc']['mean'], tabnet_ci['auc_roc']['mean'], 
                ensemble_ci['auc_roc']['mean']],
    'AUC-ROC_CI_lower': [xgb_ci['auc_roc']['ci_lower'], lgb_ci['auc_roc']['ci_lower'],
                         rf_ci['auc_roc']['ci_lower'], tabnet_ci['auc_roc']['ci_lower'],
                         ensemble_ci['auc_roc']['ci_lower']],
    'AUC-ROC_CI_upper': [xgb_ci['auc_roc']['ci_upper'], lgb_ci['auc_roc']['ci_upper'],
                         rf_ci['auc_roc']['ci_upper'], tabnet_ci['auc_roc']['ci_upper'],
                         ensemble_ci['auc_roc']['ci_upper']],
    'Sensitivity': [xgb_ci['sensitivity']['mean'], lgb_ci['sensitivity']['mean'],
                    rf_ci['sensitivity']['mean'], tabnet_ci['sensitivity']['mean'],
                    ensemble_ci['sensitivity']['mean']],
    'Specificity': [xgb_ci['specificity']['mean'], lgb_ci['specificity']['mean'],
                    rf_ci['specificity']['mean'], tabnet_ci['specificity']['mean'],
                    ensemble_ci['specificity']['mean']],
    'F1': [xgb_ci['f1']['mean'], lgb_ci['f1']['mean'],
           rf_ci['f1']['mean'], tabnet_ci['f1']['mean'],
           ensemble_ci['f1']['mean']]
})

comparison_df = comparison_df.sort_values('AUC-ROC', ascending=False)
comparison_df.to_csv(os.path.join(TASK_OUTPUT, 'model_comparison.csv'), index=False)

logger.info("\n" + comparison_df[['Model', 'AUC-ROC', 'Sensitivity', 'Specificity', 'F1']].to_string(index=False))

# ============================================================================
# 13. VISUALIZATIONS
# ============================================================================
logger.info("\n📈 Creating Performance Visualizations")

# ROC Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    fpr, tpr, _ = roc_curve(y_test, preds)
    auc = roc_auc_score(y_test, preds)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Hospital Admission Prediction', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'roc_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: roc_curves.png")

# PR Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    precision, recall, _ = precision_recall_curve(y_test, preds)
    auprc = average_precision_score(y_test, preds)
    plt.plot(recall, precision, label=f'{name} (AUPRC={auprc:.4f})', linewidth=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves - Hospital Admission Prediction', fontsize=14)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'pr_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: pr_curves.png")

# Calibration Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    prob_true, prob_pred = calibration_curve(y_test, preds, n_bins=10)
    ece = np.mean(np.abs(prob_true - prob_pred))
    plt.plot(prob_pred, prob_true, marker='o', label=f'{name} (ECE={ece:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curves - Hospital Admission Prediction', fontsize=14)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'calibration_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: calibration_curves.png")

# ============================================================================
# 14. SAVE ALL MODEL PREDICTIONS
# ============================================================================
predictions_df = pd.DataFrame({
    'XGBoost': xgb_pred,
    'LightGBM': lgb_pred,
    'RandomForest': rf_pred,
    'TabNet': tabnet_pred,
    'VotingEnsemble': ensemble_pred,
    'y_true': y_test
})
predictions_df.to_csv(os.path.join(TASK_OUTPUT, 'all_predictions.csv'), index=False)
logger.info("   ✅ Saved: all_predictions.csv")

# ============================================================================
# 15. FINAL SUMMARY
# ============================================================================
logger.info("\n" + "="*80)
logger.info("✅ TASK 2 COMPLETE")
logger.info("="*80)
logger.info(f"Best Model: {comparison_df.iloc[0]['Model']} (AUC={comparison_df.iloc[0]['AUC-ROC']:.4f})")
logger.info(f"\n📁 SHAP Output Folders (ALL with COMPLETE plots):")
logger.info(f"   {TASK_OUTPUT}/")
logger.info(f"   ├── xgboost_shap/        (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── lightgbm_shap/       (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── randomforest_shap/   (SHAP bar, summary, waterfall, dependence) ✅ FIXED")
logger.info(f"   ├── tabnet_shap/         (SHAP bar, summary, waterfall, dependence - KernelExplainer)")
logger.info(f"   ├── votingensemble_shap/ (SHAP bar, summary, waterfall, dependence - Surrogate RF)")
logger.info(f"   ├── model_comparison.csv")
logger.info(f"   ├── roc_curves.png")
logger.info(f"   ├── pr_curves.png")
logger.info(f"   ├── calibration_curves.png")
logger.info(f"   └── all_predictions.csv")
logger.info("="*80)

2026-07-24 15:06:23 | INFO | ================================================================================
2026-07-24 15:06:23 | INFO | 🎯 TASK 2: HOSPITAL ADMISSION PREDICTION
2026-07-24 15:06:23 | INFO | Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble
2026-07-24 15:06:23 | INFO | SHAP analysis for ALL 5 models (each gets FULL SHAP plots)
2026-07-24 15:06:23 | INFO | TabNet: Using KernelExplainer approximation
2026-07-24 15:06:23 | INFO | Voting Ensemble: Surrogate RandomForest + TreeExplainer
2026-07-24 15:06:23 | INFO | ================================================================================
2026-07-24 15:06:23 | INFO | 
📊 DATA LOADED
2026-07-24 15:06:23 | INFO |    Train: (81588, 29), Prevalence: 40.1%
2026-07-24 15:06:23 | INFO |    Val: (11770, 29), Prevalence: 41.0%
2026-07-24 15:06:23 | INFO |    Test: (12020, 29), Prevalence: 39.6%
2026-07-24 15:06:23 | INFO |    Scale pos weight: 1.49
2026-07-24 15:06:23 | INFO | 
📈 PREPROCESSING
2026-07-24 15:06:23 

2026-07-24 15:06:58 | INFO |       ✅ Saved: shap_force_plot.png
2026-07-24 15:06:59 | INFO |       ✅ Saved: shap_dependence_Age.png
2026-07-24 15:06:59 | INFO |       ✅ Saved: shap_dependence_Triage Acuity Level.png
2026-07-24 15:07:00 | INFO |       ✅ Saved: shap_dependence_Home Medication Count.png
2026-07-24 15:07:00 | INFO |       ✅ Saved: shap_dependence_EMS Arrival.png
2026-07-24 15:07:01 | INFO |       ✅ Saved: shap_dependence_Triage Heart Rate.png
2026-07-24 15:07:01 | INFO | 
   📊 XGBoost Top 10 Features (SHAP):
2026-07-24 15:07:01 | INFO |       1. Age: 0.5794
2026-07-24 15:07:01 | INFO |       14. Triage Acuity Level: 0.4492
2026-07-24 15:07:01 | INFO |       27. Home Medication Count: 0.2872
2026-07-24 15:07:01 | INFO |       17. EMS Arrival: 0.2115
2026-07-24 15:07:01 | INFO |       9. Triage Heart Rate: 0.1864
2026-07-24 15:07:01 | INFO |       28. Unique Medication Classes Count: 0.1570
2026-07-24 15:07:01 | INFO |       12. Triage Systolic BP: 0.1456
2026-07-24 15:07:01

2026-07-24 15:07:41 | INFO |       ✅ Saved: shap_force_plot.png
2026-07-24 15:07:42 | INFO |       ✅ Saved: shap_dependence_Age.png
2026-07-24 15:07:42 | INFO |       ✅ Saved: shap_dependence_Triage Acuity Level.png
2026-07-24 15:07:43 | INFO |       ✅ Saved: shap_dependence_Home Medication Count.png
2026-07-24 15:07:43 | INFO |       ✅ Saved: shap_dependence_EMS Arrival.png
2026-07-24 15:07:44 | INFO |       ✅ Saved: shap_dependence_Triage Heart Rate.png
2026-07-24 15:07:44 | INFO | 
   📊 LightGBM Top 10 Features (SHAP):
2026-07-24 15:07:44 | INFO |       1. Age: 0.6554
2026-07-24 15:07:44 | INFO |       14. Triage Acuity Level: 0.4319
2026-07-24 15:07:44 | INFO |       27. Home Medication Count: 0.2731
2026-07-24 15:07:44 | INFO |       17. EMS Arrival: 0.2177
2026-07-24 15:07:44 | INFO |       9. Triage Heart Rate: 0.1814
2026-07-24 15:07:44 | INFO |       26. Disease Condition Count: 0.1360
2026-07-24 15:07:44 | INFO |       12. Triage Systolic BP: 0.1340
2026-07-24 15:07:44 | INFO


Early stopping occurred at epoch 83 with best_epoch = 63 and best_val_auc = 0.81513


2026-07-24 15:35:49 | INFO | 
📊 TabNet Results:
2026-07-24 15:35:49 | INFO |    AUC-ROC: 0.8150 [0.8075-0.8223]
2026-07-24 15:35:49 | INFO |    Sensitivity: 72.4%
2026-07-24 15:35:49 | INFO | 
   🔍 Computing SHAP values for TabNet (KernelExplainer)...
2026-07-24 15:35:50 | INFO |       Creating KernelExplainer with 100 background samples...
2026-07-24 15:35:50 | INFO |       Computing SHAP values for 200 samples (this may take a few minutes)...


  0%|          | 0/200 [00:00<?, ?it/s]

2026-07-24 15:35:50 | INFO | num_full_subsets = 1
2026-07-24 15:35:50 | INFO | remaining_weight_vector = array([0.18986653, 0.13208106, 0.10356356, 0.08679613, 0.07594661,
       0.06852326, 0.06328884, 0.05956597, 0.05695996, 0.0552339 ,
       0.05424758, 0.05392659])
2026-07-24 15:35:50 | INFO | num_paired_subset_sizes = 12
2026-07-24 15:35:50 | INFO | weight_left = np.float64(0.7274603254136663)
2026-07-24 15:35:54 | INFO | np.sum(w_aug) = np.float64(26.000000000000007)
2026-07-24 15:35:54 | INFO | np.sum(self.kernelWeights) = np.float64(0.9999999999999999)
2026-07-24 15:35:54 | INFO | phi = array([ 0.03427035,  0.        ,  0.        ,  0.01239712,  0.        ,
        0.        ,  0.        , -0.01521201,  0.        ,  0.01062889,
        0.        ,  0.        , -0.00800156,  0.        ,  0.        ,
        0.052152  ,  0.        ,  0.        ,  0.        , -0.01895355,
        0.        ,  0.        ,  0.01724673, -0.01256656, -0.01528895,
        0.        ])
2026-07-24 15:35

In [13]:
# CELL 7 - TASK 3: HOSP-LOS >7 DAYS PREDICTION (COMPLETE - SHAP for ALL Models)
# Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble
# SHAP analysis for ALL 5 models - Each gets FULL SHAP plots (bar, summary, waterfall, dependence)
# TabNet uses KernelExplainer approximation
# Voting Ensemble uses surrogate RandomForest with proper SHAP

import os
import numpy as np
import pandas as pd
import logging
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    f1_score, brier_score_loss, roc_curve, precision_recall_curve,
    matthews_corrcoef
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.utils import resample, class_weight
import xgboost as xgb
import lightgbm as lgb
from pytorch_tabnet.tab_model import TabNetClassifier
import shap
import torch

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"
TASK_OUTPUT = os.path.join(OUTPUT_PATH, 'task3_hosp_los')
os.makedirs(TASK_OUTPUT, exist_ok=True)

logger.info("="*80)
logger.info("🎯 TASK 3: HOSP-LOS >7 DAYS PREDICTION")
logger.info("Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble")
logger.info("SHAP analysis for ALL 5 models (each gets FULL SHAP plots)")
logger.info("TabNet: Using KernelExplainer approximation")
logger.info("Voting Ensemble: Surrogate RandomForest + TreeExplainer")
logger.info("="*80)

# ============================================================================
# 1. LOAD DATA
# ============================================================================
X_train = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train_hosp.pkl'))
X_val = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val_hosp.pkl'))
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test_hosp.pkl'))

y_train_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_hosp.pkl'))
y_val_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_hosp.pkl'))
y_test_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))

# Extract Hosp-LOS target
y_train = pd.to_numeric(y_train_df['TARGET_Hosp_LOS_over7d'], errors='coerce').values.astype(float)
y_val = pd.to_numeric(y_val_df['TARGET_Hosp_LOS_over7d'], errors='coerce').values.astype(float)
y_test = pd.to_numeric(y_test_df['TARGET_Hosp_LOS_over7d'], errors='coerce').values.astype(float)

# Remove NaN values
train_valid_mask = ~np.isnan(y_train)
val_valid_mask = ~np.isnan(y_val)
test_valid_mask = ~np.isnan(y_test)

y_train = y_train[train_valid_mask].astype(int)
y_val = y_val[val_valid_mask].astype(int)
y_test = y_test[test_valid_mask].astype(int)

X_train = X_train[train_valid_mask]
X_val = X_val[val_valid_mask]
X_test = X_test[test_valid_mask]

# Reset indices
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

# Load class weights
with open(os.path.join(OUTPUT_PATH, 'class_weights.json'), 'r') as f:
    class_weights = json.load(f)

scale_pos_weight = class_weights['Hosp_LOS_over7d']['scale_pos_weight']
prevalence = class_weights['Hosp_LOS_over7d']['prevalence']

logger.info("\n📊 DATA LOADED")
logger.info(f"   Train: {X_train.shape}, Prevalence: {y_train.mean():.1%}")
logger.info(f"   Val: {X_val.shape}, Prevalence: {y_val.mean():.1%}")
logger.info(f"   Test: {X_test.shape}, Prevalence: {y_test.mean():.1%}")

# ============================================================================
# 2. PREPROCESSING
# ============================================================================
logger.info("\n📈 PREPROCESSING")

for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_val[col] = pd.to_numeric(X_val[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

X_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_df = pd.DataFrame(X_val_scaled, columns=X_train.columns)
X_test_df = pd.DataFrame(X_test_scaled, columns=X_train.columns)

feature_names = X_train.columns.tolist()
logger.info(f"   Features: {len(feature_names)}")

# ============================================================================
# 3. BOOTSTRAP CONFIDENCE INTERVALS FUNCTION
# ============================================================================
def bootstrap_metrics(y_true, y_pred_proba, n_bootstrap=1000, ci=95):
    np.random.seed(42)
    n_samples = len(y_true)
    
    thresholds = np.arange(0.05, 0.95, 0.05)
    best_youden = 0
    optimal_thresh = 0.5
    for thresh in thresholds:
        y_pred_class = (y_pred_proba >= thresh).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        youden = sens + spec - 1
        if youden > best_youden:
            best_youden = youden
            optimal_thresh = thresh
    
    boot_metrics = {name: [] for name in ['auc_roc', 'auc_pr', 'sensitivity', 
                                           'specificity', 'f1', 'brier', 'mcc']}
    
    for _ in range(n_bootstrap):
        indices = resample(range(n_samples), n_samples=n_samples)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred_proba[indices]
        
        if len(np.unique(y_true_boot)) == 2:
            auc_roc = roc_auc_score(y_true_boot, y_pred_boot)
            auc_pr = average_precision_score(y_true_boot, y_pred_boot)
            brier = brier_score_loss(y_true_boot, y_pred_boot)
            
            y_pred_class = (y_pred_boot >= optimal_thresh).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true_boot, y_pred_class).ravel()
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            f1 = f1_score(y_true_boot, y_pred_class) if (tp + fp + fn) > 0 else 0
            mcc = matthews_corrcoef(y_true_boot, y_pred_class)
            
            boot_metrics['auc_roc'].append(auc_roc)
            boot_metrics['auc_pr'].append(auc_pr)
            boot_metrics['sensitivity'].append(sens)
            boot_metrics['specificity'].append(spec)
            boot_metrics['f1'].append(f1)
            boot_metrics['brier'].append(brier)
            boot_metrics['mcc'].append(mcc)
    
    ci_lower = (100 - ci) / 2
    ci_upper = 100 - ci_lower
    
    results = {}
    for name, values in boot_metrics.items():
        if values:
            results[name] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'ci_lower': np.percentile(values, ci_lower),
                'ci_upper': np.percentile(values, ci_upper)
            }
    return results, optimal_thresh

# ============================================================================
# 4. SHAP ANALYSIS FOR TREE MODELS (XGBoost, LightGBM, RandomForest) - FIXED
# ============================================================================
def shap_analysis_tree(model, model_name, X_test_data, feature_names, output_folder, n_samples=300):
    """Comprehensive SHAP analysis for tree-based models"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name}...")
        
        n_use = min(n_samples, X_test_data.shape[0])
        X_subset = X_test_data[:n_use]
        
        explainer = shap.TreeExplainer(model)
        shap_values_raw = explainer.shap_values(X_subset)
        expected_value_raw = explainer.expected_value
        
        logger.info(f"      Raw SHAP type: {type(shap_values_raw)}")
        
        # Handle different output formats
        if isinstance(shap_values_raw, list):
            logger.info(f"      SHAP is list with {len(shap_values_raw)} elements")
            # For binary classification, take class 1 (positive class)
            if len(shap_values_raw) == 2:
                shap_values = shap_values_raw[1]
                logger.info(f"      Using class 1 (positive) SHAP values")
            else:
                shap_values = shap_values_raw[1] if len(shap_values_raw) > 1 else shap_values_raw[0]
        else:
            shap_values = shap_values_raw
        
        # Handle expected value
        if isinstance(expected_value_raw, list):
            if len(expected_value_raw) == 2:
                expected_value = expected_value_raw[1]
            else:
                expected_value = expected_value_raw[1] if len(expected_value_raw) > 1 else expected_value_raw[0]
        else:
            expected_value = expected_value_raw
        
        # Ensure shap_values is 2D (n_samples, n_features)
        if hasattr(shap_values, 'shape'):
            logger.info(f"      SHAP values shape before processing: {shap_values.shape}")
        
        # If shap_values is 3D (n_samples, n_features, n_classes), squeeze to 2D
        if len(shap_values.shape) == 3:
            logger.info(f"      SHAP is 3D with shape {shap_values.shape}. Taking class 1 (index 1)...")
            shap_values = shap_values[:, :, 1]
            logger.info(f"      New shape: {shap_values.shape}")
        
        # If shap_values is still 3D with different ordering, try alternative
        if len(shap_values.shape) == 3:
            if shap_values.shape[0] == 2:
                shap_values = shap_values[1, :, :]
            else:
                shap_values = shap_values[:, :, 1]
            logger.info(f"      New shape after alternative: {shap_values.shape}")
        
        # Final check: ensure 2D
        if len(shap_values.shape) != 2:
            logger.error(f"      SHAP values has {len(shap_values.shape)} dimensions. Attempting to flatten...")
            if len(shap_values.shape) == 1:
                shap_values = shap_values.reshape(1, -1)
            else:
                shap_values = shap_values.reshape(shap_values.shape[0], -1)
            logger.info(f"      Reshaped to: {shap_values.shape}")
        
        # Calculate feature importance (mean absolute SHAP values per feature)
        shap_importance = np.abs(shap_values).mean(axis=0)
        
        # Ensure shap_importance is 1D
        if len(shap_importance.shape) > 1:
            logger.info(f"      Importance shape is {shap_importance.shape}, flattening...")
            shap_importance = shap_importance.flatten()
        
        # Ensure lengths match
        if len(shap_importance) != len(feature_names):
            logger.warning(f"      Mismatch: importance length {len(shap_importance)} vs features {len(feature_names)}")
            if len(shap_importance) < len(feature_names):
                # Pad with zeros
                shap_importance = np.pad(shap_importance, (0, len(feature_names) - len(shap_importance)))
            else:
                # Truncate
                shap_importance = shap_importance[:len(feature_names)]
        
        # Create DataFrame
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot (first test sample)
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Force Plot
        try:
            shap.initjs()
            shap.force_plot(
                expected_value, shap_values[0], X_subset.iloc[0],
                feature_names=feature_names, matplotlib=True, show=False
            )
            plt.title(f'{model_name} - SHAP Force Plot', fontsize=14)
            plt.tight_layout()
            plt.savefig(os.path.join(model_folder, 'shap_force_plot.png'), dpi=300, bbox_inches='tight')
            plt.close()
            logger.info(f"      ✅ Saved: shap_force_plot.png")
        except Exception as e:
            logger.warning(f"      Force plot skipped: {str(e)[:50]}")
        
        # Plot 5: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature}', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features:")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 5. SHAP ANALYSIS FOR TABNET (using KernelExplainer)
# ============================================================================
def shap_analysis_tabnet(model, model_name, X_train_data, X_test_data, feature_names, output_folder, n_samples=200, n_background=100):
    """SHAP analysis for TabNet using KernelExplainer"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name} (KernelExplainer)...")
        
        n_background = min(n_background, X_train_data.shape[0])
        n_use = min(n_samples, X_test_data.shape[0])
        
        X_background = X_train_data[:n_background]
        X_subset = X_test_data[:n_use]
        
        # Define prediction function for TabNet
        def predict_proba_fn(x):
            return model.predict_proba(x)[:, 1]
        
        # Create KernelExplainer
        logger.info(f"      Creating KernelExplainer with {n_background} background samples...")
        explainer = shap.KernelExplainer(predict_proba_fn, X_background)
        
        # Compute SHAP values
        logger.info(f"      Computing SHAP values for {n_use} samples (this may take a few minutes)...")
        shap_values = explainer.shap_values(X_subset, nsamples=500)
        
        # Calculate feature importance
        shap_importance = np.abs(shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar - KernelExplainer)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm - KernelExplainer)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=explainer.expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0 - KernelExplainer)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (KernelExplainer)', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features (KernelExplainer):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 6. SHAP ANALYSIS FOR VOTING ENSEMBLE (using surrogate model)
# ============================================================================
def shap_analysis_ensemble(model_proba_fn, model_name, X_train_data, X_test_data, feature_names, output_folder, n_samples=200, n_background=100):
    """SHAP analysis for Voting Ensemble using surrogate RandomForest"""
    
    model_folder = os.path.join(output_folder, f'{model_name.lower()}_shap')
    os.makedirs(model_folder, exist_ok=True)
    
    try:
        logger.info(f"\n   🔍 Computing SHAP values for {model_name} (surrogate RandomForest)...")
        
        n_train = min(n_background, X_train_data.shape[0])
        n_use = min(n_samples, X_test_data.shape[0])
        
        X_train_subset = X_train_data[:n_train]
        X_subset = X_test_data[:n_use]
        
        # Get ensemble predictions on training data
        logger.info(f"      Generating ensemble predictions on {n_train} training samples...")
        ensemble_train_preds = model_proba_fn(X_train_subset)
        
        # Train surrogate RandomForest
        logger.info(f"      Training surrogate RandomForest Regressor...")
        surrogate = RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        surrogate.fit(X_train_subset, ensemble_train_preds)
        
        # Use TreeExplainer on surrogate
        logger.info(f"      Computing SHAP values for {n_use} test samples...")
        explainer = shap.TreeExplainer(surrogate)
        shap_values = explainer.shap_values(X_subset)
        expected_value = explainer.expected_value
        
        # Calculate feature importance
        shap_importance = np.abs(shap_values).mean(axis=0)
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': shap_importance
        }).sort_values('importance', ascending=False)
        importance_df.to_csv(os.path.join(model_folder, 'shap_importance.csv'), index=False)
        
        # Plot 1: Summary Bar Plot
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False)
        plt.title(f'{model_name} - SHAP Feature Importance (Bar - Surrogate RF)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_bar_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_bar_plot.png")
        
        # Plot 2: Bee Swarm Plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_subset, feature_names=feature_names, show=False, plot_type='dot')
        plt.title(f'{model_name} - SHAP Summary (Bee Swarm - Surrogate RF)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_summary_plot.png")
        
        # Plot 3: Waterfall Plot
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values[0],
                base_values=expected_value,
                data=X_subset.iloc[0],
                feature_names=feature_names
            ),
            show=False
        )
        plt.title(f'{model_name} - SHAP Waterfall Plot (Sample 0 - Surrogate RF)', fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(model_folder, 'shap_waterfall_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"      ✅ Saved: shap_waterfall_plot.png")
        
        # Plot 4: Dependence plots for top 5 features
        top5_features = importance_df.head(5)['feature'].tolist()
        for feature in top5_features:
            try:
                plt.figure(figsize=(10, 6))
                feature_idx = feature_names.index(feature)
                shap.dependence_plot(
                    feature_idx, shap_values, X_subset,
                    feature_names=feature_names, show=False
                )
                plt.title(f'{model_name} - SHAP Dependence: {feature} (Surrogate RF)', fontsize=14)
                plt.tight_layout()
                plt.savefig(os.path.join(model_folder, f'shap_dependence_{feature}.png'), dpi=300, bbox_inches='tight')
                plt.close()
                logger.info(f"      ✅ Saved: shap_dependence_{feature}.png")
            except Exception as e:
                logger.warning(f"      Dependence plot failed for {feature}: {str(e)[:50]}")
        
        logger.info(f"\n   📊 {model_name} Top 10 Features (Surrogate RandomForest):")
        for i, row in importance_df.head(10).iterrows():
            logger.info(f"      {i+1}. {row['feature'][:40]}: {row['importance']:.4f}")
        
        return importance_df
    
    except Exception as e:
        logger.warning(f"   SHAP analysis failed for {model_name}: {str(e)[:150]}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# 7. TRAIN XGBOOST
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 1: XGBoost")
logger.info("="*60)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    device='cuda:0',
    random_state=42,
    verbosity=0,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=False)
xgb_pred = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_ci, xgb_thresh = bootstrap_metrics(y_test, xgb_pred)
logger.info(f"\n📊 XGBoost Results:")
logger.info(f"   AUC-ROC: {xgb_ci['auc_roc']['mean']:.4f} [{xgb_ci['auc_roc']['ci_lower']:.4f}-{xgb_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {xgb_ci['sensitivity']['mean']:.1%}")

xgb_shap = shap_analysis_tree(xgb_model, 'XGBoost', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 8. TRAIN LIGHTGBM
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 2: LightGBM")
logger.info("="*60)

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    reg_alpha=0.01,
    reg_lambda=0.01,
    device='gpu',
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], 
              eval_metric='auc', callbacks=[lgb.early_stopping(50, verbose=False)])

lgb_pred = lgb_model.predict_proba(X_test_scaled)[:, 1]

lgb_ci, lgb_thresh = bootstrap_metrics(y_test, lgb_pred)
logger.info(f"\n📊 LightGBM Results:")
logger.info(f"   AUC-ROC: {lgb_ci['auc_roc']['mean']:.4f} [{lgb_ci['auc_roc']['ci_lower']:.4f}-{lgb_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {lgb_ci['sensitivity']['mean']:.1%}")

lgb_shap = shap_analysis_tree(lgb_model, 'LightGBM', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 9. TRAIN RANDOM FOREST
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 3: Random Forest")
logger.info("="*60)

rf_class_weight = class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train
)
rf_class_weight_dict = {0: rf_class_weight[0], 1: rf_class_weight[1]}

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight=rf_class_weight_dict,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_ci, rf_thresh = bootstrap_metrics(y_test, rf_pred)
logger.info(f"\n📊 Random Forest Results:")
logger.info(f"   AUC-ROC: {rf_ci['auc_roc']['mean']:.4f} [{rf_ci['auc_roc']['ci_lower']:.4f}-{rf_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {rf_ci['sensitivity']['mean']:.1%}")

rf_shap = shap_analysis_tree(rf_model, 'RandomForest', X_test_df, feature_names, TASK_OUTPUT, n_samples=300)

# ============================================================================
# 10. TRAIN TABNET
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 4: TabNet")
logger.info("="*60)

tabnet_model = TabNetClassifier(
    n_d=64,
    n_a=64,
    n_steps=5,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='sparsemax',
    verbose=0,
    device_name='cuda' if torch.cuda.is_available() else 'cpu',
    seed=42
)

tabnet_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    eval_name=['val'],
    eval_metric=['auc'],
    max_epochs=100,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0,
    drop_last=False,
    weights=np.where(y_train == 1, scale_pos_weight, 1.0)
)

tabnet_pred = tabnet_model.predict_proba(X_test_scaled)[:, 1]

tabnet_ci, tabnet_thresh = bootstrap_metrics(y_test, tabnet_pred)
logger.info(f"\n📊 TabNet Results:")
logger.info(f"   AUC-ROC: {tabnet_ci['auc_roc']['mean']:.4f} [{tabnet_ci['auc_roc']['ci_lower']:.4f}-{tabnet_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {tabnet_ci['sensitivity']['mean']:.1%}")

# SHAP for TabNet - Using KernelExplainer
tabnet_shap = shap_analysis_tabnet(tabnet_model, 'TabNet', X_train_df, X_test_df, feature_names, TASK_OUTPUT, n_samples=200)

# ============================================================================
# 11. VOTING ENSEMBLE
# ============================================================================
logger.info("\n" + "="*60)
logger.info("🏆 Model 5: Voting Ensemble (Weighted - 4 Models)")
logger.info("="*60)

xgb_val_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val_scaled)[:, 1])
lgb_val_auc = roc_auc_score(y_val, lgb_model.predict_proba(X_val_scaled)[:, 1])
rf_val_auc = roc_auc_score(y_val, rf_model.predict_proba(X_val_scaled)[:, 1])
tabnet_val_auc = roc_auc_score(y_val, tabnet_model.predict_proba(X_val_scaled)[:, 1])

total_auc = xgb_val_auc + lgb_val_auc + rf_val_auc + tabnet_val_auc
weights = {
    'xgb': xgb_val_auc / total_auc,
    'lgb': lgb_val_auc / total_auc,
    'rf': rf_val_auc / total_auc,
    'tabnet': tabnet_val_auc / total_auc
}

logger.info(f"\n   Validation AUCs:")
logger.info(f"      XGBoost: {xgb_val_auc:.4f}")
logger.info(f"      LightGBM: {lgb_val_auc:.4f}")
logger.info(f"      Random Forest: {rf_val_auc:.4f}")
logger.info(f"      TabNet: {tabnet_val_auc:.4f}")
logger.info(f"\n   Ensemble Weights:")
logger.info(f"      XGB: {weights['xgb']:.3f}, LGB: {weights['lgb']:.3f}, RF: {weights['rf']:.3f}, TabNet: {weights['tabnet']:.3f}")

ensemble_pred = (
    weights['xgb'] * xgb_pred +
    weights['lgb'] * lgb_pred +
    weights['rf'] * rf_pred +
    weights['tabnet'] * tabnet_pred
)

ensemble_ci, ensemble_thresh = bootstrap_metrics(y_test, ensemble_pred)
logger.info(f"\n📊 Voting Ensemble Results:")
logger.info(f"   AUC-ROC: {ensemble_ci['auc_roc']['mean']:.4f} [{ensemble_ci['auc_roc']['ci_lower']:.4f}-{ensemble_ci['auc_roc']['ci_upper']:.4f}]")
logger.info(f"   Sensitivity: {ensemble_ci['sensitivity']['mean']:.1%}")

# Define prediction function for ensemble
def ensemble_predict_proba(X):
    """Returns ensemble predictions for a given input matrix"""
    xgb_proba = xgb_model.predict_proba(X)[:, 1]
    lgb_proba = lgb_model.predict_proba(X)[:, 1]
    rf_proba = rf_model.predict_proba(X)[:, 1]
    tabnet_proba = tabnet_model.predict_proba(X)[:, 1]
    
    return (weights['xgb'] * xgb_proba +
            weights['lgb'] * lgb_proba +
            weights['rf'] * rf_proba +
            weights['tabnet'] * tabnet_proba)

# SHAP for Voting Ensemble
ensemble_shap = shap_analysis_ensemble(
    ensemble_predict_proba, 'VotingEnsemble', X_train_df, X_test_df, feature_names, TASK_OUTPUT, n_samples=200
)

# ============================================================================
# 12. MODEL COMPARISON
# ============================================================================
logger.info("\n" + "="*80)
logger.info("📊 MODEL COMPARISON SUMMARY")
logger.info("="*80)

comparison_df = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Random Forest', 'TabNet', 'Voting Ensemble'],
    'AUC-ROC': [xgb_ci['auc_roc']['mean'], lgb_ci['auc_roc']['mean'], 
                rf_ci['auc_roc']['mean'], tabnet_ci['auc_roc']['mean'], 
                ensemble_ci['auc_roc']['mean']],
    'AUC-ROC_CI_lower': [xgb_ci['auc_roc']['ci_lower'], lgb_ci['auc_roc']['ci_lower'],
                         rf_ci['auc_roc']['ci_lower'], tabnet_ci['auc_roc']['ci_lower'],
                         ensemble_ci['auc_roc']['ci_lower']],
    'AUC-ROC_CI_upper': [xgb_ci['auc_roc']['ci_upper'], lgb_ci['auc_roc']['ci_upper'],
                         rf_ci['auc_roc']['ci_upper'], tabnet_ci['auc_roc']['ci_upper'],
                         ensemble_ci['auc_roc']['ci_upper']],
    'Sensitivity': [xgb_ci['sensitivity']['mean'], lgb_ci['sensitivity']['mean'],
                    rf_ci['sensitivity']['mean'], tabnet_ci['sensitivity']['mean'],
                    ensemble_ci['sensitivity']['mean']],
    'Specificity': [xgb_ci['specificity']['mean'], lgb_ci['specificity']['mean'],
                    rf_ci['specificity']['mean'], tabnet_ci['specificity']['mean'],
                    ensemble_ci['specificity']['mean']],
    'F1': [xgb_ci['f1']['mean'], lgb_ci['f1']['mean'],
           rf_ci['f1']['mean'], tabnet_ci['f1']['mean'],
           ensemble_ci['f1']['mean']]
})

comparison_df = comparison_df.sort_values('AUC-ROC', ascending=False)
comparison_df.to_csv(os.path.join(TASK_OUTPUT, 'model_comparison.csv'), index=False)

logger.info("\n" + comparison_df[['Model', 'AUC-ROC', 'Sensitivity', 'Specificity', 'F1']].to_string(index=False))

# ============================================================================
# 13. VISUALIZATIONS
# ============================================================================
logger.info("\n📈 Creating Performance Visualizations")

# ROC Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    fpr, tpr, _ = roc_curve(y_test, preds)
    auc = roc_auc_score(y_test, preds)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Hosp-LOS >7 Days Prediction', fontsize=14)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'roc_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: roc_curves.png")

# PR Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    precision, recall, _ = precision_recall_curve(y_test, preds)
    auprc = average_precision_score(y_test, preds)
    plt.plot(recall, precision, label=f'{name} (AUPRC={auprc:.4f})', linewidth=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves - Hosp-LOS >7 Days Prediction', fontsize=14)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'pr_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: pr_curves.png")

# Calibration Curves
plt.figure(figsize=(12, 8))
for name, preds in [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), 
                     ('Random Forest', rf_pred), ('TabNet', tabnet_pred),
                     ('Voting Ensemble', ensemble_pred)]:
    prob_true, prob_pred = calibration_curve(y_test, preds, n_bins=10)
    ece = np.mean(np.abs(prob_true - prob_pred))
    plt.plot(prob_pred, prob_true, marker='o', label=f'{name} (ECE={ece:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curves - Hosp-LOS >7 Days Prediction', fontsize=14)
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.savefig(os.path.join(TASK_OUTPUT, 'calibration_curves.png'), dpi=300)
plt.close()
logger.info("   ✅ Saved: calibration_curves.png")

# ============================================================================
# 14. SAVE PREDICTIONS
# ============================================================================
predictions_df = pd.DataFrame({
    'XGBoost': xgb_pred,
    'LightGBM': lgb_pred,
    'RandomForest': rf_pred,
    'TabNet': tabnet_pred,
    'VotingEnsemble': ensemble_pred,
    'y_true': y_test
})
predictions_df.to_csv(os.path.join(TASK_OUTPUT, 'all_predictions.csv'), index=False)
logger.info("   ✅ Saved: all_predictions.csv")

# ============================================================================
# 15. FINAL SUMMARY
# ============================================================================
logger.info("\n" + "="*80)
logger.info("✅ TASK 3 COMPLETE")
logger.info("="*80)
logger.info(f"Best Model: {comparison_df.iloc[0]['Model']} (AUC={comparison_df.iloc[0]['AUC-ROC']:.4f})")
logger.info(f"\n📁 SHAP Output Folders (ALL with COMPLETE plots):")
logger.info(f"   {TASK_OUTPUT}/")
logger.info(f"   ├── xgboost_shap/        (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── lightgbm_shap/       (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── randomforest_shap/   (SHAP bar, summary, waterfall, dependence)")
logger.info(f"   ├── tabnet_shap/         (SHAP bar, summary, waterfall, dependence - KernelExplainer)")
logger.info(f"   ├── votingensemble_shap/ (SHAP bar, summary, waterfall, dependence - Surrogate RF)")
logger.info(f"   ├── model_comparison.csv")
logger.info(f"   ├── roc_curves.png")
logger.info(f"   ├── pr_curves.png")
logger.info(f"   ├── calibration_curves.png")
logger.info(f"   └── all_predictions.csv")
logger.info("="*80)

2026-07-24 15:47:04 | INFO | ================================================================================
2026-07-24 15:47:04 | INFO | 🎯 TASK 3: HOSP-LOS >7 DAYS PREDICTION
2026-07-24 15:47:04 | INFO | Models: XGBoost, LightGBM, RandomForest, TabNet, VotingEnsemble
2026-07-24 15:47:04 | INFO | SHAP analysis for ALL 5 models (each gets FULL SHAP plots)
2026-07-24 15:47:04 | INFO | TabNet: Using KernelExplainer approximation
2026-07-24 15:47:04 | INFO | Voting Ensemble: Surrogate RandomForest + TreeExplainer
2026-07-24 15:47:04 | INFO | ================================================================================
2026-07-24 15:47:05 | INFO | 
📊 DATA LOADED
2026-07-24 15:47:05 | INFO |    Train: (32733, 29), Prevalence: 16.5%
2026-07-24 15:47:05 | INFO |    Val: (4820, 29), Prevalence: 18.8%
2026-07-24 15:47:05 | INFO |    Test: (4765, 29), Prevalence: 19.7%
2026-07-24 15:47:05 | INFO | 
📈 PREPROCESSING
2026-07-24 15:47:05 | INFO |    Features: 29
2026-07-24 15:47:05 | INFO | 
2026

2026-07-24 15:47:31 | INFO |       ✅ Saved: shap_force_plot.png
2026-07-24 15:47:32 | INFO |       ✅ Saved: shap_dependence_Triage Systolic BP.png
2026-07-24 15:47:32 | INFO |       ✅ Saved: shap_dependence_Triage Heart Rate.png
2026-07-24 15:47:33 | INFO |       ✅ Saved: shap_dependence_Age.png
2026-07-24 15:47:33 | INFO |       ✅ Saved: shap_dependence_Home Medication Count.png
2026-07-24 15:47:34 | INFO |       ✅ Saved: shap_dependence_Triage Acuity Level.png
2026-07-24 15:47:34 | INFO | 
   📊 XGBoost Top 10 Features:
2026-07-24 15:47:34 | INFO |       12. Triage Systolic BP: 0.2391
2026-07-24 15:47:34 | INFO |       9. Triage Heart Rate: 0.2146
2026-07-24 15:47:34 | INFO |       1. Age: 0.2145
2026-07-24 15:47:34 | INFO |       27. Home Medication Count: 0.2072
2026-07-24 15:47:34 | INFO |       14. Triage Acuity Level: 0.1625
2026-07-24 15:47:34 | INFO |       28. Unique Medication Classes Count: 0.1577
2026-07-24 15:47:34 | INFO |       26. Disease Condition Count: 0.1557
2026-07

2026-07-24 15:47:53 | INFO |       ✅ Saved: shap_force_plot.png
2026-07-24 15:47:54 | INFO |       ✅ Saved: shap_dependence_Triage Acuity Level.png
2026-07-24 15:47:54 | INFO |       ✅ Saved: shap_dependence_Triage Systolic BP.png
2026-07-24 15:47:55 | INFO |       ✅ Saved: shap_dependence_Triage Heart Rate.png
2026-07-24 15:47:56 | INFO |       ✅ Saved: shap_dependence_Disease Condition Count.png
2026-07-24 15:47:56 | INFO |       ✅ Saved: shap_dependence_CC: Chest Pain.png
2026-07-24 15:47:56 | INFO | 
   📊 LightGBM Top 10 Features:
2026-07-24 15:47:56 | INFO |       14. Triage Acuity Level: 0.0427
2026-07-24 15:47:56 | INFO |       12. Triage Systolic BP: 0.0325
2026-07-24 15:47:56 | INFO |       9. Triage Heart Rate: 0.0271
2026-07-24 15:47:56 | INFO |       26. Disease Condition Count: 0.0144
2026-07-24 15:47:56 | INFO |       20. CC: Chest Pain: 0.0127
2026-07-24 15:47:56 | INFO |       1. Age: 0.0122
2026-07-24 15:47:56 | INFO |       11. Triage SpO2: 0.0119
2026-07-24 15:47:56 


Early stopping occurred at epoch 86 with best_epoch = 66 and best_val_auc = 0.64399


2026-07-24 15:57:51 | INFO | 
📊 TabNet Results:
2026-07-24 15:57:51 | INFO |    AUC-ROC: 0.6425 [0.6223-0.6608]
2026-07-24 15:57:51 | INFO |    Sensitivity: 56.3%
2026-07-24 15:57:51 | INFO | 
   🔍 Computing SHAP values for TabNet (KernelExplainer)...
2026-07-24 15:57:51 | INFO |       Creating KernelExplainer with 100 background samples...
2026-07-24 15:57:51 | INFO |       Computing SHAP values for 200 samples (this may take a few minutes)...


  0%|          | 0/200 [00:00<?, ?it/s]

2026-07-24 15:57:52 | INFO | num_full_subsets = 1
2026-07-24 15:57:52 | INFO | remaining_weight_vector = array([0.19176421, 0.13316959, 0.10421968, 0.08716555, 0.07609691,
       0.06848722, 0.06308033, 0.05918649, 0.05640124, 0.05447847,
       0.05326784, 0.05268248])
2026-07-24 15:57:52 | INFO | num_paired_subset_sizes = 13
2026-07-24 15:57:52 | INFO | weight_left = np.float64(0.730579019691897)
2026-07-24 15:57:55 | INFO | np.sum(w_aug) = np.float64(27.000000000000007)
2026-07-24 15:57:55 | INFO | np.sum(self.kernelWeights) = np.float64(1.0000000000000004)
2026-07-24 15:57:55 | INFO | phi = array([-0.00808399, -0.01726215,  0.        ,  0.        ,  0.        ,
       -0.00272113,  0.        , -0.06430948, -0.00539094,  0.        ,
       -0.055624  , -0.01364481, -0.00932987,  0.        ,  0.        ,
        0.        , -0.01206592,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        , -0.00805819,  0.        ,
        0.        ,  0.        ])
2026

In [14]:
# CELL 8 - COMBINE ALL TASK RESULTS INTO SINGLE PAPER RESULTS FILE
# Combines model_comparison.csv from Task 1 (ED-LOS), Task 2 (Admission), Task 3 (Hosp-LOS)

import os
import pandas as pd
import numpy as np
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"

# Task output paths
TASK_PATHS = {
    'ED-LOS (≥8 hours)': os.path.join(OUTPUT_PATH, 'task1_ed_los', 'model_comparison.csv'),
    'Admission (Binary)': os.path.join(OUTPUT_PATH, 'task2_admission', 'model_comparison.csv'),
    'Hosp-LOS (>7 days)': os.path.join(OUTPUT_PATH, 'task3_hosp_los', 'model_comparison.csv')
}

logger.info("="*80)
logger.info("📊 COMBINING RESULTS FROM ALL THREE TASKS")
logger.info("="*80)

# ============================================================================
# 1. LOAD ALL MODEL COMPARISON FILES
# ============================================================================
all_results = {}

for task_name, file_path in TASK_PATHS.items():
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        all_results[task_name] = df
        logger.info(f"✅ Loaded {task_name}: {df.shape[0]} models, {df.shape[1]} metrics")
    else:
        logger.warning(f"⚠️ File not found: {file_path}")

# ============================================================================
# 2. CREATE COMPREHENSIVE RESULTS TABLE
# ============================================================================

# Define the order of models (consistent across all tasks)
model_order = ['XGBoost', 'LightGBM', 'Random Forest', 'TabNet', 'Voting Ensemble']

# Create a list to store all rows
all_rows = []

for task_name, df in all_results.items():
    
    # Ensure Model column exists and set as index
    if 'Model' not in df.columns:
        logger.warning(f"⚠️ No 'Model' column in {task_name}")
        continue
    
    # Sort by model order
    df['Model_Order'] = df['Model'].map({m: i for i, m in enumerate(model_order)})
    df = df.sort_values('Model_Order').drop('Model_Order', axis=1)
    
    for _, row in df.iterrows():
        model_name = row['Model']
        
        # Base row with Task and Model
        base_row = {
            'Task': task_name,
            'Model': model_name,
        }
        
        # Add all available metrics
        for col in df.columns:
            if col != 'Model':
                value = row[col]
                
                # Format confidence intervals nicely
                if col == 'AUC-ROC_CI' and isinstance(value, str):
                    base_row['AUC-ROC (95% CI)'] = value
                elif col == 'AUC-ROC_CI_lower' and 'AUC-ROC_CI_upper' in df.columns:
                    # Already handled by AUC-ROC_CI
                    continue
                elif col == 'AUC-ROC_CI_upper':
                    continue
                elif col == 'AUC-ROC' and 'AUC-ROC_CI' in df.columns:
                    # Use CI version instead
                    continue
                else:
                    # Format numeric values with 4 decimal places
                    if pd.api.types.is_numeric_dtype(type(value)):
                        base_row[col] = round(float(value), 4)
                    else:
                        base_row[col] = value
        
        all_rows.append(base_row)

# Convert to DataFrame
combined_df = pd.DataFrame(all_rows)

# ============================================================================
# 3. CREATE PIVOT TABLE FOR BETTER READABILITY (Models as rows, Tasks as columns)
# ============================================================================

# Metrics to include in the final output
key_metrics = ['AUC-ROC', 'AUC-ROC (95% CI)', 'AUPRC', 'Sensitivity', 'Specificity', 'F1', 'Brier', 'MCC']

# Create a pivot table for each metric
pivot_tables = {}

for metric in key_metrics:
    if metric in combined_df.columns:
        # Filter rows that have this metric
        metric_df = combined_df[['Task', 'Model', metric]].dropna()
        
        # Pivot: Models as rows, Tasks as columns
        pivot = metric_df.pivot(index='Model', columns='Task', values=metric)
        
        # Reorder columns (tasks) and rows (models)
        task_order = ['ED-LOS (≥8 hours)', 'Admission (Binary)', 'Hosp-LOS (>7 days)']
        pivot = pivot[[t for t in task_order if t in pivot.columns]]
        pivot = pivot.reindex([m for m in model_order if m in pivot.index])
        
        pivot_tables[metric] = pivot

# ============================================================================
# 4. CREATE A SINGLE COMBINED RESULTS TABLE (Wide format)
# ============================================================================

# Build a multi-level column DataFrame
combined_wide = pd.DataFrame(index=model_order)

for task_name in task_order:
    if task_name not in all_results:
        continue
    
    task_df = all_results[task_name].set_index('Model')
    task_df = task_df.reindex(model_order)
    
    for metric in ['AUC-ROC', 'AUPRC', 'Sensitivity', 'Specificity', 'F1']:
        if metric in task_df.columns:
            col_name = f"{task_name} - {metric}"
            combined_wide[col_name] = task_df[metric].values
        
        # Add CI if available in separate columns
        if 'AUC-ROC_CI_lower' in task_df.columns and 'AUC-ROC_CI_upper' in task_df.columns:
            ci_col = f"{task_name} - AUC-ROC (95% CI)"
            combined_wide[ci_col] = task_df.apply(
                lambda x: f"[{x['AUC-ROC_CI_lower']:.4f}-{x['AUC-ROC_CI_upper']:.4f}]" 
                if pd.notna(x['AUC-ROC_CI_lower']) else "",
                axis=1
            )
        elif 'AUC-ROC_CI' in task_df.columns:
            ci_col = f"{task_name} - AUC-ROC (95% CI)"
            combined_wide[ci_col] = task_df['AUC-ROC_CI']

# ============================================================================
# 5. CREATE SUMMARY STATISTICS TABLE
# ============================================================================
summary_rows = []

for task_name, df in all_results.items():
    best_auc_row = df.loc[df['AUC-ROC'].idxmax()] if 'AUC-ROC' in df.columns else None
    best_f1_row = df.loc[df['F1'].idxmax()] if 'F1' in df.columns else None
    
    if best_auc_row is not None:
        summary_rows.append({
            'Task': task_name,
            'Best Model (AUC)': best_auc_row['Model'],
            'Best AUC': round(best_auc_row['AUC-ROC'], 4),
            'Best Model (F1)': best_f1_row['Model'] if best_f1_row is not None else 'N/A',
            'Best F1': round(best_f1_row['F1'], 4) if best_f1_row is not None else 'N/A',
            'Models Compared': len(df)
        })

summary_df = pd.DataFrame(summary_rows)

# ============================================================================
# 6. SAVE ALL OUTPUTS
# ============================================================================

# Save combined results (long format)
combined_output_path = os.path.join(OUTPUT_PATH, 'paper_results_long.csv')
combined_df.to_csv(combined_output_path, index=False)
logger.info(f"\n💾 Saved: paper_results_long.csv")

# Save combined results (wide format - best for papers)
wide_output_path = os.path.join(OUTPUT_PATH, 'paper_results_wide.csv')
combined_wide.to_csv(wide_output_path)
logger.info(f"💾 Saved: paper_results_wide.csv")

# Save summary statistics
summary_output_path = os.path.join(OUTPUT_PATH, 'paper_results_summary.csv')
summary_df.to_csv(summary_output_path, index=False)
logger.info(f"💾 Saved: paper_results_summary.csv")

# Save pivot tables for each key metric
for metric, pivot in pivot_tables.items():
    pivot_path = os.path.join(OUTPUT_PATH, f'paper_results_{metric.lower().replace(" ", "_").replace("(", "").replace(")", "")}.csv')
    pivot.to_csv(pivot_path)
    logger.info(f"💾 Saved: paper_results_{metric.lower().replace(' ', '_')}.csv")

# ============================================================================
# 7. CREATE A MARKDOWN/TEXT SUMMARY FOR QUICK REFERENCE
# ============================================================================
markdown_lines = []
markdown_lines.append("# Paper Results Summary")
markdown_lines.append(f"\n*Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*")
markdown_lines.append("\n## Task Overview")
markdown_lines.append("\n| Task | Positive Class | Prevalence (Train) | Prevalence (Test) |")
markdown_lines.append("|------|---------------|-------------------|------------------|")

task_prevalence = {
    'ED-LOS (≥8 hours)': {'train': 0.190, 'test': 0.223},
    'Admission (Binary)': {'train': 0.401, 'test': 0.396},
    'Hosp-LOS (>7 days)': {'train': 0.165, 'test': 0.197}
}

for task_name, prev in task_prevalence.items():
    markdown_lines.append(f"| {task_name} | 1 = positive case | {prev['train']:.1%} | {prev['test']:.1%} |")

markdown_lines.append("\n## Best Model Performance by Task")
markdown_lines.append("\n| Task | Best Model | AUC-ROC | Sensitivity | Specificity | F1 |")
markdown_lines.append("|------|------------|---------|-------------|-------------|-----|")

for _, row in summary_df.iterrows():
    task = row['Task']
    task_df = all_results[task]
    best_model_row = task_df.loc[task_df['AUC-ROC'].idxmax()]
    
    markdown_lines.append(
        f"| {task} | {best_model_row['Model']} | "
        f"{best_model_row['AUC-ROC']:.4f} | "
        f"{best_model_row['Sensitivity']:.1%} | "
        f"{best_model_row['Specificity']:.1%} | "
        f"{best_model_row['F1']:.4f} |"
    )

markdown_lines.append("\n## Full Results Table")
markdown_lines.append("\n```")
markdown_lines.append(combined_wide.round(4).to_string())
markdown_lines.append("```")

# Save markdown
markdown_path = os.path.join(OUTPUT_PATH, 'paper_results_summary.md')
with open(markdown_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(markdown_lines))
logger.info(f"💾 Saved: paper_results_summary.md")

# ============================================================================
# 8. CREATE AN EXCEL FILE WITH MULTIPLE SHEETS (BEST FOR PAPERS)
# ============================================================================
try:
    excel_path = os.path.join(OUTPUT_PATH, 'paper_results_complete.xlsx')
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: Summary
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        
        # Sheet 2: Wide format (all metrics side by side)
        combined_wide.to_excel(writer, sheet_name='All_Metrics_Wide')
        
        # Sheet 3: Long format
        combined_df.to_excel(writer, sheet_name='All_Metrics_Long', index=False)
        
        # Sheet 4-6: Individual task results
        for task_name, df in all_results.items():
            sheet_name = task_name.replace(' ', '_')[:31]  # Excel sheet name max 31 chars
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Sheet 7: Pivot tables
        for metric, pivot in pivot_tables.items():
            sheet_name = f'Pivot_{metric}'[:31]
            pivot.to_excel(writer, sheet_name=sheet_name)
    
    logger.info(f"💾 Saved: paper_results_complete.xlsx (Excel workbook with multiple sheets)")
except Exception as e:
    logger.warning(f"Could not create Excel file: {e}")

# ============================================================================
# 9. PRINT FINAL SUMMARY
# ============================================================================
logger.info("\n" + "="*80)
logger.info("📊 FINAL PAPER RESULTS SUMMARY")
logger.info("="*80)

print("\n" + "="*80)
print("PAPER RESULTS - COMPLETE SUMMARY")
print("="*80)

for task_name, df in all_results.items():
    print(f"\n📌 {task_name}")
    print("-" * 50)
    
    # Sort by AUC-ROC descending
    df_sorted = df.sort_values('AUC-ROC', ascending=False)
    
    for _, row in df_sorted.iterrows():
        print(f"   {row['Model']:15s} | AUC: {row['AUC-ROC']:.4f} | "
              f"Sens: {row['Sensitivity']:.1%} | Spec: {row['Specificity']:.1%} | F1: {row['F1']:.4f}")

print("\n" + "="*80)
print("✅ All results combined and saved to:")
print(f"   📁 {OUTPUT_PATH}")
print("\n   Generated files:")
print("   1. paper_results_long.csv - All results in long format")
print("   2. paper_results_wide.csv - Results side-by-side by task")
print("   3. paper_results_summary.csv - Best model per task")
print("   4. paper_results_summary.md - Markdown summary for reports")
print("   5. paper_results_complete.xlsx - Excel workbook with all sheets")
print("   6. paper_results_*.csv - Pivot tables for each metric")
print("="*80)

# ============================================================================
# 10. VERIFICATION CHECK
# ============================================================================
logger.info("\n📋 VERIFICATION CHECK:")
logger.info(f"   Total unique models: {combined_df['Model'].nunique()}")
logger.info(f"   Total tasks: {combined_df['Task'].nunique()}")
logger.info(f"   Total metric rows: {len(combined_df)}")
logger.info(f"   Metrics available: {[c for c in combined_df.columns if c not in ['Task', 'Model']]}")

2026-07-24 16:09:45 | INFO | ================================================================================
2026-07-24 16:09:45 | INFO | 📊 COMBINING RESULTS FROM ALL THREE TASKS
2026-07-24 16:09:45 | INFO | ================================================================================
2026-07-24 16:09:45 | INFO | ✅ Loaded ED-LOS (≥8 hours): 5 models, 7 metrics
2026-07-24 16:09:46 | INFO | ✅ Loaded Admission (Binary): 5 models, 7 metrics
2026-07-24 16:09:46 | INFO | ✅ Loaded Hosp-LOS (>7 days): 5 models, 7 metrics
2026-07-24 16:09:46 | INFO | 
💾 Saved: paper_results_long.csv
2026-07-24 16:09:46 | INFO | 💾 Saved: paper_results_wide.csv
2026-07-24 16:09:46 | INFO | 💾 Saved: paper_results_summary.csv
2026-07-24 16:09:46 | INFO | 💾 Saved: paper_results_auc-roc.csv
2026-07-24 16:09:46 | INFO | 💾 Saved: paper_results_sensitivity.csv
2026-07-24 16:09:46 | INFO | 💾 Saved: paper_results_specificity.csv
2026-07-24 16:09:46 | INFO | 💾 Saved: paper_results_f1.csv
2026-07-24 16:09:46 | INFO | 💾 


PAPER RESULTS - COMPLETE SUMMARY

📌 ED-LOS (≥8 hours)
--------------------------------------------------
   Random Forest   | AUC: 0.6603 | Sens: 75.9% | Spec: 47.2% | F1: 0.4214
   Voting Ensemble | AUC: 0.6603 | Sens: 66.6% | Spec: 56.2% | F1: 0.4165
   XGBoost         | AUC: 0.6594 | Sens: 75.1% | Spec: 47.7% | F1: 0.4202
   LightGBM        | AUC: 0.6479 | Sens: 26.1% | Spec: 86.3% | F1: 0.3002
   TabNet          | AUC: 0.6385 | Sens: 73.3% | Spec: 48.1% | F1: 0.4135

📌 Admission (Binary)
--------------------------------------------------
   Voting Ensemble | AUC: 0.8184 | Sens: 70.8% | Spec: 75.9% | F1: 0.6817
   LightGBM        | AUC: 0.8168 | Sens: 70.5% | Spec: 76.4% | F1: 0.6825
   XGBoost         | AUC: 0.8163 | Sens: 70.0% | Spec: 76.4% | F1: 0.6794
   TabNet          | AUC: 0.8150 | Sens: 72.4% | Spec: 74.7% | F1: 0.6860
   Random Forest   | AUC: 0.8122 | Sens: 69.6% | Spec: 76.1% | F1: 0.6761

📌 Hosp-LOS (>7 days)
--------------------------------------------------
   Votin

In [15]:
# ESI_Baseline_Evaluation.py
# Purpose: Evaluate ESI triage acuity as a baseline predictor for all three tasks

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, roc_curve
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# Paths
BASE_PATH = Path(r"E:\TSINGHUA\thisis\Paper\output_paper")
OUTPUT_PATH = BASE_PATH / "esi_baseline_results"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Tasks definition
TASKS = {
    'ED_LOS': {
        'display': 'ED Stay ≥8h',
        'feature_file': 'X_test.pkl',
        'target_file': 'y_test_admit.pkl',
        'target_column': 'TARGET_ED_LOS_over8h'
    },
    'Admission': {
        'display': 'Hospital Admission',
        'feature_file': 'X_test.pkl',
        'target_file': 'y_test_admit.pkl',
        'target_column': 'TARGET_Admitted'
    },
    'Hosp_LOS': {
        'display': 'Hospital Stay >7d',
        'feature_file': 'X_test_hosp.pkl',
        'target_file': 'y_test_hosp.pkl',
        'target_column': 'TARGET_Hosp_LOS_over7d'
    }
}

def evaluate_esi_baseline(y_true, X_test):
    """
    ESI baseline: use triage_acuity (1-5) as predictor
    Lower ESI = higher acuity = higher probability
    """
    if 'triage_acuity' in X_test.columns:
        esi_scores = X_test['triage_acuity'].values
    else:
        logger.warning("triage_acuity not found, using synthetic ESI")
        esi_scores = np.random.randint(1, 6, len(y_true))
    
    # Convert ESI 1-5 to probability (1->1.0, 5->0.0)
    esi_proba = 1 - (esi_scores - 1) / 4
    esi_proba = np.clip(esi_proba, 0.01, 0.99)
    
    # Find optimal threshold
    fpr, tpr, thresholds = roc_curve(y_true, esi_proba)
    youden_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[youden_idx] if youden_idx < len(thresholds) else 0.5
    
    y_pred_binary = (esi_proba >= optimal_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary, labels=[0, 1]).ravel()
    
    metrics = {
        'auroc': roc_auc_score(y_true, esi_proba),
        'auprc': average_precision_score(y_true, esi_proba),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1': f1_score(y_true, y_pred_binary),
        'optimal_threshold': optimal_threshold,
        'n_positive': tp + fn,
        'n_negative': tn + fp
    }
    
    return metrics

def main():
    logger.info("="*60)
    logger.info("ESI BASELINE EVALUATION")
    logger.info("="*60)
    
    all_results = []
    
    for task_key, task_info in TASKS.items():
        logger.info(f"\n📌 {task_info['display']}")
        
        # Load data
        X_test = pd.read_pickle(BASE_PATH / task_info['feature_file'])
        y_df = pd.read_pickle(BASE_PATH / task_info['target_file'])
        y_true = y_df[task_info['target_column']].values.astype(int)
        
        # Remove NaN
        valid_mask = ~np.isnan(y_true)
        y_true = y_true[valid_mask].astype(int)
        X_test = X_test[valid_mask]
        
        # Evaluate ESI
        metrics = evaluate_esi_baseline(y_true, X_test)
        metrics['task'] = task_info['display']
        all_results.append(metrics)
        
        logger.info(f"   AUC-ROC: {metrics['auroc']:.4f}")
        logger.info(f"   Sensitivity: {metrics['sensitivity']:.1%}")
        logger.info(f"   Specificity: {metrics['specificity']:.1%}")
        logger.info(f"   F1: {metrics['f1']:.4f}")
    
    # Save results
    df_results = pd.DataFrame(all_results)
    df_results.to_csv(OUTPUT_PATH / "esi_baseline_results.csv", index=False)
    logger.info(f"\n✅ Results saved to: {OUTPUT_PATH}/esi_baseline_results.csv")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-07-24 16:09:50 | INFO | ============================================================
2026-07-24 16:09:50 | INFO | ESI BASELINE EVALUATION
2026-07-24 16:09:50 | INFO | ============================================================
2026-07-24 16:09:50 | INFO | 
📌 ED Stay ≥8h
2026-07-24 16:09:50 | WARNING | triage_acuity not found, using synthetic ESI
2026-07-24 16:09:50 | INFO |    AUC-ROC: 0.5043
2026-07-24 16:09:50 | INFO |    Sensitivity: 20.7%
2026-07-24 16:09:50 | INFO |    Specificity: 80.3%
2026-07-24 16:09:50 | INFO |    F1: 0.2184
2026-07-24 16:09:50 | INFO | 
📌 Hospital Admission
2026-07-24 16:09:50 | WARNING | triage_acuity not found, using synthetic ESI
2026-07-24 16:09:50 | INFO |    AUC-ROC: 0.4986
2026-07-24 16:09:50 | INFO |    Sensitivity: 80.2%
2026-07-24 16:09:50 | INFO |    Specificity: 19.9%
2026-07-24 16:09:50 | INFO |    F1: 0.5309
2026-07-24 16:09:50 | INFO | 
📌 Hospital Stay >7d
2026-07-24 16:09:50 | WARNING | triage_acuity not found, using synthetic ESI
2026-

In [16]:
# CELL 8 - DEMOGRAPHIC BIAS ANALYSIS (Loads all_predictions.csv directly)
# No need to load models - predictions are already saved

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# Paths
BASE_PATH = Path(r"E:\TSINGHUA\thisis\Paper\output_paper")
OUTPUT_PATH = BASE_PATH / "demographic_bias_analysis"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Task configuration with their respective folders and all_predictions.csv files
TASKS = {
    'Admission': {
        'display': 'Hospital Admission',
        'folder': 'task2_admission',
        'predictions_file': 'all_predictions.csv',
        'target_column': 'TARGET_Admitted'  # for verification
    },
    'ED_LOS': {
        'display': 'ED Stay ≥8h',
        'folder': 'task1_ed_los',
        'predictions_file': 'all_predictions.csv',
        'target_column': 'TARGET_ED_LOS_over8h'
    },
    'Hosp_LOS': {
        'display': 'Hospital Stay >7d',
        'folder': 'task3_hosp_los',
        'predictions_file': 'all_predictions.csv',
        'target_column': 'TARGET_Hosp_LOS_over7d'
    }
}

# Model to analyze (change as needed)
MODEL_TO_ANALYZE = 'VotingEnsemble'  # Options: 'XGBoost', 'LightGBM', 'RandomForest', 'TabNet', 'VotingEnsemble'

def load_demographics():
    """Load demographics from test data"""
    test_raw = pd.read_pickle(BASE_PATH / 'test_data' / 'visits.pkl')
    
    demographics = pd.DataFrame({
        'race': test_raw['Race'].fillna('Unknown'),
        'gender': test_raw['Gender'].fillna('Unknown'),
        'age': test_raw['Age'].fillna(50),
        'ethnicity': test_raw['Ethnicity'].fillna('Unknown'),
        'insurance': test_raw['Payor_class'].fillna('Unknown')
    })
    
    # Clean race categories
    race_map = {
        'White': 'White',
        'Black or African American': 'Black',
        'Hispanic/Latino': 'Hispanic',
        'Asian': 'Asian',
        'Other': 'Other',
        'Unknown': 'Unknown'
    }
    demographics['race_clean'] = demographics['race'].map(lambda x: race_map.get(x, 'Other'))
    
    # Create age groups
    demographics['age_group'] = pd.cut(
        demographics['age'], 
        bins=[0, 40, 65, 80, 120], 
        labels=['18-40', '41-64', '65-79', '80+']
    )
    
    # Clean gender
    demographics['gender_clean'] = demographics['gender'].map({'F': 'Female', 'M': 'Male'}).fillna('Unknown')
    
    return demographics

def calculate_subgroup_metrics(y_true, y_pred_proba, subgroups, group_col, group_name):
    """Calculate metrics for a specific subgroup"""
    
    mask = subgroups[group_col] == group_name
    if mask.sum() < 30:
        return None
    
    y_true_sub = y_true[mask]
    y_pred_sub = y_pred_proba[mask]
    
    if len(np.unique(y_true_sub)) < 2:
        return None
    
    # Find optimal threshold using Youden's index
    fpr, tpr, thresholds = roc_curve(y_true_sub, y_pred_sub)
    youden_idx = np.argmax(tpr - fpr)
    threshold = thresholds[youden_idx] if youden_idx < len(thresholds) else 0.5
    
    y_pred_binary = (y_pred_sub >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_sub, y_pred_binary, labels=[0, 1]).ravel()
    
    return {
        'group': group_name,
        'n': mask.sum(),
        'prevalence': y_true_sub.mean(),
        'optimal_threshold': threshold,
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'auroc': roc_auc_score(y_true_sub, y_pred_sub),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn
    }

def analyze_task(task_name, task_config, demographics, model_name):
    """Complete bias analysis for a single task"""
    
    logger.info(f"\n{'='*80}")
    logger.info(f"TASK: {task_config['display']}")
    logger.info(f"Model: {model_name}")
    logger.info(f"{'='*80}")
    
    # Load predictions
    pred_file = BASE_PATH / task_config['folder'] / task_config['predictions_file']
    
    if not pred_file.exists():
        logger.error(f"   File not found: {pred_file}")
        return None
    
    pred_df = pd.read_csv(pred_file)
    logger.info(f"   ✅ Loaded predictions: {len(pred_df)} samples")
    
    # Check if model exists in predictions
    if model_name not in pred_df.columns:
        logger.error(f"   Model '{model_name}' not found in predictions. Available: {list(pred_df.columns)}")
        return None
    
    y_true = pred_df['y_true'].values
    y_pred_proba = pred_df[model_name].values
    
    logger.info(f"   Prevalence: {y_true.mean():.1%}")
    logger.info(f"   Overall AUC: {roc_auc_score(y_true, y_pred_proba):.4f}")
    
    # Align demographics with predictions
    demo_aligned = demographics.iloc[:len(y_true)].copy()
    
    results = {}
    
    # ================================================================
    # 1. Race Analysis
    # ================================================================
    logger.info(f"\n   📊 RACE ANALYSIS")
    race_results = []
    for race in ['White', 'Black', 'Hispanic', 'Asian', 'Other']:
        metrics = calculate_subgroup_metrics(y_true, y_pred_proba, demo_aligned, 'race_clean', race)
        if metrics:
            race_results.append(metrics)
            logger.info(f"      {race:10s} (n={metrics['n']:5d}): Sens={metrics['sensitivity']:.1%}, AUC={metrics['auroc']:.4f}")
    
    df_race = pd.DataFrame(race_results)
    if len(df_race) > 0:
        df_race.to_csv(OUTPUT_PATH / f"{task_name}_race_bias.csv", index=False)
        
        # Calculate disparities
        white_sens = df_race[df_race['group'] == 'White']['sensitivity'].values
        if len(white_sens) > 0:
            white_sens = white_sens[0]
            logger.info(f"\n      Disparities (White - Others):")
            for _, row in df_race.iterrows():
                if row['group'] != 'White':
                    disparity = white_sens - row['sensitivity']
                    logger.info(f"         White - {row['group']:10s}: {disparity:.1%}")
    
    results['race'] = df_race
    
    # ================================================================
    # 2. Age Group Analysis
    # ================================================================
    logger.info(f"\n   📊 AGE GROUP ANALYSIS")
    age_results = []
    for age_group in ['18-40', '41-64', '65-79', '80+']:
        metrics = calculate_subgroup_metrics(y_true, y_pred_proba, demo_aligned, 'age_group', age_group)
        if metrics:
            age_results.append(metrics)
            logger.info(f"      {age_group:8s} (n={metrics['n']:5d}): Sens={metrics['sensitivity']:.1%}, AUC={metrics['auroc']:.4f}")
    
    df_age = pd.DataFrame(age_results)
    if len(df_age) > 0:
        df_age.to_csv(OUTPUT_PATH / f"{task_name}_age_bias.csv", index=False)
    
    results['age'] = df_age
    
    # ================================================================
    # 3. Gender Analysis
    # ================================================================
    logger.info(f"\n   📊 GENDER ANALYSIS")
    gender_results = []
    for gender in ['Female', 'Male']:
        metrics = calculate_subgroup_metrics(y_true, y_pred_proba, demo_aligned, 'gender_clean', gender)
        if metrics:
            gender_results.append(metrics)
            logger.info(f"      {gender:8s} (n={metrics['n']:5d}): Sens={metrics['sensitivity']:.1%}, AUC={metrics['auroc']:.4f}")
    
    df_gender = pd.DataFrame(gender_results)
    if len(df_gender) > 0:
        df_gender.to_csv(OUTPUT_PATH / f"{task_name}_gender_bias.csv", index=False)
        
        # Calculate gender disparity
        if len(df_gender) == 2:
            female_sens = df_gender[df_gender['group'] == 'Female']['sensitivity'].values[0]
            male_sens = df_gender[df_gender['group'] == 'Male']['sensitivity'].values[0]
            logger.info(f"\n      Gender Disparity (Female - Male): {female_sens - male_sens:.1%}")
    
    results['gender'] = df_gender
    
    # ================================================================
    # 4. Summary Table
    # ================================================================
    logger.info(f"\n   📋 SUMMARY TABLE")
    
    summary_rows = []
    for _, row in df_race.iterrows():
        summary_rows.append({
            'Subgroup': row['group'],
            'Type': 'Race',
            'N': row['n'],
            'Sensitivity': f"{row['sensitivity']:.1%}",
            'AUC-ROC': f"{row['auroc']:.3f}",
            'Threshold': f"{row['optimal_threshold']:.2f}"
        })
    
    for _, row in df_age.iterrows():
        summary_rows.append({
            'Subgroup': row['group'],
            'Type': 'Age',
            'N': row['n'],
            'Sensitivity': f"{row['sensitivity']:.1%}",
            'AUC-ROC': f"{row['auroc']:.3f}",
            'Threshold': f"{row['optimal_threshold']:.2f}"
        })
    
    for _, row in df_gender.iterrows():
        summary_rows.append({
            'Subgroup': row['group'],
            'Type': 'Gender',
            'N': row['n'],
            'Sensitivity': f"{row['sensitivity']:.1%}",
            'AUC-ROC': f"{row['auroc']:.3f}",
            'Threshold': f"{row['optimal_threshold']:.2f}"
        })
    
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(OUTPUT_PATH / f"{task_name}_bias_summary.csv", index=False)
    
    print("\n" + df_summary.to_string(index=False))
    
    return results

def create_manuscript_table(all_results):
    """Create a combined table for manuscript"""
    
    rows = []
    for task_name, results in all_results.items():
        if results and 'race' in results and len(results['race']) > 0:
            df_race = results['race']
            white_sens = df_race[df_race['group'] == 'White']['sensitivity'].values
            white_sens = white_sens[0] if len(white_sens) > 0 else 0
            
            for _, row in df_race.iterrows():
                if row['group'] != 'White':
                    rows.append({
                        'Task': task_name,
                        'Demographic': 'Race',
                        'Group': row['group'],
                        'N': row['n'],
                        'Sensitivity': f"{row['sensitivity']:.1%}",
                        'Disparity_vs_White': f"{white_sens - row['sensitivity']:.1%}"
                    })
    
    df_manuscript = pd.DataFrame(rows)
    df_manuscript.to_csv(OUTPUT_PATH / "manuscript_bias_table.csv", index=False)
    return df_manuscript

def main():
    logger.info("="*80)
    logger.info("DEMOGRAPHIC BIAS ANALYSIS")
    logger.info(f"Analyzing model: {MODEL_TO_ANALYZE}")
    logger.info("Loading predictions from all_predictions.csv files")
    logger.info("="*80)
    
    # Load demographics once
    logger.info("\n📊 Loading demographics...")
    demographics = load_demographics()
    logger.info(f"   Demographics loaded: {len(demographics)} samples")
    logger.info(f"   Race distribution:\n{demographics['race_clean'].value_counts().to_string()}")
    
    # Analyze each task
    all_results = {}
    
    for task_name, task_config in TASKS.items():
        results = analyze_task(task_name, task_config, demographics, MODEL_TO_ANALYZE)
        if results:
            all_results[task_name] = results
    
    # Create manuscript table
    if all_results:
        df_manuscript = create_manuscript_table(all_results)
        logger.info(f"\n✅ Manuscript table saved to: {OUTPUT_PATH}/manuscript_bias_table.csv")
    
    # Final summary
    logger.info("\n" + "="*80)
    logger.info("✅ BIAS ANALYSIS COMPLETE")
    logger.info(f"Results saved to: {OUTPUT_PATH}")
    logger.info("="*80)
    
    # Print file listing
    logger.info("\n📁 Files created:")
    for f in OUTPUT_PATH.glob("*.csv"):
        logger.info(f"   {f.name}")

if __name__ == "__main__":
    main()

2026-07-24 16:09:50 | INFO | ================================================================================
2026-07-24 16:09:50 | INFO | DEMOGRAPHIC BIAS ANALYSIS
2026-07-24 16:09:50 | INFO | Analyzing model: VotingEnsemble
2026-07-24 16:09:50 | INFO | Loading predictions from all_predictions.csv files
2026-07-24 16:09:50 | INFO | ================================================================================
2026-07-24 16:09:50 | INFO | 
📊 Loading demographics...
2026-07-24 16:09:50 | INFO |    Demographics loaded: 12020 samples
2026-07-24 16:09:50 | INFO |    Race distribution:
race_clean
White      4905
Other      4234
Asian      2234
Black       591
Unknown      56
2026-07-24 16:09:50 | INFO | 
2026-07-24 16:09:50 | INFO | TASK: Hospital Admission
2026-07-24 16:09:50 | INFO | Model: VotingEnsemble
2026-07-24 16:09:50 | INFO | ================================================================================
2026-07-24 16:09:50 | INFO |    ✅ Loaded predictions: 12020 samples
2026-0


Subgroup   Type    N Sensitivity AUC-ROC Threshold
   White   Race 4905       74.0%   0.803      0.53
   Black   Race  591       79.5%   0.798      0.38
   Asian   Race 2234       81.1%   0.822      0.43
   Other   Race 4234       75.8%   0.822      0.39
   18-40    Age 4031       74.6%   0.798      0.25
   41-64    Age 4237       77.3%   0.786      0.41
   65-79    Age 2409       69.3%   0.748      0.61
     80+    Age 1343       46.3%   0.715      0.79
  Female Gender 6411       76.9%   0.816      0.43
    Male Gender 5602       78.3%   0.818      0.46


2026-07-24 16:09:51 | INFO |       Asian      (n= 2234): Sens=67.8%, AUC=0.6886
2026-07-24 16:09:51 | INFO |       Other      (n= 4234): Sens=75.1%, AUC=0.6751
2026-07-24 16:09:51 | INFO | 
      Disparities (White - Others):
2026-07-24 16:09:51 | INFO |          White - Black     : -12.9%
2026-07-24 16:09:51 | INFO |          White - Asian     : 3.1%
2026-07-24 16:09:51 | INFO |          White - Other     : -4.2%
2026-07-24 16:09:51 | INFO | 
   📊 AGE GROUP ANALYSIS
2026-07-24 16:09:51 | INFO |       18-40    (n= 4031): Sens=61.2%, AUC=0.6899
2026-07-24 16:09:51 | INFO |       41-64    (n= 4237): Sens=59.4%, AUC=0.6435
2026-07-24 16:09:51 | INFO |       65-79    (n= 2409): Sens=61.2%, AUC=0.5982
2026-07-24 16:09:51 | INFO |       80+      (n= 1343): Sens=83.8%, AUC=0.5753
2026-07-24 16:09:51 | INFO | 
   📊 GENDER ANALYSIS
2026-07-24 16:09:51 | INFO |       Female   (n= 6411): Sens=64.5%, AUC=0.6689
2026-07-24 16:09:51 | INFO |       Male     (n= 5602): Sens=66.0%, AUC=0.6505
2026-07-2


Subgroup   Type    N Sensitivity AUC-ROC Threshold
   White   Race 4905       70.9%   0.643      0.41
   Black   Race  591       83.8%   0.606      0.34
   Asian   Race 2234       67.8%   0.689      0.40
   Other   Race 4234       75.1%   0.675      0.34
   18-40    Age 4031       61.2%   0.690      0.33
   41-64    Age 4237       59.4%   0.644      0.41
   65-79    Age 2409       61.2%   0.598      0.45
     80+    Age 1343       83.8%   0.575      0.44
  Female Gender 6411       64.5%   0.669      0.41
    Male Gender 5602       66.0%   0.651      0.40

Subgroup   Type    N Sensitivity AUC-ROC Threshold
   White   Race 2510       79.1%   0.670      0.34
   Black   Race  265       78.2%   0.691      0.36
   Asian   Race  753       55.4%   0.700      0.42
   Other   Race 1206       45.6%   0.636      0.42
   18-40    Age  546       45.6%   0.641      0.42
   41-64    Age 1901       53.6%   0.697      0.42
   65-79    Age 1391       85.1%   0.672      0.33
     80+    Age  927       41

In [17]:
# CELL 9 - ESI vs ML COMPARISON (Using all_predictions.csv and ESI baseline)
# Compares ESI triage acuity against your trained ML models

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, roc_curve
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# Paths
BASE_PATH = Path(r"E:\TSINGHUA\thisis\Paper\output_paper")
OUTPUT_PATH = BASE_PATH / "esi_vs_ml_comparison"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Task configuration
TASKS = {
    'Admission': {
        'display': 'Hospital Admission',
        'folder': 'task2_admission',
        'predictions_file': 'all_predictions.csv',
        'feature_file': 'X_test.pkl',
        'target_file': 'y_test_admit.pkl',
        'target_column': 'TARGET_Admitted',
        'prevalence': 0.396
    },
    'ED_LOS': {
        'display': 'ED Stay ≥8h',
        'folder': 'task1_ed_los',
        'predictions_file': 'all_predictions.csv',
        'feature_file': 'X_test.pkl',
        'target_file': 'y_test_admit.pkl',
        'target_column': 'TARGET_ED_LOS_over8h',
        'prevalence': 0.223
    },
    'Hosp_LOS': {
        'display': 'Hospital Stay >7d',
        'folder': 'task3_hosp_los',
        'predictions_file': 'all_predictions.csv',
        'feature_file': 'X_test_hosp.pkl',
        'target_file': 'y_test_hosp.pkl',
        'target_column': 'TARGET_Hosp_LOS_over7d',
        'prevalence': 0.197
    }
}

# ML models to compare (all models from your predictions)
ML_MODELS = ['XGBoost', 'LightGBM', 'RandomForest', 'TabNet', 'VotingEnsemble']

def evaluate_esi_baseline(y_true, X_test):
    """
    ESI baseline using triage_acuity (1-5)
    Lower ESI = higher acuity = higher probability of positive outcome
    """
    if 'triage_acuity' in X_test.columns:
        esi_scores = X_test['triage_acuity'].values
    else:
        logger.warning("   triage_acuity not found, using synthetic ESI")
        esi_scores = np.random.randint(1, 6, len(y_true))
    
    # Convert ESI to probability (1->1.0, 5->0.0)
    esi_proba = 1 - (esi_scores - 1) / 4
    esi_proba = np.clip(esi_proba, 0.01, 0.99)
    
    # Find optimal threshold
    fpr, tpr, thresholds = roc_curve(y_true, esi_proba)
    youden_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[youden_idx] if youden_idx < len(thresholds) else 0.5
    
    y_pred_binary = (esi_proba >= optimal_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary, labels=[0, 1]).ravel()
    
    metrics = {
        'model': 'ESI_Triage',
        'auroc': roc_auc_score(y_true, esi_proba),
        'auprc': average_precision_score(y_true, esi_proba),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1': f1_score(y_true, y_pred_binary),
        'accuracy': (tp + tn) / (tp + tn + fp + fn),
        'optimal_threshold': optimal_threshold
    }
    
    return metrics

def evaluate_ml_model(y_true, y_pred_proba, model_name):
    """Evaluate a single ML model"""
    
    # Find optimal threshold
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    youden_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[youden_idx] if youden_idx < len(thresholds) else 0.5
    
    y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary, labels=[0, 1]).ravel()
    
    metrics = {
        'model': model_name,
        'auroc': roc_auc_score(y_true, y_pred_proba),
        'auprc': average_precision_score(y_true, y_pred_proba),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1': f1_score(y_true, y_pred_binary),
        'accuracy': (tp + tn) / (tp + tn + fp + fn),
        'optimal_threshold': optimal_threshold
    }
    
    return metrics

def main():
    logger.info("="*80)
    logger.info("ESI vs ML MODEL COMPARISON")
    logger.info("Comparing ESI triage acuity against all trained ML models")
    logger.info("="*80)
    
    all_comparisons = []
    
    for task_key, task_config in TASKS.items():
        logger.info(f"\n{'='*60}")
        logger.info(f"TASK: {task_config['display']}")
        logger.info(f"{'='*60}")
        
        # ================================================================
        # Load data for ESI evaluation
        # ================================================================
        # Load features
        X_test = pd.read_pickle(BASE_PATH / task_config['feature_file'])
        
        # Load targets
        y_df = pd.read_pickle(BASE_PATH / task_config['target_file'])
        y_true = y_df[task_config['target_column']].values.astype(int)
        
        # Remove NaN
        valid_mask = ~np.isnan(y_true)
        y_true = y_true[valid_mask].astype(int)
        X_test = X_test[valid_mask]
        
        logger.info(f"   Test samples: {len(y_true)}")
        logger.info(f"   Prevalence: {y_true.mean():.1%}")
        
        # ================================================================
        # Evaluate ESI baseline
        # ================================================================
        esi_metrics = evaluate_esi_baseline(y_true, X_test)
        logger.info(f"\n   📊 ESI BASELINE:")
        logger.info(f"      AUC-ROC: {esi_metrics['auroc']:.4f}")
        logger.info(f"      Sensitivity: {esi_metrics['sensitivity']:.1%}")
        logger.info(f"      Specificity: {esi_metrics['specificity']:.1%}")
        logger.info(f"      F1: {esi_metrics['f1']:.4f}")
        
        # Store ESI results
        esi_metrics['task'] = task_config['display']
        all_comparisons.append(esi_metrics)
        
        # ================================================================
        # Load ML predictions
        # ================================================================
        pred_file = BASE_PATH / task_config['folder'] / task_config['predictions_file']
        
        if not pred_file.exists():
            logger.error(f"   Predictions file not found: {pred_file}")
            continue
        
        pred_df = pd.read_csv(pred_file)
        
        # Align with valid mask if needed
        if len(pred_df) != len(y_true):
            # Try to align
            pred_df = pred_df.iloc[valid_mask] if len(pred_df) == len(valid_mask) else pred_df
        
        logger.info(f"\n   📊 ML MODELS:")
        
        # Evaluate each ML model
        for model_name in ML_MODELS:
            if model_name in pred_df.columns:
                y_pred_proba = pred_df[model_name].values
                ml_metrics = evaluate_ml_model(y_true, y_pred_proba, model_name)
                ml_metrics['task'] = task_config['display']
                
                # Calculate improvement over ESI
                ml_metrics['improvement_vs_esi_auc'] = ml_metrics['auroc'] - esi_metrics['auroc']
                ml_metrics['improvement_vs_esi_sens'] = ml_metrics['sensitivity'] - esi_metrics['sensitivity']
                ml_metrics['improvement_vs_esi_spec'] = ml_metrics['specificity'] - esi_metrics['specificity']
                ml_metrics['improvement_pct_auc'] = (ml_metrics['auroc'] - esi_metrics['auroc']) / esi_metrics['auroc'] * 100
                
                all_comparisons.append(ml_metrics)
                
                # Log results
                logger.info(f"\n      {model_name}:")
                logger.info(f"         AUC: {ml_metrics['auroc']:.4f} (+{ml_metrics['improvement_vs_esi_auc']:.4f})")
                logger.info(f"         Sens: {ml_metrics['sensitivity']:.1%} (+{ml_metrics['improvement_vs_esi_sens']:.1%})")
                logger.info(f"         Spec: {ml_metrics['specificity']:.1%} (+{ml_metrics['improvement_vs_esi_spec']:.1%})")
                logger.info(f"         F1: {ml_metrics['f1']:.4f}")
    
    # ================================================================
    # Create comparison DataFrames
    # ================================================================
    df_all = pd.DataFrame(all_comparisons)
    
    # Save full results
    df_all.to_csv(OUTPUT_PATH / "esi_vs_ml_full_comparison.csv", index=False)
    logger.info(f"\n✅ Full comparison saved to: {OUTPUT_PATH}/esi_vs_ml_full_comparison.csv")
    
    # ================================================================
    # Create summary table (best ML vs ESI per task)
    # ================================================================
    summary_rows = []
    
    for task_key, task_config in TASKS.items():
        task_display = task_config['display']
        
        # Get ESI for this task
        esi_row = df_all[(df_all['task'] == task_display) & (df_all['model'] == 'ESI_Triage')]
        if len(esi_row) == 0:
            continue
        
        esi_auc = esi_row['auroc'].values[0]
        esi_sens = esi_row['sensitivity'].values[0]
        
        # Get best ML for this task by AUC
        ml_rows = df_all[(df_all['task'] == task_display) & (df_all['model'] != 'ESI_Triage')]
        if len(ml_rows) == 0:
            continue
        
        best_auc_row = ml_rows.loc[ml_rows['auroc'].idxmax()]
        best_sens_row = ml_rows.loc[ml_rows['sensitivity'].idxmax()]
        
        summary_rows.append({
            'Task': task_display,
            'Metric': 'AUC-ROC',
            'ESI_Baseline': esi_auc,
            'Best_ML_Model': best_auc_row['model'],
            'Best_ML_Value': best_auc_row['auroc'],
            'Absolute_Improvement': best_auc_row['auroc'] - esi_auc,
            'Percent_Improvement': (best_auc_row['auroc'] - esi_auc) / esi_auc * 100
        })
        
        summary_rows.append({
            'Task': task_display,
            'Metric': 'Sensitivity',
            'ESI_Baseline': esi_sens,
            'Best_ML_Model': best_sens_row['model'],
            'Best_ML_Value': best_sens_row['sensitivity'],
            'Absolute_Improvement': best_sens_row['sensitivity'] - esi_sens,
            'Percent_Improvement': (best_sens_row['sensitivity'] - esi_sens) / esi_sens * 100
        })
    
    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(OUTPUT_PATH / "esi_vs_ml_summary.csv", index=False)
    logger.info(f"✅ Summary saved to: {OUTPUT_PATH}/esi_vs_ml_summary.csv")
    
    # ================================================================
    # Create manuscript-ready table
    # ================================================================
    manuscript_rows = []
    
    for task_key, task_config in TASKS.items():
        task_display = task_config['display']
        
        # Get ESI
        esi_row = df_all[(df_all['task'] == task_display) & (df_all['model'] == 'ESI_Triage')]
        if len(esi_row) == 0:
            continue
        
        # Get best ML overall
        ml_rows = df_all[(df_all['task'] == task_display) & (df_all['model'] != 'ESI_Triage')]
        best_ml = ml_rows.loc[ml_rows['auroc'].idxmax()]
        
        manuscript_rows.append({
            'Task': task_display,
            'Model': 'ESI Triage',
            'AUC-ROC': f"{esi_row['auroc'].values[0]:.4f}",
            'Sensitivity': f"{esi_row['sensitivity'].values[0]:.1%}",
            'Specificity': f"{esi_row['specificity'].values[0]:.1%}",
            'F1': f"{esi_row['f1'].values[0]:.4f}"
        })
        
        manuscript_rows.append({
            'Task': task_display,
            'Model': f"ML ({best_ml['model']})",
            'AUC-ROC': f"{best_ml['auroc']:.4f}",
            'Sensitivity': f"{best_ml['sensitivity']:.1%}",
            'Specificity': f"{best_ml['specificity']:.1%}",
            'F1': f"{best_ml['f1']:.4f}"
        })
        
        manuscript_rows.append({
            'Task': task_display,
            'Model': 'Improvement',
            'AUC-ROC': f"+{best_ml['auroc'] - esi_row['auroc'].values[0]:.4f}",
            'Sensitivity': f"+{(best_ml['sensitivity'] - esi_row['sensitivity'].values[0])*100:.1f}pp",
            'Specificity': f"+{(best_ml['specificity'] - esi_row['specificity'].values[0])*100:.1f}pp",
            'F1': f"+{best_ml['f1'] - esi_row['f1'].values[0]:.4f}"
        })
    
    df_manuscript = pd.DataFrame(manuscript_rows)
    df_manuscript.to_csv(OUTPUT_PATH / "esi_vs_ml_manuscript_table.csv", index=False)
    
    # ================================================================
    # Print summary
    # ================================================================
    logger.info("\n" + "="*80)
    logger.info("📊 ESI vs ML - SUMMARY TABLE")
    logger.info("="*80)
    
    print("\n" + df_summary.to_string(index=False))
    
    logger.info("\n" + "="*80)
    logger.info("✅ ESI vs ML COMPARISON COMPLETE")
    logger.info(f"Results saved to: {OUTPUT_PATH}")
    logger.info("="*80)
    
    # Print file listing
    logger.info("\n📁 Files created:")
    for f in OUTPUT_PATH.glob("*.csv"):
        logger.info(f"   {f.name}")

if __name__ == "__main__":
    main()

2026-07-24 16:09:51 | INFO | ================================================================================
2026-07-24 16:09:51 | INFO | ESI vs ML MODEL COMPARISON
2026-07-24 16:09:51 | INFO | Comparing ESI triage acuity against all trained ML models
2026-07-24 16:09:51 | INFO | ================================================================================
2026-07-24 16:09:51 | INFO | 
2026-07-24 16:09:51 | INFO | TASK: Hospital Admission
2026-07-24 16:09:51 | INFO | ============================================================
2026-07-24 16:09:51 | INFO |    Test samples: 12020
2026-07-24 16:09:51 | INFO |    Prevalence: 39.6%
2026-07-24 16:09:51 | WARNING |    triage_acuity not found, using synthetic ESI
2026-07-24 16:09:51 | INFO | 
   📊 ESI BASELINE:
2026-07-24 16:09:51 | INFO |       AUC-ROC: 0.5047
2026-07-24 16:09:51 | INFO |       Sensitivity: 40.8%
2026-07-24 16:09:51 | INFO |       Specificity: 60.2%
2026-07-24 16:09:51 | INFO |       F1: 0.4050
2026-07-24 16:09:51 | INFO 


              Task      Metric  ESI_Baseline  Best_ML_Model  Best_ML_Value  Absolute_Improvement  Percent_Improvement
Hospital Admission     AUC-ROC      0.504701 VotingEnsemble       0.818056              0.313355            62.087192
Hospital Admission Sensitivity      0.407765 VotingEnsemble       0.791186              0.383421            94.029851
       ED Stay ≥8h     AUC-ROC      0.493772   RandomForest       0.660268              0.166496            33.719230
       ED Stay ≥8h Sensitivity      0.797091       LightGBM       0.828422              0.031332             3.930744
 Hospital Stay >7d     AUC-ROC      0.477488 VotingEnsemble       0.666790              0.189301            39.645198
 Hospital Stay >7d Sensitivity      0.000000        XGBoost       0.689765              0.689765                  inf


In [18]:
# CELL 10 - COMBINE ALL RESULTS FOR PAPER (CSV + TEX files with proper encoding)
# Creates a single comprehensive results file for your manuscript

import pandas as pd
import numpy as np
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

# Paths
BASE_PATH = Path(r"E:\TSINGHUA\thisis\Paper\output_paper")
OUTPUT_PATH = BASE_PATH / "paper_results_combined"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Task display names
TASK_NAMES = {
    'Admission': 'Hospital Admission',
    'ED_LOS': 'ED Stay >=8h',
    'Hosp_LOS': 'Hospital Stay >7d'
}

def load_all_results():
    """Load all results from previous analyses"""
    
    results = {}
    
    # ================================================================
    # Load ESI vs ML comparison results
    # ================================================================
    esi_ml_path = BASE_PATH / "esi_vs_ml_comparison" / "esi_vs_ml_full_comparison.csv"
    if esi_ml_path.exists():
        df = pd.read_csv(esi_ml_path)
        results['esi_ml'] = df
        logger.info(f"✅ Loaded ESI vs ML results: {len(df)} rows")
    else:
        logger.warning("ESI vs ML results not found")
    
    # ================================================================
    # Load Bias Analysis results
    # ================================================================
    bias_path = BASE_PATH / "demographic_bias_analysis"
    results['bias'] = {}
    
    for task in ['Admission', 'ED_LOS', 'Hosp_LOS']:
        race_file = bias_path / f"{task}_race_bias.csv"
        age_file = bias_path / f"{task}_age_bias.csv"
        gender_file = bias_path / f"{task}_gender_bias.csv"
        
        if race_file.exists():
            results['bias'][f"{task}_race"] = pd.read_csv(race_file)
            logger.info(f"✅ Loaded {task} race bias")
        if age_file.exists():
            results['bias'][f"{task}_age"] = pd.read_csv(age_file)
            logger.info(f"✅ Loaded {task} age bias")
        if gender_file.exists():
            results['bias'][f"{task}_gender"] = pd.read_csv(gender_file)
            logger.info(f"✅ Loaded {task} gender bias")
    
    # ================================================================
    # Load model comparison from each task
    # ================================================================
    results['model_comparison'] = {}
    
    for task, folder in [('Admission', 'task2_admission'), ('ED_LOS', 'task1_ed_los'), ('Hosp_LOS', 'task3_hosp_los')]:
        model_file = BASE_PATH / folder / "model_comparison.csv"
        if model_file.exists():
            df = pd.read_csv(model_file)
            df['Task'] = TASK_NAMES[task]
            results['model_comparison'][task] = df
            logger.info(f"✅ Loaded model comparison for {task}: {len(df)} models")
    
    return results

def create_main_performance_table(results):
    """Create main performance table (ESI vs Best ML)"""
    
    rows = []
    
    esi_ml = results.get('esi_ml')
    if esi_ml is not None:
        tasks = esi_ml['task'].unique()
        
        for task in tasks:
            esi_row = esi_ml[(esi_ml['model'] == 'ESI_Triage') & (esi_ml['task'] == task)]
            if len(esi_row) == 0:
                continue
            
            esi_auc = esi_row['auroc'].values[0]
            esi_sens = esi_row['sensitivity'].values[0]
            esi_spec = esi_row['specificity'].values[0]
            esi_f1 = esi_row['f1'].values[0]
            
            rows.append({
                'Task': task,
                'Model_Type': 'ESI Triage',
                'Model_Name': 'ESI',
                'AUC-ROC': esi_auc,
                'Sensitivity': esi_sens,
                'Specificity': esi_spec,
                'F1': esi_f1,
                'Improvement': '-'
            })
            
            ml_rows = esi_ml[(esi_ml['model'] != 'ESI_Triage') & (esi_ml['task'] == task)]
            if len(ml_rows) > 0:
                best_auc_row = ml_rows.loc[ml_rows['auroc'].idxmax()]
                
                rows.append({
                    'Task': task,
                    'Model_Type': 'Machine Learning',
                    'Model_Name': best_auc_row['model'],
                    'AUC-ROC': best_auc_row['auroc'],
                    'Sensitivity': best_auc_row['sensitivity'],
                    'Specificity': best_auc_row['specificity'],
                    'F1': best_auc_row['f1'],
                    'Improvement': f"+{best_auc_row['improvement_vs_esi_auc']:.4f} (+{best_auc_row['improvement_pct_auc']:.1f}%)"
                })
    
    return pd.DataFrame(rows)

def create_bias_summary_table(results):
    """Create bias summary table (race, age, gender disparities)"""
    
    rows = []
    
    for task_key, task_display in TASK_NAMES.items():
        # Race bias
        race_file = f"{task_key}_race"
        if race_file in results['bias']:
            df_race = results['bias'][race_file]
            
            white_rows = df_race[df_race['group'] == 'White']
            white_sens = white_rows['sensitivity'].values[0] if len(white_rows) > 0 else 0
            
            for _, row in df_race.iterrows():
                if row['group'] != 'White':
                    rows.append({
                        'Task': task_display,
                        'Demographic': 'Race',
                        'Group': row['group'],
                        'N': row['n'],
                        'Sensitivity': f"{row['sensitivity']:.1%}",
                        'AUC': f"{row['auroc']:.3f}",
                        'Disparity_vs_White': f"{white_sens - row['sensitivity']:.1%}"
                    })
        
        # Age bias
        age_file = f"{task_key}_age"
        if age_file in results['bias']:
            df_age = results['bias'][age_file]
            
            young_rows = df_age[df_age['group'] == '18-40']
            young_sens = young_rows['sensitivity'].values[0] if len(young_rows) > 0 else 0
            
            for _, row in df_age.iterrows():
                if row['group'] != '18-40':
                    rows.append({
                        'Task': task_display,
                        'Demographic': 'Age',
                        'Group': row['group'],
                        'N': row['n'],
                        'Sensitivity': f"{row['sensitivity']:.1%}",
                        'AUC': f"{row['auroc']:.3f}",
                        'Disparity_vs_Young': f"{young_sens - row['sensitivity']:.1%}"
                    })
        
        # Gender bias
        gender_file = f"{task_key}_gender"
        if gender_file in results['bias']:
            df_gender = results['bias'][gender_file]
            
            if len(df_gender) >= 2:
                female_row = df_gender[df_gender['group'] == 'Female']
                male_row = df_gender[df_gender['group'] == 'Male']
                
                if len(female_row) > 0 and len(male_row) > 0:
                    female_sens = female_row['sensitivity'].values[0]
                    male_sens = male_row['sensitivity'].values[0]
                    
                    rows.append({
                        'Task': task_display,
                        'Demographic': 'Gender',
                        'Group': 'Female vs Male',
                        'N': f"{female_row['n'].values[0]} / {male_row['n'].values[0]}",
                        'Sensitivity': f"{female_sens:.1%} / {male_sens:.1%}",
                        'AUC': f"{female_row['auroc'].values[0]:.3f} / {male_row['auroc'].values[0]:.3f}",
                        'Disparity': f"{female_sens - male_sens:.1%}"
                    })
    
    return pd.DataFrame(rows)

def create_model_ranking_table(results):
    """Create model ranking table for each task"""
    
    rows = []
    
    for task_key, task_display in TASK_NAMES.items():
        if task_key in results['model_comparison']:
            df = results['model_comparison'][task_key].copy()
            
            if 'AUC-ROC' in df.columns:
                df = df.sort_values('AUC-ROC', ascending=False)
            
            for rank, (_, row) in enumerate(df.iterrows(), 1):
                rows.append({
                    'Task': task_display,
                    'Rank': rank,
                    'Model': row.get('Model', 'Unknown'),
                    'AUC-ROC': row.get('AUC-ROC', np.nan),
                    'AUC-ROC_CI': row.get('AUC-ROC_CI', '-'),
                    'Sensitivity': row.get('Sensitivity', np.nan),
                    'Specificity': row.get('Specificity', np.nan),
                    'F1': row.get('F1', np.nan),
                    'AUPRC': row.get('AUPRC', np.nan)
                })
    
    return pd.DataFrame(rows)

def create_complete_results_table(results):
    """Create a complete results table combining ESI and all ML models"""
    
    rows = []
    
    esi_ml = results.get('esi_ml')
    if esi_ml is not None:
        for _, row in esi_ml.iterrows():
            rows.append({
                'Task': row['task'],
                'Model': row['model'],
                'Type': 'ESI' if row['model'] == 'ESI_Triage' else 'ML',
                'AUC-ROC': row['auroc'],
                'AUPRC': row['auprc'],
                'Sensitivity': row['sensitivity'],
                'Specificity': row['specificity'],
                'F1': row['f1'],
                'Accuracy': row['accuracy'],
                'Optimal_Threshold': row['optimal_threshold'],
                'Improvement_vs_ESI_AUC': row.get('improvement_vs_esi_auc', 0),
                'Improvement_vs_ESI_Sensitivity': row.get('improvement_vs_esi_sens', 0),
                'Improvement_Percent': row.get('improvement_pct_auc', 0)
            })
    
    return pd.DataFrame(rows)

def save_tables_as_csv(results):
    """Save all tables as CSV files"""
    
    # Table 0: Complete Results
    df_complete = create_complete_results_table(results)
    df_complete.to_csv(OUTPUT_PATH / "Table0_Complete_Results.csv", index=False)
    logger.info("✅ Saved: Table0_Complete_Results.csv")
    
    # Table 1: ESI vs Best ML
    df_performance = create_main_performance_table(results)
    df_performance.to_csv(OUTPUT_PATH / "Table1_ESI_vs_ML_Performance.csv", index=False)
    logger.info("✅ Saved: Table1_ESI_vs_ML_Performance.csv")
    
    # Table 2: Bias Summary
    df_bias = create_bias_summary_table(results)
    if len(df_bias) > 0:
        df_bias.to_csv(OUTPUT_PATH / "Table2_Demographic_Bias_Summary.csv", index=False)
        logger.info("✅ Saved: Table2_Demographic_Bias_Summary.csv")
    
    # Table 3: Model Ranking
    df_ranking = create_model_ranking_table(results)
    if len(df_ranking) > 0:
        df_ranking.to_csv(OUTPUT_PATH / "Table3_Model_Ranking.csv", index=False)
        logger.info("✅ Saved: Table3_Model_Ranking.csv")
    
    return df_performance, df_bias, df_ranking, df_complete

def save_tables_as_tex(df_performance, df_bias, df_ranking):
    """Save tables as LaTeX .tex files with proper encoding"""
    
    # Table 1: ESI vs ML Performance (TeX)
    tex_content = r"""\begin{table}[htbp]
\centering
\caption{Comparison of ESI triage baseline against best performing machine learning models}
\label{tab:esi_vs_ml}
\begin{tabular}{@{}lcccccc@{}}
\toprule
\textbf{Task} & \textbf{Model} & \textbf{AUC-ROC} & \textbf{Sensitivity} & \textbf{Specificity} & \textbf{F1} & \textbf{Improvement} \\
\midrule
"""
    
    for _, row in df_performance.iterrows():
        task = row['Task'].replace('_', ' ').replace('>=', r'$\ge$')
        model_name = row['Model_Name']
        auc = f"{row['AUC-ROC']:.4f}"
        sens = f"{row['Sensitivity']:.1%}"
        spec = f"{row['Specificity']:.1%}"
        f1 = f"{row['F1']:.4f}"
        improvement = row['Improvement']
        
        tex_content += f"{task} & {model_name} & {auc} & {sens} & {spec} & {f1} & {improvement} \\\\\n"
    
    tex_content += r"""
\bottomrule
\end{tabular}
\end{table}
"""
    
    with open(OUTPUT_PATH / "Table1_ESI_vs_ML.tex", "w", encoding='utf-8') as f:
        f.write(tex_content)
    logger.info("✅ Saved: Table1_ESI_vs_ML.tex")
    
    # Table 2: Bias Summary (TeX)
    if len(df_bias) > 0:
        tex_content2 = r"""\begin{table}[htbp]
\centering
\caption{Demographic bias analysis across subgroups}
\label{tab:bias_analysis}
\begin{tabular}{@{}lcccccc@{}}
\toprule
\textbf{Task} & \textbf{Demographic} & \textbf{Subgroup} & \textbf{N} & \textbf{Sensitivity} & \textbf{AUC-ROC} & \textbf{Disparity} \\
\midrule
"""
        
        for _, row in df_bias.iterrows():
            task = row['Task']
            demo = row['Demographic']
            group = row['Group']
            n = row['N']
            sens = row['Sensitivity']
            auc = row['AUC']
            
            if demo == 'Gender':
                disparity = row['Disparity']
                tex_content2 += f"{task} & {demo} & {group} & {n} & {sens} & {auc} & {disparity} \\\\\n"
            elif demo == 'Race':
                disparity = row['Disparity_vs_White']
                tex_content2 += f"{task} & {demo} & {group} & {n} & {sens} & {auc} & {disparity} \\\\\n"
            else:
                disparity = row['Disparity_vs_Young']
                tex_content2 += f"{task} & {demo} & {group} & {n} & {sens} & {auc} & {disparity} \\\\\n"
        
        tex_content2 += r"""
\bottomrule
\end{tabular}
\end{table}
"""
        
        with open(OUTPUT_PATH / "Table2_Bias_Analysis.tex", "w", encoding='utf-8') as f:
            f.write(tex_content2)
        logger.info("✅ Saved: Table2_Bias_Analysis.tex")
    
    # Table 3: Model Ranking (TeX)
    if len(df_ranking) > 0:
        tex_content3 = r"""\begin{table}[htbp]
\centering
\caption{Model performance ranking by task}
\label{tab:model_ranking}
\begin{tabular}{@{}lccccccc@{}}
\toprule
\textbf{Task} & \textbf{Rank} & \textbf{Model} & \textbf{AUC-ROC} & \textbf{Sensitivity} & \textbf{Specificity} & \textbf{F1} \\
\midrule
"""
        
        for _, row in df_ranking.iterrows():
            task = row['Task']
            rank = row['Rank']
            model = row['Model']
            auc = f"{row['AUC-ROC']:.4f}" if pd.notna(row['AUC-ROC']) else '-'
            sens = f"{row['Sensitivity']:.1%}" if pd.notna(row['Sensitivity']) else '-'
            spec = f"{row['Specificity']:.1%}" if pd.notna(row['Specificity']) else '-'
            f1 = f"{row['F1']:.4f}" if pd.notna(row['F1']) else '-'
            
            tex_content3 += f"{task} & {rank} & {model} & {auc} & {sens} & {spec} & {f1} \\\\\n"
        
        tex_content3 += r"""
\bottomrule
\end{tabular}
\end{table}
"""
        
        with open(OUTPUT_PATH / "Table3_Model_Ranking.tex", "w", encoding='utf-8') as f:
            f.write(tex_content3)
        logger.info("✅ Saved: Table3_Model_Ranking.tex")

def save_excel_file(results, df_performance, df_bias, df_ranking, df_complete):
    """Save all results to a comprehensive Excel file"""
    
    excel_path = OUTPUT_PATH / "Paper_Results_Complete.xlsx"
    
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Sheet 1: Complete Results
        if len(df_complete) > 0:
            df_complete.to_excel(writer, sheet_name='Complete_Results', index=False)
        
        # Sheet 2: ESI vs Best ML
        if len(df_performance) > 0:
            df_performance.to_excel(writer, sheet_name='ESI_vs_Best_ML', index=False)
        
        # Sheet 3: Bias Summary
        if len(df_bias) > 0:
            df_bias.to_excel(writer, sheet_name='Bias_Summary', index=False)
        
        # Sheet 4: Model Ranking
        if len(df_ranking) > 0:
            df_ranking.to_excel(writer, sheet_name='Model_Ranking', index=False)
        
        # Sheet 5: Full ESI vs ML results
        if 'esi_ml' in results and results['esi_ml'] is not None:
            results['esi_ml'].to_excel(writer, sheet_name='Full_ESI_vs_ML', index=False)
        
        # Sheet 6-8: Detailed bias by task
        for name, df in results['bias'].items():
            sheet_name = name[:31]
            df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Sheet 9-11: Model comparisons
        for task, df in results['model_comparison'].items():
            sheet_name = f"{task}_Models"[:31]
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    logger.info(f"✅ Saved: Paper_Results_Complete.xlsx")

def print_summary_tables(df_performance, df_bias, df_ranking):
    """Print summary tables to console"""
    
    print("\n" + "="*100)
    print("TABLE 1: ESI vs BEST ML PERFORMANCE")
    print("="*100)
    print(df_performance[['Task', 'Model_Type', 'Model_Name', 'AUC-ROC', 'Sensitivity', 'Specificity', 'F1']].to_string(index=False))
    
    if len(df_bias) > 0:
        print("\n" + "="*100)
        print("TABLE 2: DEMOGRAPHIC BIAS SUMMARY")
        print("="*100)
        print(df_bias[['Task', 'Demographic', 'Group', 'N', 'Sensitivity', 'Disparity_vs_White']].to_string(index=False))
    
    if len(df_ranking) > 0:
        print("\n" + "="*100)
        print("TABLE 3: MODEL RANKING BY TASK")
        print("="*100)
        print(df_ranking[['Task', 'Rank', 'Model', 'AUC-ROC', 'Sensitivity', 'Specificity', 'F1']].to_string(index=False))

def main():
    """Main function to create all manuscript tables"""
    
    logger.info("="*80)
    logger.info("CREATING MANUSCRIPT TABLES")
    logger.info("="*80)
    
    # Load all results
    results = load_all_results()
    
    if results.get('esi_ml') is None:
        logger.error("No ESI vs ML results found. Please run the ESI comparison first.")
        return
    
    # Create all tables
    df_complete = create_complete_results_table(results)
    df_performance = create_main_performance_table(results)
    df_bias = create_bias_summary_table(results)
    df_ranking = create_model_ranking_table(results)
    
    # Save as CSV
    logger.info("\n📊 Saving CSV files...")
    save_tables_as_csv(results)
    
    # Save as TeX
    logger.info("\n📊 Saving TeX files...")
    save_tables_as_tex(df_performance, df_bias, df_ranking)
    
    # Save Excel file
    logger.info("\n📊 Saving Excel file...")
    save_excel_file(results, df_performance, df_bias, df_ranking, df_complete)
    
    # Print summary
    print_summary_tables(df_performance, df_bias, df_ranking)
    
    # Final summary
    logger.info("\n" + "="*80)
    logger.info("✅ PAPER RESULTS COMBINED SUCCESSFULLY")
    logger.info("="*80)
    logger.info(f"📁 Results saved to: {OUTPUT_PATH}")
    logger.info("\n📄 Files created:")
    for f in OUTPUT_PATH.glob("*.*"):
        logger.info(f"   - {f.name}")
    logger.info("="*80)

if __name__ == "__main__":
    main()

2026-07-24 16:09:52 | INFO | ================================================================================
2026-07-24 16:09:52 | INFO | CREATING MANUSCRIPT TABLES
2026-07-24 16:09:52 | INFO | ================================================================================
2026-07-24 16:09:52 | INFO | ✅ Loaded ESI vs ML results: 18 rows
2026-07-24 16:09:52 | INFO | ✅ Loaded Admission race bias
2026-07-24 16:09:52 | INFO | ✅ Loaded Admission age bias
2026-07-24 16:09:52 | INFO | ✅ Loaded Admission gender bias
2026-07-24 16:09:52 | INFO | ✅ Loaded ED_LOS race bias
2026-07-24 16:09:52 | INFO | ✅ Loaded ED_LOS age bias
2026-07-24 16:09:52 | INFO | ✅ Loaded ED_LOS gender bias
2026-07-24 16:09:52 | INFO | ✅ Loaded Hosp_LOS race bias
2026-07-24 16:09:52 | INFO | ✅ Loaded Hosp_LOS age bias
2026-07-24 16:09:52 | INFO | ✅ Loaded Hosp_LOS gender bias
2026-07-24 16:09:52 | INFO | ✅ Loaded model comparison for Admission: 5 models
2026-07-24 16:09:52 | INFO | ✅ Loaded model comparison for ED_LOS: 


TABLE 1: ESI vs BEST ML PERFORMANCE
              Task       Model_Type     Model_Name  AUC-ROC  Sensitivity  Specificity       F1
Hospital Admission       ESI Triage            ESI 0.504701     0.407765     0.601930 0.404960
Hospital Admission Machine Learning VotingEnsemble 0.818056     0.791186     0.685183 0.696922
       ED Stay ≥8h       ESI Triage            ESI 0.493772     0.797091     0.206874 0.349611
       ED Stay ≥8h Machine Learning   RandomForest 0.660268     0.756061     0.476389 0.422380
 Hospital Stay >7d       ESI Triage            ESI 0.477488     0.000000     1.000000 0.000000
 Hospital Stay >7d Machine Learning VotingEnsemble 0.666790     0.491471     0.745754 0.388702

TABLE 2: DEMOGRAPHIC BIAS SUMMARY
              Task Demographic          Group           N   Sensitivity Disparity_vs_White
Hospital Admission        Race          Black         591         79.5%              -5.5%
Hospital Admission        Race          Asian        2234         81.1%          

In [19]:
# CELL 11 - CREATE PUBLICATION-READY FIGURES
# Generate visualizations for your manuscript

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set publication-ready style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 300

# Paths
BASE_PATH = Path(r"E:\TSINGHUA\thisis\Paper\output_paper")
RESULTS_PATH = BASE_PATH / "paper_results_combined"
FIGURE_PATH = RESULTS_PATH / "figures"
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

def load_results():
    """Load all results from CSV files"""
    
    results = {}
    
    # Load Table 0 - Complete Results
    table0_path = RESULTS_PATH / "Table0_Complete_Results.csv"
    if table0_path.exists():
        results['complete'] = pd.read_csv(table0_path)
        print(f"✅ Loaded Complete Results: {len(results['complete'])} rows")
    
    # Load Table 1 - ESI vs ML
    table1_path = RESULTS_PATH / "Table1_ESI_vs_ML_Performance.csv"
    if table1_path.exists():
        results['esi_vs_ml'] = pd.read_csv(table1_path)
        print(f"✅ Loaded ESI vs ML: {len(results['esi_vs_ml'])} rows")
    
    # Load Table 3 - Model Ranking
    table3_path = RESULTS_PATH / "Table3_Model_Ranking.csv"
    if table3_path.exists():
        results['ranking'] = pd.read_csv(table3_path)
        print(f"✅ Loaded Model Ranking: {len(results['ranking'])} rows")
    
    return results

def create_figure1_esi_vs_ml(results):
    """Figure 1: ESI vs ML Performance Comparison"""
    
    df = results.get('esi_vs_ml')
    if df is None or len(df) == 0:
        print("No ESI vs ML data found")
        return None
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Subplot 1: AUC-ROC Comparison
    tasks = df['Task'].unique()
    x = np.arange(len(tasks))
    width = 0.35
    
    esi_auc = []
    ml_auc = []
    for task in tasks:
        esi_row = df[(df['Task'] == task) & (df['Model_Type'] == 'ESI Triage')]
        ml_row = df[(df['Task'] == task) & (df['Model_Type'] == 'Machine Learning')]
        esi_auc.append(esi_row['AUC-ROC'].values[0] if len(esi_row) > 0 else 0)
        ml_auc.append(ml_row['AUC-ROC'].values[0] if len(ml_row) > 0 else 0)
    
    bars1 = axes[0].bar(x - width/2, esi_auc, width, label='ESI Triage', color='#AAAAAA')
    bars2 = axes[0].bar(x + width/2, ml_auc, width, label='Best ML Model', color='#2E86AB')
    
    axes[0].set_ylabel('AUC-ROC')
    axes[0].set_title('Model Performance Comparison', fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(tasks, rotation=15, ha='right')
    axes[0].legend(loc='lower right')
    axes[0].set_ylim([0.5, 0.9])
    
    # Add value labels on bars
    for bars, values in [(bars1, esi_auc), (bars2, ml_auc)]:
        for bar, val in zip(bars, values):
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Subplot 2: Improvement Percentage
    improvements = []
    for task in tasks:
        ml_row = df[(df['Task'] == task) & (df['Model_Type'] == 'Machine Learning')]
        imp_str = ml_row['Improvement'].values[0] if len(ml_row) > 0 else '+0.0000 (0.0%)'
        # Extract percentage
        import re
        match = re.search(r'\(([\d\.]+)%\)', imp_str)
        improvements.append(float(match.group(1)) if match else 0)
    
    colors = ['#2E86AB' if imp > 0 else '#A23B72' for imp in improvements]
    bars = axes[1].barh(tasks, improvements, color=colors)
    axes[1].set_xlabel('Improvement (%)')
    axes[1].set_title('ML Improvement over ESI Baseline', fontweight='bold')
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    
    # Add value labels
    for bar, val in zip(bars, improvements):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    f'+{val:.1f}%', ha='left', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(FIGURE_PATH / 'Figure1_ESI_vs_ML_Performance.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_PATH / 'Figure1_ESI_vs_ML_Performance.pdf', bbox_inches='tight')
    plt.close()
    print("✅ Saved: Figure1_ESI_vs_ML_Performance.png/pdf")

def create_figure2_model_ranking(results):
    """Figure 2: Model Ranking Heatmap"""
    
    df = results.get('ranking')
    if df is None or len(df) == 0:
        print("No ranking data found")
        return None
    
    # Pivot for heatmap
    pivot_data = df.pivot(index='Model', columns='Task', values='AUC-ROC')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create heatmap
    im = ax.imshow(pivot_data.values, cmap='RdYlGn', aspect='auto', vmin=0.6, vmax=0.85)
    
    # Set ticks
    ax.set_xticks(np.arange(len(pivot_data.columns)))
    ax.set_yticks(np.arange(len(pivot_data.index)))
    ax.set_xticklabels(pivot_data.columns, rotation=15, ha='right')
    ax.set_yticklabels(pivot_data.index)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('AUC-ROC', fontsize=11)
    
    # Add value annotations
    for i in range(len(pivot_data.index)):
        for j in range(len(pivot_data.columns)):
            text = ax.text(j, i, f'{pivot_data.values[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=9)
    
    ax.set_title('Model Performance by Task (AUC-ROC)', fontweight='bold', fontsize=14)
    ax.set_xlabel('Task', fontsize=12)
    ax.set_ylabel('Model', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(FIGURE_PATH / 'Figure2_Model_Ranking_Heatmap.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_PATH / 'Figure2_Model_Ranking_Heatmap.pdf', bbox_inches='tight')
    plt.close()
    print("✅ Saved: Figure2_Model_Ranking_Heatmap.png/pdf")

def create_figure3_bias_analysis(results):
    """Figure 3: Demographic Bias Analysis"""
    
    # Load bias data from CSV files
    bias_path = BASE_PATH / "demographic_bias_analysis"
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    tasks = ['Admission', 'ED_LOS', 'Hosp_LOS']
    colors = {'White': '#2E86AB', 'Black': '#A23B72', 'Hispanic': '#F18F01', 
              'Asian': '#06A77D', 'Other': '#C73E1D'}
    
    for idx, task in enumerate(tasks):
        race_file = bias_path / f"{task}_race_bias.csv"
        if race_file.exists():
            df = pd.read_csv(race_file)
            
            # Sort by sensitivity
            df = df.sort_values('sensitivity', ascending=False)
            
            bars = axes[idx].barh(df['group'], df['sensitivity'], 
                                  color=[colors.get(g, '#999999') for g in df['group']])
            axes[idx].set_xlim([0, 1])
            axes[idx].set_xlabel('Sensitivity')
            axes[idx].set_title(f'{task.replace("_", " ")}')
            axes[idx].axvline(x=df[df['group'] == 'White']['sensitivity'].values[0], 
                             color='red', linestyle='--', alpha=0.7, label='White reference')
            
            # Add value labels
            for bar, val in zip(bars, df['sensitivity']):
                axes[idx].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                              f'{val:.1%}', ha='left', va='center', fontsize=8)
            
            if idx == 0:
                axes[idx].legend(loc='lower right')
    
    fig.suptitle('Demographic Bias Analysis by Race', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURE_PATH / 'Figure3_Race_Bias_Analysis.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_PATH / 'Figure3_Race_Bias_Analysis.pdf', bbox_inches='tight')
    plt.close()
    print("✅ Saved: Figure3_Race_Bias_Analysis.png/pdf")
    
    # Figure 3b: Age bias
    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))
    
    for idx, task in enumerate(tasks):
        age_file = bias_path / f"{task}_age_bias.csv"
        if age_file.exists():
            df = pd.read_csv(age_file)
            df = df.sort_values('group')
            
            bars = axes2[idx].bar(df['group'], df['sensitivity'], color='#2E86AB')
            axes2[idx].set_ylim([0, 1])
            axes2[idx].set_ylabel('Sensitivity')
            axes2[idx].set_title(f'{task.replace("_", " ")}')
            axes2[idx].tick_params(axis='x', rotation=15)
            
            # Add value labels
            for bar, val in zip(bars, df['sensitivity']):
                axes2[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                               f'{val:.1%}', ha='center', va='bottom', fontsize=8)
    
    fig2.suptitle('Age Group Bias Analysis', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURE_PATH / 'Figure3b_Age_Bias_Analysis.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_PATH / 'Figure3b_Age_Bias_Analysis.pdf', bbox_inches='tight')
    plt.close()
    print("✅ Saved: Figure3b_Age_Bias_Analysis.png/pdf")

def create_figure4_performance_radar(results):
    """Figure 4: Radar chart comparing model performance across metrics"""
    
    df = results.get('ranking')
    if df is None or len(df) == 0:
        print("No ranking data found")
        return None
    
    # Get top 3 models for Admission task
    admission_df = df[df['Task'] == 'Hospital Admission'].head(4)
    models = admission_df['Model'].tolist()
    
    # Metrics for radar chart
    metrics = ['AUC-ROC', 'Sensitivity', 'Specificity', 'F1']
    
    # Normalize metrics to 0-1 scale
    def normalize(series):
        return (series - series.min()) / (series.max() - series.min())
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
    
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # Close the loop
    
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#06A77D']
    
    for i, (_, row) in enumerate(admission_df.iterrows()):
        values = []
        for metric in metrics:
            val = row[metric]
            if pd.isna(val):
                val = 0
            values.append(val)
        values += values[:1]
        
        ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_ylim([0, 1])
    ax.set_title('Model Performance Radar Chart (Hospital Admission)', fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.grid(True)
    
    plt.tight_layout()
    plt.savefig(FIGURE_PATH / 'Figure4_Performance_Radar.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIGURE_PATH / 'Figure4_Performance_Radar.pdf', bbox_inches='tight')
    plt.close()
    print("✅ Saved: Figure4_Performance_Radar.png/pdf")

def create_summary_statistics(results):
    """Create summary statistics table"""
    
    df = results.get('complete')
    if df is None:
        print("No complete results found")
        return None
    
    summary = []
    
    for task in df['Task'].unique():
        task_df = df[df['Task'] == task]
        esi_row = task_df[task_df['Type'] == 'ESI']
        ml_rows = task_df[task_df['Type'] == 'ML']
        
        if len(esi_row) > 0 and len(ml_rows) > 0:
            best_ml = ml_rows.loc[ml_rows['AUC-ROC'].idxmax()]
            
            summary.append({
                'Task': task,
                'ESI_AUC': esi_row['AUC-ROC'].values[0],
                'Best_ML_Model': best_ml['Model'],
                'Best_ML_AUC': best_ml['AUC-ROC'],
                'Improvement': best_ml['Improvement_vs_ESI_AUC'],
                'Improvement_%': best_ml['Improvement_Percent']
            })
    
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(RESULTS_PATH / 'Summary_Statistics.csv', index=False)
    print("✅ Saved: Summary_Statistics.csv")
    
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    print(summary_df.to_string(index=False))
    
    return summary_df

def main():
    """Main function to create all figures"""
    
    print("="*80)
    print("CREATING PUBLICATION-READY FIGURES")
    print("="*80)
    
    # Load results
    results = load_results()
    
    # Create figures
    create_figure1_esi_vs_ml(results)
    create_figure2_model_ranking(results)
    create_figure3_bias_analysis(results)
    create_figure4_performance_radar(results)
    create_summary_statistics(results)
    
    print("\n" + "="*80)
    print(f"✅ ALL FIGURES SAVED TO: {FIGURE_PATH}")
    print("="*80)
    print("\n📁 Files created:")
    for f in FIGURE_PATH.glob("*.*"):
        print(f"   - {f.name}")
    print("="*80)

if __name__ == "__main__":
    main()

CREATING PUBLICATION-READY FIGURES
✅ Loaded Complete Results: 18 rows
✅ Loaded ESI vs ML: 6 rows
✅ Loaded Model Ranking: 15 rows
✅ Saved: Figure1_ESI_vs_ML_Performance.png/pdf
✅ Saved: Figure2_Model_Ranking_Heatmap.png/pdf
✅ Saved: Figure3_Race_Bias_Analysis.png/pdf
✅ Saved: Figure3b_Age_Bias_Analysis.png/pdf
✅ Saved: Figure4_Performance_Radar.png/pdf
✅ Saved: Summary_Statistics.csv

SUMMARY STATISTICS
              Task  ESI_AUC  Best_ML_Model  Best_ML_AUC  Improvement  Improvement_%
Hospital Admission 0.504701 VotingEnsemble     0.818056     0.313355      62.087192
       ED Stay ≥8h 0.493772   RandomForest     0.660268     0.166496      33.719230
 Hospital Stay >7d 0.477488 VotingEnsemble     0.666790     0.189301      39.645198

✅ ALL FIGURES SAVED TO: E:\TSINGHUA\thisis\Paper\output_paper\paper_results_combined\figures

📁 Files created:
   - Figure1_ESI_vs_ML_Performance.pdf
   - Figure1_ESI_vs_ML_Performance.png
   - Figure2_Model_Ranking_Heatmap.pdf
   - Figure2_Model_Ranking_He

In [20]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

# Configure paths
OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"

print("="*80)
print("DETAILED DATA STATISTICS FOR ALL THREE TASKS")
print("="*80)

# ============================================================================
# TASK 1: ED-LOS ≥8h
# ============================================================================
print("\n" + "="*80)
print("📌 TASK 1: ED-LOS ≥8 HOURS PREDICTION")
print("="*80)

y_train_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_admit.pkl'))
y_val_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_admit.pkl'))
y_test_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

X_train = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))

ed_los_train = y_train_admit['TARGET_ED_LOS_over8h'].astype(float)
ed_los_val = y_val_admit['TARGET_ED_LOS_over8h'].astype(float)
ed_los_test = y_test_admit['TARGET_ED_LOS_over8h'].astype(float)

print(f"\n📊 SAMPLE SIZES:")
print(f"   Train: {len(ed_los_train):,} samples")
print(f"   Validation: {len(ed_los_val):,} samples")
print(f"   Test: {len(ed_los_test):,} samples")
print(f"   Total: {len(ed_los_train) + len(ed_los_val) + len(ed_los_test):,} samples")

print(f"\n📈 FEATURE DIMENSIONS:")
print(f"   Train features: {X_train.shape}")
print(f"   Val features: {X_val.shape}")
print(f"   Test features: {X_test.shape}")
print(f"   Number of features: {X_train.shape[1]}")

print(f"\n🎯 CLASS DISTRIBUTION (ED-LOS ≥8h):")
for name, data in [('Train', ed_los_train), ('Val', ed_los_val), ('Test', ed_los_test)]:
    pos = int(data.sum())
    neg = int(len(data) - pos)
    prevalence = data.mean() * 100
    print(f"   {name:6s}: Negative (≤8h)={neg:,} | Positive (≥8h)={pos:,} | Prevalence={prevalence:.2f}%")

# ============================================================================
# TASK 2: HOSPITAL ADMISSION
# ============================================================================
print("\n" + "="*80)
print("📌 TASK 2: HOSPITAL ADMISSION PREDICTION")
print("="*80)

admit_train = y_train_admit['TARGET_Admitted'].astype(float)
admit_val = y_val_admit['TARGET_Admitted'].astype(float)
admit_test = y_test_admit['TARGET_Admitted'].astype(float)

print(f"\n📊 SAMPLE SIZES:")
print(f"   Train: {len(admit_train):,} samples")
print(f"   Validation: {len(admit_val):,} samples")
print(f"   Test: {len(admit_test):,} samples")
print(f"   Total: {len(admit_train) + len(admit_val) + len(admit_test):,} samples")

print(f"\n🎯 CLASS DISTRIBUTION (Admission):")
for name, data in [('Train', admit_train), ('Val', admit_val), ('Test', admit_test)]:
    pos = int(data.sum())
    neg = int(len(data) - pos)
    prevalence = data.mean() * 100
    print(f"   {name:6s}: Negative (Discharge)={neg:,} | Positive (Admitted)={pos:,} | Prevalence={prevalence:.2f}%")

# Calculate admission breakdown
print(f"\n📋 ADMISSION CATEGORIES (Train set):")
if 'ED_dispo' in y_train_admit.columns:
    dispo_counts = y_train_admit['ED_dispo'].value_counts()
    for category, count in dispo_counts.items():
        print(f"   {category:20s}: {count:,} ({count/len(admit_train)*100:.1f}%)")

# ============================================================================
# TASK 3: HOSP-LOS >7 DAYS (Admitted Patients Only)
# ============================================================================
print("\n" + "="*80)
print("📌 TASK 3: HOSP-LOS >7 DAYS PREDICTION (ADMITTED PATIENTS ONLY)")
print("="*80)

y_train_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_hosp.pkl'))
y_val_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_hosp.pkl'))
y_test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))

X_train_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train_hosp.pkl'))
X_val_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val_hosp.pkl'))
X_test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test_hosp.pkl'))

hosp_train = pd.to_numeric(y_train_hosp['TARGET_Hosp_LOS_over7d'], errors='coerce').dropna()
hosp_val = pd.to_numeric(y_val_hosp['TARGET_Hosp_LOS_over7d'], errors='coerce').dropna()
hosp_test = pd.to_numeric(y_test_hosp['TARGET_Hosp_LOS_over7d'], errors='coerce').dropna()

print(f"\n📊 SAMPLE SIZES (Admitted Patients Only):")
print(f"   Train: {len(hosp_train):,} samples")
print(f"   Validation: {len(hosp_val):,} samples")
print(f"   Test: {len(hosp_test):,} samples")
print(f"   Total: {len(hosp_train) + len(hosp_val) + len(hosp_test):,} samples")

print(f"\n📈 FEATURE DIMENSIONS:")
print(f"   Train features: {X_train_hosp.shape}")
print(f"   Val features: {X_val_hosp.shape}")
print(f"   Test features: {X_test_hosp.shape}")
print(f"   Number of features: {X_train_hosp.shape[1]}")

print(f"\n🎯 CLASS DISTRIBUTION (Hosp-LOS >7d):")
for name, data in [('Train', hosp_train), ('Val', hosp_val), ('Test', hosp_test)]:
    pos = int(data.sum())
    neg = int(len(data) - pos)
    prevalence = data.mean() * 100
    print(f"   {name:6s}: Negative (≤7d)={neg:,} | Positive (>7d)={pos:,} | Prevalence={prevalence:.2f}%")

# Show filtering details
print(f"\n🔍 FILTERING DETAILS:")
print(f"   Original total patients (Train): {len(admit_train):,}")
print(f"   Admitted patients (Train): {int(admit_train.sum()):,}")
print(f"   Used for Hosp-LOS (Train): {len(hosp_train):,}")
print(f"   → Filtering rate: {len(hosp_train)/admit_train.sum()*100:.1f}% of admitted patients")

# ============================================================================
# COMPARISON SUMMARY
# ============================================================================
print("\n" + "="*80)
print("📊 COMPARISON SUMMARY")
print("="*80)

print(f"\n{'Task':<25} {'Train':>12} {'Val':>12} {'Test':>12} {'Features':>10}")
print("-"*75)
print(f"{'ED-LOS ≥8h':<25} {len(ed_los_train):>12,} {len(ed_los_val):>12,} {len(ed_los_test):>12,} {X_train.shape[1]:>10}")
print(f"{'Admission':<25} {len(admit_train):>12,} {len(admit_val):>12,} {len(admit_test):>12,} {X_train.shape[1]:>10}")
print(f"{'Hosp-LOS >7d':<25} {len(hosp_train):>12,} {len(hosp_val):>12,} {len(hosp_test):>12,} {X_train_hosp.shape[1]:>10}")

print(f"\n{'Task':<25} {'Test Prevalence':>20} {'Imbalance Ratio':>20}")
print("-"*65)
print(f"{'ED-LOS ≥8h':<25} {ed_los_test.mean()*100:>19.2f}% {((1-ed_los_test.mean())/ed_los_test.mean()):>19.2f}:1")
print(f"{'Admission':<25} {admit_test.mean()*100:>19.2f}% {((1-admit_test.mean())/admit_test.mean()):>19.2f}:1")
print(f"{'Hosp-LOS >7d':<25} {hosp_test.mean()*100:>19.2f}% {((1-hosp_test.mean())/hosp_test.mean()):>19.2f}:1")

# ============================================================================
# SAVE SUMMARY
# ============================================================================
summary = {
    'task1_ed_los': {
        'train_samples': len(ed_los_train),
        'val_samples': len(ed_los_val),
        'test_samples': len(ed_los_test),
        'features': X_train.shape[1],
        'test_prevalence': float(ed_los_test.mean()),
        'test_positive': int(ed_los_test.sum()),
        'test_negative': int(len(ed_los_test) - ed_los_test.sum())
    },
    'task2_admission': {
        'train_samples': len(admit_train),
        'val_samples': len(admit_val),
        'test_samples': len(admit_test),
        'features': X_train.shape[1],
        'test_prevalence': float(admit_test.mean()),
        'test_positive': int(admit_test.sum()),
        'test_negative': int(len(admit_test) - admit_test.sum())
    },
    'task3_hosp_los': {
        'train_samples': len(hosp_train),
        'val_samples': len(hosp_val),
        'test_samples': len(hosp_test),
        'features': X_train_hosp.shape[1],
        'test_prevalence': float(hosp_test.mean()),
        'test_positive': int(hosp_test.sum()),
        'test_negative': int(len(hosp_test) - hosp_test.sum()),
        'filtering_note': 'Admitted patients only'
    }
}

summary_path = os.path.join(OUTPUT_PATH, 'data_statistics_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n💾 Summary saved to: {summary_path}")
print("="*80)

DETAILED DATA STATISTICS FOR ALL THREE TASKS

📌 TASK 1: ED-LOS ≥8 HOURS PREDICTION

📊 SAMPLE SIZES:
   Train: 81,588 samples
   Validation: 11,770 samples
   Test: 12,020 samples
   Total: 105,378 samples

📈 FEATURE DIMENSIONS:
   Train features: (81588, 29)
   Val features: (11770, 29)
   Test features: (12020, 29)
   Number of features: 29

🎯 CLASS DISTRIBUTION (ED-LOS ≥8h):
   Train : Negative (≤8h)=66,067 | Positive (≥8h)=15,521 | Prevalence=19.02%
   Val   : Negative (≤8h)=9,502 | Positive (≥8h)=2,268 | Prevalence=19.27%
   Test  : Negative (≤8h)=9,339 | Positive (≥8h)=2,681 | Prevalence=22.30%

📌 TASK 2: HOSPITAL ADMISSION PREDICTION

📊 SAMPLE SIZES:
   Train: 81,588 samples
   Validation: 11,770 samples
   Test: 12,020 samples
   Total: 105,378 samples

🎯 CLASS DISTRIBUTION (Admission):
   Train : Negative (Discharge)=48,855 | Positive (Admitted)=32,733 | Prevalence=40.12%
   Val   : Negative (Discharge)=6,950 | Positive (Admitted)=4,820 | Prevalence=40.95%
   Test  : Negative (

In [21]:
# CELL X - SUBGROUP SENSITIVITY ANALYSIS (ALL 3 TASKS) - FIXED
# Reviewer request: Run model on sub-samples by CC, Age, Triage Acuity
# For: (1) ED-LOS ≥8h, (2) Hospital Admission, (3) Hosp-LOS >7d

import pandas as pd
import numpy as np
import logging
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

logger = logging.getLogger(__name__)
OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"

logger.info("="*80)
logger.info("🔬 SUBGROUP SENSITIVITY ANALYSIS (ALL 3 TASKS)")
logger.info("Subgroups: CC | Age | Triage Acuity")
logger.info("Tasks: ED-LOS ≥8h | Admission | Hosp-LOS >7d")
logger.info("="*80)

# ============================================================================
# 1. LOAD ALL DATA
# ============================================================================
# Feature matrices (same for all tasks)
X_train = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))

# Task 1 & 2 targets (admission-available)
y_train_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_admit.pkl'))
y_val_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_admit.pkl'))
y_test_admit = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

# Task 3 targets (admitted patients only - different indices!)
X_train_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train_hosp.pkl'))
X_val_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val_hosp.pkl'))
X_test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test_hosp.pkl'))

# FIXED: Load Hosp-LOS targets as Series, then extract values
y_train_hosp_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_hosp.pkl'))
y_val_hosp_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_hosp.pkl'))
y_test_hosp_df = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))

# Extract the target column as numpy array
y_train_hosp = y_train_hosp_df['TARGET_Hosp_LOS_over7d'].values.astype(int)
y_val_hosp = y_val_hosp_df['TARGET_Hosp_LOS_over7d'].values.astype(int)
y_test_hosp = y_test_hosp_df['TARGET_Hosp_LOS_over7d'].values.astype(int)

# Load raw test data for CC text (in case needed)
test_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'test_data', 'visits.pkl'))

# ============================================================================
# 2. DEFINE SUBGROUPS (same for all tasks)
# ============================================================================
SUBGROUPS = {
    # Chief Complaint
    'CC: Abdominal Pain': X_test['CC: Abdominal Pain'] == 1,
    'CC: Chest Pain': X_test['CC: Chest Pain'] == 1,
    'CC: Shortness of Breath': X_test['CC: Shortness of Breath'] == 1,
    'CC: Fall': X_test['CC:Fall'] == 1,
    'CC: Fever': X_test['CC: Fever'] == 1,
    'CC: Critical': X_test['CC:Critical'] == 1,
    'CC: Other': ~(X_test[[c for c in X_test.columns if c.startswith('CC:')]].any(axis=1)),
    
    # Age subgroups
    'Age <65': X_test['Age'] < 65,
    'Age 65-74': (X_test['Age'] >= 65) & (X_test['Age'] < 75),
    'Age 75-84': (X_test['Age'] >= 75) & (X_test['Age'] < 85),
    'Age ≥85': X_test['Age'] >= 85,
    
    # Triage Acuity subgroups
    'Acuity 1 (Resuscitation)': X_test['Triage Acuity Level'] == 1,
    'Acuity 2 (Emergent)': X_test['Triage Acuity Level'] == 2,
    'Acuity 3 (Urgent)': X_test['Triage Acuity Level'] == 3,
    'Acuity 4-5 (Semi/Non-urgent)': X_test['Triage Acuity Level'] >= 4,
}

# ============================================================================
# 3. TRAIN MODELS FOR EACH TASK (once, reuse for all subgroups)
# ============================================================================
logger.info("\n📊 Training base models...")

# Preprocessing
imputer = SimpleImputer(strategy='median')
scaler = RobustScaler()

X_train_imp = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_imp = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imp)

# Task 1: ED-LOS ≥8h
y_train_ed = y_train_admit['TARGET_ED_LOS_over8h'].values.astype(int)
y_test_ed = y_test_admit['TARGET_ED_LOS_over8h'].values.astype(int)

model_ed = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=4.26, tree_method='hist', random_state=42, verbosity=0
)
model_ed.fit(X_train_scaled, y_train_ed)
y_pred_ed = model_ed.predict_proba(X_test_scaled)[:, 1]
overall_auc_ed = roc_auc_score(y_test_ed, y_pred_ed)

# Task 2: Hospital Admission
y_train_admit_target = y_train_admit['TARGET_Admitted'].values.astype(int)
y_test_admit_target = y_test_admit['TARGET_Admitted'].values.astype(int)

model_admit = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=1.49, tree_method='hist', random_state=42, verbosity=0
)
model_admit.fit(X_train_scaled, y_train_admit_target)
y_pred_admit = model_admit.predict_proba(X_test_scaled)[:, 1]
overall_auc_admit = roc_auc_score(y_test_admit_target, y_pred_admit)

# Task 3: Hosp-LOS >7d (different preprocessing - admitted patients only)
X_train_hosp_imp = imputer.fit_transform(X_train_hosp)
X_train_hosp_scaled = scaler.fit_transform(X_train_hosp_imp)
X_test_hosp_imp = imputer.transform(X_test_hosp)
X_test_hosp_scaled = scaler.transform(X_test_hosp_imp)

# Calculate scale_pos_weight for Hosp-LOS
train_pos_hosp = y_train_hosp.sum()
train_neg_hosp = len(y_train_hosp) - train_pos_hosp
hosp_scale_weight = train_neg_hosp / train_pos_hosp if train_pos_hosp > 0 else 1

model_hosp = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=hosp_scale_weight, tree_method='hist', random_state=42, verbosity=0
)
model_hosp.fit(X_train_hosp_scaled, y_train_hosp)
y_pred_hosp = model_hosp.predict_proba(X_test_hosp_scaled)[:, 1]

# FIXED: Ensure y_test_hosp is 1D array
if isinstance(y_test_hosp, pd.Series):
    y_test_hosp = y_test_hosp.values
elif isinstance(y_test_hosp, pd.DataFrame):
    y_test_hosp = y_test_hosp.iloc[:, 0].values

overall_auc_hosp = roc_auc_score(y_test_hosp, y_pred_hosp)

logger.info(f"   Task 1 (ED-LOS) overall AUC: {overall_auc_ed:.4f}")
logger.info(f"   Task 2 (Admission) overall AUC: {overall_auc_admit:.4f}")
logger.info(f"   Task 3 (Hosp-LOS) overall AUC: {overall_auc_hosp:.4f}")

# ============================================================================
# 4. EVALUATE ALL SUBGROUPS FOR ALL TASKS
# ============================================================================
def evaluate_subgroup(mask, y_true, y_pred, subgroup_name, task_name):
    """Evaluate a single subgroup for one task"""
    y_sub = y_true[mask]
    
    if len(y_sub) < 30 or len(np.unique(y_sub)) != 2:
        return None
    
    y_pred_sub = y_pred[mask]
    auc = roc_auc_score(y_sub, y_pred_sub)
    auprc = average_precision_score(y_sub, y_pred_sub)
    
    return {
        'Task': task_name,
        'Subgroup': subgroup_name,
        'N': len(y_sub),
        'Prevalence': y_sub.mean(),
        'AUC': auc,
        'AUPRC': auprc
    }

# Collect results for all tasks
all_results = []

for subgroup_name, mask in SUBGROUPS.items():
    # Task 1
    res1 = evaluate_subgroup(mask, y_test_ed, y_pred_ed, subgroup_name, 'ED-LOS ≥8h')
    if res1:
        all_results.append(res1)
    
    # Task 2
    res2 = evaluate_subgroup(mask, y_test_admit_target, y_pred_admit, subgroup_name, 'Hospital Admission')
    if res2:
        all_results.append(res2)

# Task 3: Need to align subgroups with Hosp-LOS test set (different indices)
# Get indices of Hosp-LOS test set
hosp_test_indices = X_test_hosp.index

for subgroup_name, full_mask in SUBGROUPS.items():
    # Create mask for Hosp-LOS test set indices
    # full_mask is a pandas Series/boolean array aligned with X_test (12020 samples)
    # We need to subset to the Hosp-LOS test set indices (4765 samples)
    hosp_mask = full_mask.iloc[hosp_test_indices]
    
    res3 = evaluate_subgroup(hosp_mask, y_test_hosp, y_pred_hosp, subgroup_name, 'Hosp-LOS >7d')
    if res3:
        all_results.append(res3)

results_df = pd.DataFrame(all_results)

# ============================================================================
# 5. CREATE COMPARISON TABLES
# ============================================================================
# Pivot table: Subgroup x Task = AUC
pivot_auc = results_df.pivot_table(index='Subgroup', columns='Task', values='AUC')
pivot_n = results_df.pivot_table(index='Subgroup', columns='Task', values='N', aggfunc='first')

logger.info("\n" + "="*80)
logger.info("📊 SUBGROUP AUC BY TASK")
logger.info("="*80)
logger.info("\n" + pivot_auc.round(3).to_string())

# ============================================================================
# 6. CREATE VISUALIZATION (3-panel figure)
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

tasks = ['ED-LOS ≥8h', 'Hospital Admission', 'Hosp-LOS >7d']
colors_task = {'ED-LOS ≥8h': '#3498db', 'Hospital Admission': '#2ecc71', 'Hosp-LOS >7d': '#e74c3c'}

for idx, task in enumerate(tasks):
    ax = axes[idx]
    task_results = results_df[results_df['Task'] == task].copy()
    task_results = task_results.sort_values('AUC', ascending=True)
    
    overall_auc = {
        'ED-LOS ≥8h': overall_auc_ed,
        'Hospital Admission': overall_auc_admit,
        'Hosp-LOS >7d': overall_auc_hosp
    }[task]
    
    y_pos = np.arange(len(task_results))
    aucs = task_results['AUC'].values
    
    bars = ax.barh(y_pos, aucs, color=colors_task[task], alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.axvline(x=overall_auc, color='black', linestyle='--', linewidth=2, 
               label=f'Overall AUC = {overall_auc:.3f}')
    
    # Add sample size annotations
    for i, (bar, row) in enumerate(zip(bars, task_results.itertuples())):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'n={row.N}', va='center', fontsize=8)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(task_results['Subgroup'].str.replace('CC: ', ''))
    ax.set_xlabel('AUC-ROC')
    ax.set_title(f'{task}', fontsize=12)
    ax.legend(loc='lower right', fontsize=9)
    ax.set_xlim(0.5, 0.9)
    ax.grid(alpha=0.3, axis='x')

plt.suptitle('Subgroup Sensitivity Analysis Across All Three Prediction Tasks', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'subgroup_analysis_all_tasks.png'), dpi=300, bbox_inches='tight')
plt.close()

logger.info(f"\n✅ Figure saved: {OUTPUT_PATH}/subgroup_analysis_all_tasks.png")

# ============================================================================
# 7. SUMMARY TABLE FOR PAPER
# ============================================================================
# Create formatted table
summary_table = results_df.pivot_table(
    index='Subgroup', 
    columns='Task', 
    values='AUC',
    aggfunc='first'
).round(3)

# Add sample sizes
n_table = results_df.pivot_table(
    index='Subgroup', 
    columns='Task', 
    values='N',
    aggfunc='first'
)

# Combined table with N in parentheses
for task in tasks:
    if task in summary_table.columns and task in n_table.columns:
        summary_table[task] = summary_table[task].astype(str) + ' (n=' + n_table[task].astype(int).astype(str) + ')'

summary_table.to_csv(os.path.join(OUTPUT_PATH, 'subgroup_analysis_summary.csv'))

logger.info("\n" + "="*80)
logger.info("📝 SUMMARY TABLE FOR PAPER")
logger.info("="*80)
logger.info("\n" + summary_table.to_string())

# ============================================================================
# 8. KEY FINDINGS FOR REVIEWER
# ============================================================================
logger.info("\n" + "="*80)
logger.info("📝 KEY FINDINGS FOR REVIEWER RESPONSE")
logger.info("="*80)

for task in tasks:
    task_df = results_df[results_df['Task'] == task]
    if len(task_df) > 0:
        best = task_df.loc[task_df['AUC'].idxmax()]
        worst = task_df.loc[task_df['AUC'].idxmin()]
        logger.info(f"\n{task}:")
        logger.info(f"   Best: {best['Subgroup']} (AUC={best['AUC']:.3f}, n={best['N']})")
        logger.info(f"   Worst: {worst['Subgroup']} (AUC={worst['AUC']:.3f}, n={worst['N']})")
        logger.info(f"   Range: {task_df['AUC'].max() - task_df['AUC'].min():.3f}")

# Age consistency check
logger.info(f"\nAge Group Performance (AUC range across tasks):")
for task in tasks:
    age_df = results_df[(results_df['Task'] == task) & (results_df['Subgroup'].str.startswith('Age'))]
    if len(age_df) > 0:
        age_range = age_df['AUC'].max() - age_df['AUC'].min()
        logger.info(f"   {task}: {age_range:.3f}")

logger.info("\n" + "="*80)
logger.info("✅ SUBGROUP ANALYSIS COMPLETE (ALL 3 TASKS)")
logger.info("="*80)

2026-07-24 16:10:07 | INFO | ================================================================================
2026-07-24 16:10:07 | INFO | 🔬 SUBGROUP SENSITIVITY ANALYSIS (ALL 3 TASKS)
2026-07-24 16:10:07 | INFO | Subgroups: CC | Age | Triage Acuity
2026-07-24 16:10:07 | INFO | Tasks: ED-LOS ≥8h | Admission | Hosp-LOS >7d
2026-07-24 16:10:07 | INFO | ================================================================================
2026-07-24 16:10:07 | INFO | 
📊 Training base models...
2026-07-24 16:10:12 | INFO |    Task 1 (ED-LOS) overall AUC: 0.6672
2026-07-24 16:10:12 | INFO |    Task 2 (Admission) overall AUC: 0.8162
2026-07-24 16:10:12 | INFO |    Task 3 (Hosp-LOS) overall AUC: 0.6751
2026-07-24 16:10:12 | INFO | 
2026-07-24 16:10:12 | INFO | 📊 SUBGROUP AUC BY TASK
2026-07-24 16:10:12 | INFO | ================================================================================
2026-07-24 16:10:12 | INFO | 
Task                          ED-LOS ≥8h  Hosp-LOS >7d  Hospital Admission
Subg

In [22]:
# CELL: Generate Evidence for Reviewer Responses
import pandas as pd
import numpy as np
import json
import os
from sklearn.metrics import confusion_matrix, roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

OUTPUT_PATH = r"E:\TSINGHUA\thisis\Paper\output_paper"

# ============================================================================
# 1. LOAD ALL DATA
# ============================================================================

# Load feature matrices
X_train = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_train.pkl'))
X_val = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_val.pkl'))
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))

# Load targets
y_train_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_admit.pkl'))
y_val_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_admit.pkl'))
y_test_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

# Load Hosp-LOS targets
y_train_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_train_hosp.pkl'))
y_val_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_val_hosp.pkl'))
y_test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))

# Load predictions from Task 1 (ED-LOS)
task1_path = os.path.join(OUTPUT_PATH, 'task1_ed_los', 'all_predictions.csv')
if os.path.exists(task1_path):
    preds_edlos = pd.read_csv(task1_path)
else:
    print("Task 1 predictions not found. Run CELL 5 first.")
    preds_edlos = None

# Load predictions from Task 2 (Admission)
task2_path = os.path.join(OUTPUT_PATH, 'task2_admission', 'all_predictions.csv')
if os.path.exists(task2_path):
    preds_admission = pd.read_csv(task2_path)
else:
    print("Task 2 predictions not found. Run CELL 6 first.")
    preds_admission = None

# Load predictions from Task 3 (Hosp-LOS)
task3_path = os.path.join(OUTPUT_PATH, 'task3_hosp_los', 'all_predictions.csv')
if os.path.exists(task3_path):
    preds_hosplos = pd.read_csv(task3_path)
else:
    print("Task 3 predictions not found. Run CELL 7 first.")
    preds_hosplos = None

# ============================================================================
# 2. EVIDENCE FOR REVIEWER 1 - ISSUE 1: NUMERICAL INCONSISTENCIES
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 1 - ISSUE 1: NUMERICAL INCONSISTENCIES")
print("="*80)

if preds_edlos is not None:
    y_true = preds_edlos['y_true'].values
    
    for model in ['XGBoost', 'LightGBM', 'RandomForest', 'TabNet', 'VotingEnsemble']:
        y_pred = preds_edlos[model].values
        
        # Calculate AUC
        auc = roc_auc_score(y_true, y_pred)
        
        # Find optimal threshold using Youden's J
        from sklearn.metrics import roc_curve
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        youden = tpr - fpr
        optimal_idx = np.argmax(youden)
        optimal_threshold = thresholds[optimal_idx]
        
        # Apply threshold
        y_pred_class = (y_pred >= optimal_threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()
        
        # Calculate metrics
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
        
        print(f"\n{model} (ED-LOS):")
        print(f"  Threshold: {optimal_threshold:.3f}")
        print(f"  AUC-ROC: {auc:.4f}")
        print(f"  Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
        print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
        print(f"  Sensitivity: {sensitivity:.4f} ({sensitivity*100:.1f}%)")
        print(f"  Specificity: {specificity:.4f} ({specificity*100:.1f}%)")
        print(f"  Precision: {precision:.4f} ({precision*100:.1f}%)")
        print(f"  F1: {f1:.4f}")
        
        # Verify mathematical consistency
        expected_acc = (sensitivity * len(y_true[y_true==1]) + specificity * len(y_true[y_true==0])) / len(y_true)
        print(f"  Consistency check - expected accuracy: {expected_acc:.4f}")

# ============================================================================
# 3. EVIDENCE FOR REVIEWER 1 - ISSUE 3: ADMISSION RATE CONFLICT
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 1 - ISSUE 3: ADMISSION RATE CONFLICT")
print("="*80)

print("\nFull Cohort Admission Rates:")
print(f"  Training set: {y_train_full['TARGET_Admitted'].mean():.3f} ({y_train_full['TARGET_Admitted'].mean()*100:.1f}%)")
print(f"  Validation set: {y_val_full['TARGET_Admitted'].mean():.3f} ({y_val_full['TARGET_Admitted'].mean()*100:.1f}%)")
print(f"  Test set: {y_test_full['TARGET_Admitted'].mean():.3f} ({y_test_full['TARGET_Admitted'].mean()*100:.1f}%)")
print(f"  Full cohort: {y_train_full['TARGET_Admitted'].mean():.3f} ({y_train_full['TARGET_Admitted'].mean()*100:.1f}%)")

print("\nAdmitted Patients with Hospital LOS:")
print(f"  Training: {len(y_train_hosp):,} patients")
print(f"  Validation: {len(y_val_hosp):,} patients")
print(f"  Test: {len(y_test_hosp):,} patients")
print(f"  Total admitted: {len(y_train_hosp) + len(y_val_hosp) + len(y_test_hosp):,} patients")

# ============================================================================
# 4. EVIDENCE FOR REVIEWER 1 - ISSUE 5: MISSING DATA
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 1 - ISSUE 5: MISSING DATA")
print("="*80)

# Check missingness in training data
missing_counts = X_train.isnull().sum()
missing_pct = (missing_counts / len(X_train)) * 100

print("\nMissing Data in Training Set:")
print(f"  Total samples: {len(X_train):,}")
print(f"  Features with missing values:")
for col in missing_counts[missing_counts > 0].index:
    print(f"    {col}: {missing_counts[col]:,} ({missing_pct[col]:.1f}%)")

# ============================================================================
# 5. EVIDENCE FOR REVIEWER 1 - ISSUE 6: DATA LEAKAGE CHECK
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 1 - ISSUE 6: DATA LEAKAGE PREVENTION")
print("="*80)

# Load raw data to check temporal filtering
DATA_PATH = r"D:\Thesis\Data"

try:
    meds_df = pd.read_csv(os.path.join(DATA_PATH, 'meds.csv'), 
                          dtype={'MRN': str}, nrows=1000)
    pmh_df = pd.read_csv(os.path.join(DATA_PATH, 'pmh.csv'), 
                         dtype={'MRN': str}, nrows=1000)
    
    print("\nMedications data structure:")
    print(f"  Columns: {meds_df.columns.tolist()}")
    print(f"  Date columns present: {'Start_date' in meds_df.columns}")
    
    print("\nPMH data structure:")
    print(f"  Columns: {pmh_df.columns.tolist()}")
    print(f"  Date columns present: {'Noted_date' in pmh_df.columns}")
    
except Exception as e:
    print(f"  Could not load raw data: {e}")

# ============================================================================
# 6. EVIDENCE FOR REVIEWER 1 - ISSUE 7: HYPERPARAMETERS
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 1 - ISSUE 7: HYPERPARAMETERS")
print("="*80)

# Load class weights
with open(os.path.join(OUTPUT_PATH, 'class_weights.json'), 'r') as f:
    class_weights = json.load(f)

print("\nClass weights by task:")
for task, weights in class_weights.items():
    print(f"\n  {task}:")
    print(f"    scale_pos_weight: {weights['scale_pos_weight']:.2f}")
    print(f"    prevalence: {weights['prevalence']:.3f}")

# ============================================================================
# 7. EVIDENCE FOR REVIEWER 3 - COMMENT 13: TEMPORAL LEAKAGE
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 3 - COMMENT 13: TEMPORAL LEAKAGE")
print("="*80)

print("\nTo verify temporal leakage prevention, check the following in CELL 1:")
print("  1. meds.csv is filtered by MRN only - dates are NOT filtered in current code")
print("  2. pmh.csv is filtered by MRN only - dates are NOT filtered in current code")
print("\nThis is a CRITICAL issue that must be fixed!")

# ============================================================================
# 8. EVIDENCE FOR REVIEWER 3 - COMMENT 19: TRIAGE ACUITY SENSITIVITY
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 3 - COMMENT 19: TRIAGE ACUITY SENSITIVITY")
print("="*80)

# Check triage acuity distribution
acuity_counts = X_train['Triage Acuity Level'].value_counts().sort_index()
print("\nTriage Acuity Distribution (Training Set):")
print(f"  Missing count: {X_train['Triage Acuity Level'].isnull().sum()}")
for level, count in acuity_counts.items():
    print(f"  Level {level}: {count:,} ({count/len(X_train)*100:.1f}%)")

print("\nTo run sensitivity analysis excluding triage acuity:")
print("  - Remove 'Triage Acuity Level' from feature set")
print("  - Retrain models and compare performance")

# ============================================================================
# 9. EVIDENCE FOR REVIEWER 3 - COMMENT 30: METRICS INCOMPATIBILITY
# ============================================================================

print("\n" + "="*80)
print("EVIDENCE FOR REVIEWER 3 - COMMENT 30: METRICS INCOMPATIBILITY")
print("="*80)

# This is a critical issue - we need to verify all metrics
if preds_edlos is not None:
    y_true = preds_edlos['y_true'].values
    prevalence = y_true.mean()
    
    print(f"\nED-LOS Test Set:")
    print(f"  Prevalence: {prevalence:.3f} ({prevalence*100:.1f}%)")
    
    for model in ['XGBoost', 'LightGBM', 'RandomForest', 'TabNet', 'VotingEnsemble']:
        y_pred = preds_edlos[model].values
        
        # Use optimal threshold
        fpr, tpr, thresholds = roc_curve(y_true, y_pred)
        youden = tpr - fpr
        optimal_idx = np.argmax(youden)
        optimal_threshold = thresholds[optimal_idx]
        
        y_pred_class = (y_pred >= optimal_threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()
        
        # Check consistency
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        
        # Verify accuracy from sensitivity, specificity, and prevalence
        expected_accuracy = sensitivity * prevalence + specificity * (1 - prevalence)
        
        print(f"\n  {model}:")
        print(f"    TP={tp}, TN={tn}, FP={fp}, FN={fn}")
        print(f"    Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")
        print(f"    Precision={precision:.4f}")
        print(f"    Reported Accuracy={accuracy:.4f}")
        print(f"    Expected Accuracy={expected_accuracy:.4f}")
        print(f"    Match: {abs(accuracy - expected_accuracy) < 0.01}")

# ============================================================================
# 10. SAVE EVIDENCE FOR RESPONSE
# ============================================================================

# Create evidence summary
evidence_summary = {
    'reviewer_1': {
        'issue_1_metrics_consistency': 'Run the confusion matrix code above',
        'issue_3_admission_rates': {
            'full_cohort': float(y_train_full['TARGET_Admitted'].mean()),
            'task_2_train': float(y_train_full['TARGET_Admitted'].mean()),
            'task_3_samples': len(y_train_hosp) + len(y_val_hosp) + len(y_test_hosp)
        },
        'issue_5_missing_data': missing_counts[missing_counts > 0].to_dict(),
        'issue_7_hyperparameters': class_weights
    },
    'reviewer_3': {
        'comment_13_temporal_leakage': 'CRITICAL: Need to filter by dates in CELL 1',
        'comment_19_triage_acuity': acuity_counts.to_dict(),
        'comment_30_metrics_incompatibility': 'Check using confusion matrix code above'
    }
}

# Save evidence
with open(os.path.join(OUTPUT_PATH, 'reviewer_evidence.json'), 'w') as f:
    json.dump(evidence_summary, f, indent=2, default=str)

print(f"\nEvidence saved to: {os.path.join(OUTPUT_PATH, 'reviewer_evidence.json')}")
print("\n" + "="*80)
print("EVIDENCE GENERATION COMPLETE")
print("="*80)


EVIDENCE FOR REVIEWER 1 - ISSUE 1: NUMERICAL INCONSISTENCIES

XGBoost (ED-LOS):
  Threshold: 0.421
  AUC-ROC: 0.6592
  Confusion Matrix: TP=1919, TN=4837, FP=4502, FN=762
  Accuracy: 0.5621 (56.2%)
  Sensitivity: 0.7158 (71.6%)
  Specificity: 0.5179 (51.8%)
  Precision: 0.2989 (29.9%)
  F1: 0.4217
  Consistency check - expected accuracy: 0.5621

LightGBM (ED-LOS):
  Threshold: 0.226
  AUC-ROC: 0.6478
  Confusion Matrix: TP=2221, TN=3563, FP=5776, FN=460
  Accuracy: 0.4812 (48.1%)
  Sensitivity: 0.8284 (82.8%)
  Specificity: 0.3815 (38.2%)
  Precision: 0.2777 (27.8%)
  F1: 0.4160
  Consistency check - expected accuracy: 0.4812

RandomForest (ED-LOS):
  Threshold: 0.401
  AUC-ROC: 0.6603
  Confusion Matrix: TP=2027, TN=4449, FP=4890, FN=654
  Accuracy: 0.5388 (53.9%)
  Sensitivity: 0.7561 (75.6%)
  Specificity: 0.4764 (47.6%)
  Precision: 0.2930 (29.3%)
  F1: 0.4224
  Consistency check - expected accuracy: 0.5388

TabNet (ED-LOS):
  Threshold: 0.457
  AUC-ROC: 0.6385
  Confusion Matrix:

In [24]:
import numpy as np
from sklearn.metrics import brier_score_loss, average_precision_score, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

# All model predictions from Cell 5
models = {
    'XGBoost': xgb_pred,
    'LightGBM': lgb_pred,
    'RandomForest': rf_pred,
    'TabNet': tabnet_pred,
    'VotingEnsemble': ensemble_pred
}

y_true = y_test  # From Cell 3 (ED-LOS target)

print("="*80)
print("ADDITIONAL METRICS FOR ED-LOS PREDICTION")
print("="*80)

for model_name, y_pred in models.items():
    print(f"\n{'-'*40}")
    print(f"Model: {model_name}")
    print(f"{'-'*40}")
    
    # 1. Brier Score
    brier = brier_score_loss(y_true, y_pred)
    print(f"Brier Score: {brier:.4f}")
    
    # 2. AUC-PR
    auc_pr = average_precision_score(y_true, y_pred)
    print(f"AUC-PR: {auc_pr:.4f}")
    
    # 3. Calibration
    log_odds = np.log(y_pred / (1 - y_pred))
    log_odds = np.clip(log_odds, -10, 10)
    lr = LogisticRegression()
    lr.fit(log_odds.reshape(-1, 1), y_true)
    print(f"Calibration Intercept: {lr.intercept_[0]:.3f}")
    print(f"Calibration Slope: {lr.coef_[0][0]:.3f}")
    
    # 4. Threshold-based performance
    for percentile in [10, 20]:
        threshold = np.percentile(y_pred, 100 - percentile)
        y_pred_binary = (y_pred >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred_binary).ravel()
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        print(f"\n  Top {percentile}% Risk (Threshold >= {threshold:.3f}):")
        print(f"    Sensitivity: {sensitivity:.1%}")
        print(f"    Specificity: {specificity:.1%}")
        print(f"    PPV: {ppv:.1%}")

ADDITIONAL METRICS FOR ED-LOS PREDICTION

----------------------------------------
Model: XGBoost
----------------------------------------
Brier Score: 0.1944
AUC-PR: 0.3118
Calibration Intercept: -1.136
Calibration Slope: 0.693

  Top 10% Risk (Threshold >= 0.622):
    Sensitivity: 19.6%
    Specificity: 92.3%
    PPV: 38.6%

  Top 20% Risk (Threshold >= 0.543):
    Sensitivity: 34.2%
    Specificity: 83.5%
    PPV: 33.7%

----------------------------------------
Model: LightGBM
----------------------------------------
Brier Score: 0.1559
AUC-PR: 0.3059
Calibration Intercept: 4.929
Calibration Slope: 4.834

  Top 10% Risk (Threshold >= 0.232):
    Sensitivity: 18.7%
    Specificity: 92.1%
    PPV: 36.5%

  Top 20% Risk (Threshold >= 0.226):
    Sensitivity: 36.4%
    Specificity: 82.4%
    PPV: 33.6%

----------------------------------------
Model: RandomForest
----------------------------------------
Brier Score: 0.1863
AUC-PR: 0.3278
Calibration Intercept: -0.924
Calibration Slope: 

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, brier_score_loss, roc_auc_score
from sklearn.calibration import calibration_curve
import os
import json

# Set global publication formatting
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['figure.titlesize'] = 14

OUTPUT_DIR = r"E:\TSINGHUA\thisis\Paper\output_paper\figures_and_tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------------------------------
# 1. DECISION CURVE ANALYSIS (DCA) PLOT - ALL TASKS TOGETHER
# -------------------------------------------------------------------------
def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefit = []
    n = len(y_true)
    for t in thresholds:
        tp = np.sum((y_prob >= t) & (y_true == 1))
        fp = np.sum((y_prob >= t) & (y_true == 0))
        if t == 1.0:
            nb = (tp / n) - (fp / n) * 0
        else:
            nb = (tp / n) - (fp / n) * (t / (1 - t))
        net_benefit.append(nb)
    return np.array(net_benefit)

def plot_dca_all_tasks(tasks_data, thresholds=np.linspace(0.01, 0.99, 100)):
    """Plot DCA for all three tasks in a single figure with subplots"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=300)
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    task_names = ['ED-LOS ≥8h', 'Hospital Admission', 'Hosp-LOS >7d']
    task_keys = ['ED_LOS_Over8h', 'Admission', 'Hosp_LOS_Over7d']
    
    for idx, (task_key, task_info) in enumerate(tasks_data.items()):
        ax = axes[idx]
        y_true = task_info['y_true']
        model_predictions = task_info['model_predictions']
        
        # Baseline: Treat All
        p_prev = np.mean(y_true)
        net_benefit_all = p_prev - (1 - p_prev) * (thresholds / (1 - thresholds))
        ax.plot(thresholds, net_benefit_all, label='Treat All', color='gray', linestyle='--', alpha=0.7)
        
        # Baseline: Treat None
        ax.plot(thresholds, np.zeros_like(thresholds), label='Treat None', color='black', linestyle=':')
        
        # Plot Models (excluding Voting Ensemble)
        model_list = [(name, y_prob) for name, y_prob in model_predictions.items() if 'Voting' not in name]
        
        for model_idx, (name, y_prob) in enumerate(model_list):
            nb = calculate_net_benefit(y_true, y_prob, thresholds)
            ax.plot(thresholds, nb, label=name, color=colors[model_idx % len(colors)], linewidth=2)
        
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([-0.05, max(p_prev * 1.1, 0.3)])
        ax.set_xlabel('Threshold Probability')
        ax.set_ylabel('Net Benefit')
        ax.set_title(task_names[idx])
        ax.legend(loc='upper right', frameon=True, fontsize=9)
    
    plt.suptitle('Decision Curve Analysis (DCA) - All Tasks', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig1_DCA_All_Tasks.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Combined DCA plot saved (all 3 tasks)")

# -------------------------------------------------------------------------
# 2. ROC & CALIBRATION PLOTS - ALL TASKS TOGETHER
# -------------------------------------------------------------------------
def plot_roc_and_calibration_all_tasks(tasks_data):
    """Plot ROC and Calibration for all three tasks in a single figure"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=300)
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    task_names = ['ED-LOS ≥8h', 'Hospital Admission', 'Hosp-LOS >7d']
    task_keys = ['ED_LOS_Over8h', 'Admission', 'Hosp_LOS_Over7d']
    
    for idx, (task_key, task_info) in enumerate(tasks_data.items()):
        y_true = task_info['y_true']
        model_predictions = task_info['model_predictions']
        
        # ROC Curve (top row)
        ax_roc = axes[0, idx]
        model_list = [(name, y_prob) for name, y_prob in model_predictions.items() if 'Voting' not in name]
        
        for model_idx, (name, y_prob) in enumerate(model_list):
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            roc_auc = auc(fpr, tpr)
            ax_roc.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', 
                       color=colors[model_idx % len(colors)], lw=2)
        
        ax_roc.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.7)
        ax_roc.set_xlabel('False Positive Rate')
        ax_roc.set_ylabel('True Positive Rate')
        ax_roc.set_title(f'ROC - {task_names[idx]}')
        ax_roc.legend(loc='lower right', fontsize=8)
        
        # Calibration Curve (bottom row)
        ax_cal = axes[1, idx]
        
        for model_idx, (name, y_prob) in enumerate(model_list):
            try:
                prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)
                ax_cal.plot(prob_pred, prob_true, marker='o', label=name, 
                           color=colors[model_idx % len(colors)], lw=2)
            except Exception as e:
                print(f"Warning: Calibration failed for {name} on {task_names[idx]}: {e}")
        
        ax_cal.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.7, label='Perfect')
        ax_cal.set_xlabel('Mean Predicted Probability')
        ax_cal.set_ylabel('Fraction of Positives')
        ax_cal.set_title(f'Calibration - {task_names[idx]}')
        ax_cal.legend(loc='upper left', fontsize=8)
    
    plt.suptitle('ROC and Calibration Curves - All Tasks', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig2_ROC_Calibration_All_Tasks.png"), dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Combined ROC and Calibration plot saved (all 3 tasks)")

# -------------------------------------------------------------------------
# 3. GENERATE TABLE CONTENT (LATEX & CSV) - INDIVIDUAL TASKS
# -------------------------------------------------------------------------
def generate_tables(y_true, model_predictions, task_name):
    metrics_list = []
    model_list = [(name, y_prob) for name, y_prob in model_predictions.items() if 'Voting' not in name]
    
    for name, y_prob in model_list:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        brier = brier_score_loss(y_true, y_prob)
        
        y_pred = (y_prob >= 0.5).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        tn = np.sum((y_pred == 0) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        accuracy = (tp + tn) / len(y_true)
        
        metrics_list.append({
            'Model': name,
            'AUROC': f"{roc_auc:.3f}",
            'Sensitivity': f"{sensitivity:.3f}",
            'Specificity': f"{specificity:.3f}",
            'Accuracy': f"{accuracy:.3f}",
            'Brier Score': f"{brier:.3f}"
        })
        
    df_metrics = pd.DataFrame(metrics_list)
    
    # Save CSV
    csv_path = os.path.join(OUTPUT_DIR, f"Table_{task_name}_Performance.csv")
    df_metrics.to_csv(csv_path, index=False)
    print(f"✅ Table saved for {task_name}")
    
    return df_metrics

# =============================================================================
# EXECUTION: LOAD ALL THREE TASKS
# =============================================================================

OUTPUT_PATH = r"E:\\TSINGHUA\\thisis\\Paper\\output_paper"

# Define tasks and their prediction file paths
tasks = {
    'ED_LOS_Over8h': {
        'path': os.path.join(OUTPUT_PATH, 'task1_ed_los', 'all_predictions.csv'),
        'label': 'ED-LOS ≥8 hours'
    },
    'Admission': {
        'path': os.path.join(OUTPUT_PATH, 'task2_admission', 'all_predictions.csv'),
        'label': 'Hospital Admission'
    },
    'Hosp_LOS_Over7d': {
        'path': os.path.join(OUTPUT_PATH, 'task3_hosp_los', 'all_predictions.csv'),
        'label': 'Hosp-LOS >7 days'
    }
}

print("=" * 60)
print("📊 Generating Publication Figures and Tables for ALL 3 Tasks")
print("=" * 60)

# Store data for combined plots
tasks_data = {}

for task_key, task_info in tasks.items():
    task_path = task_info['path']
    task_label = task_info['label']
    
    # Check if predictions file exists
    if not os.path.exists(task_path):
        print(f"⚠️ Warning: {task_path} not found. Skipping {task_label}")
        continue
    
    print(f"\n{'='*60}")
    print(f"📊 Processing: {task_label}")
    print(f"{'='*60}")
    
    # Load predictions
    predictions_df = pd.read_csv(task_path)
    
    # Extract y_true and model predictions
    y_test = predictions_df['y_true'].values
    
    # Check if predictions DataFrame has the expected columns
    available_models = [col for col in predictions_df.columns if col != 'y_true']
    
    model_predictions = {}
    for model in available_models:
        if 'Voting' not in model:
            model_predictions[model] = predictions_df[model].values
    
    print(f"Test set size: {len(y_test)}")
    print(f"Prevalence: {np.mean(y_test):.3f}")
    print(f"Models: {list(model_predictions.keys())}")
    
    # Store for combined plots
    tasks_data[task_key] = {
        'y_true': y_test,
        'model_predictions': model_predictions,
        'label': task_label
    }
    
    # Generate individual table
    df_metrics = generate_tables(y_test, model_predictions, task_key)
    
    # Print summary for this task
    print(f"\n📊 {task_label} - Model Performance Summary:")
    print(df_metrics.to_string(index=False))

# =============================================================================
# GENERATE COMBINED FIGURES
# =============================================================================
print("\n" + "=" * 60)
print("📊 Generating Combined Figures")
print("=" * 60)

# Generate combined DCA plot
plot_dca_all_tasks(tasks_data)

# Generate combined ROC + Calibration plot
plot_roc_and_calibration_all_tasks(tasks_data)

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("✅ ALL FIGURES AND TABLES GENERATED SUCCESSFULLY!")
print("=" * 60)
print(f"📁 Output directory: {OUTPUT_DIR}")
print("\nGenerated files:")

print("\n  Combined Figures (All 3 Tasks):")
print("    - Fig1_DCA_All_Tasks.png (3 subplots - one per task)")
print("    - Fig2_ROC_Calibration_All_Tasks.png (2x3 subplots - ROC top, Calibration bottom)")

print("\n  Individual Tables (CSV):")
for task_key in tasks.keys():
    print(f"    - Table_{task_key}_Performance.csv")

print("\n" + "=" * 60)

📊 Generating Publication Figures and Tables for ALL 3 Tasks

📊 Processing: ED-LOS ≥8 hours
Test set size: 12020
Prevalence: 0.223
Models: ['XGBoost', 'LightGBM', 'RandomForest', 'TabNet']
✅ Table saved for ED_LOS_Over8h

📊 ED-LOS ≥8 hours - Model Performance Summary:
       Model AUROC Sensitivity Specificity Accuracy Brier Score
     XGBoost 0.659       0.528       0.677    0.644       0.209
    LightGBM 0.648       0.000       1.000    0.777       0.170
RandomForest 0.660       0.423       0.759    0.684       0.199
      TabNet 0.638       0.630       0.571    0.584       0.226

📊 Processing: Hospital Admission
Test set size: 12020
Prevalence: 0.396
Models: ['XGBoost', 'LightGBM', 'RandomForest', 'TabNet']
✅ Table saved for Admission

📊 Hospital Admission - Model Performance Summary:
       Model AUROC Sensitivity Specificity Accuracy Brier Score
     XGBoost 0.816       0.699       0.763    0.738       0.172
    LightGBM 0.816       0.704       0.764    0.740       0.172
RandomFore

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import os

# =============================================================================
# LOAD DATA
# =============================================================================

OUTPUT_PATH = r"E:\\TSINGHUA\\thisis\\Paper\\output_paper"

# Load test features and targets
X_test = pd.read_pickle(os.path.join(OUTPUT_PATH, 'X_test.pkl'))
y_test_full = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_admit.pkl'))

# Extract ED-LOS target (for Task 1)
y_test_ed_los = y_test_full['TARGET_ED_LOS_over8h'].values

# Load test raw data to get ESI triage acuity
test_raw = pd.read_pickle(os.path.join(OUTPUT_PATH, 'test_data', 'visits.pkl'))

# =============================================================================
# CREATE ESI PREDICTED RISK - WITH NaN HANDLING
# =============================================================================

test_df = test_raw.copy()

# Extract numeric ESI level from Triage_acuity column
def extract_esi_level(acuity_str):
    if pd.isna(acuity_str) or acuity_str == '':
        return np.nan
    # Extract first character (e.g., "3-Urgent" -> 3)
    try:
        return int(str(acuity_str)[0])
    except:
        return np.nan

test_df['esi_level'] = test_df['Triage_acuity'].apply(extract_esi_level)

# Invert ESI so 1 becomes highest risk (5), 5 becomes lowest risk (1)
test_df['esi_predicted_risk'] = 6 - test_df['esi_level']

# =============================================================================
# HANDLE MISSING VALUES - OPTION 1: DROP ROWS WITH NaN
# =============================================================================

# Create a clean dataset without NaN values
clean_mask = ~(test_df['esi_predicted_risk'].isna() | pd.isna(y_test_ed_los))
y_test_clean = y_test_ed_los[clean_mask]
esi_risk_clean = test_df.loc[clean_mask, 'esi_predicted_risk'].values

# Calculate AUC on clean data
esi_auc_ed_los = roc_auc_score(y_test_clean, esi_risk_clean)
print(f"ESI AUC for ED-LOS ≥8h prediction: {esi_auc_ed_los:.4f}")
print(f"Removed {sum(~clean_mask)} rows with missing ESI values")

# =============================================================================
# HANDLE MISSING VALUES - OPTION 2: IMPUTE WITH MEDIAN
# =============================================================================

# Alternatively, impute missing values with median
median_esi = test_df['esi_predicted_risk'].median()
test_df['esi_predicted_risk_imputed'] = test_df['esi_predicted_risk'].fillna(median_esi)

esi_auc_ed_los_imputed = roc_auc_score(y_test_ed_los, test_df['esi_predicted_risk_imputed'])
print(f"ESI AUC (median imputed) for ED-LOS ≥8h: {esi_auc_ed_los_imputed:.4f}")

# =============================================================================
# HANDLE MISSING VALUES - OPTION 3: USE MODE (most common ESI level)
# =============================================================================

mode_esi = test_df['esi_predicted_risk'].mode()[0]
test_df['esi_predicted_risk_mode'] = test_df['esi_predicted_risk'].fillna(mode_esi)

esi_auc_ed_los_mode = roc_auc_score(y_test_ed_los, test_df['esi_predicted_risk_mode'])
print(f"ESI AUC (mode imputed) for ED-LOS ≥8h: {esi_auc_ed_los_mode:.4f}")

# =============================================================================
# CALCULATE FOR ALL THREE TASKS (using drop method)
# =============================================================================

print("\n" + "=" * 60)
print("📊 ESI AUC for All Tasks (drop NaN method)")
print("=" * 60)

# Task 1: ED-LOS
esi_auc_ed_los = roc_auc_score(y_test_clean, esi_risk_clean)
print(f"Task 1 - ED-LOS ≥8h:              {esi_auc_ed_los:.4f}")

# Task 2: Hospital Admission
if 'TARGET_Admitted' in y_test_full.columns:
    y_test_admission = y_test_full['TARGET_Admitted'].values
    clean_mask_admit = ~(test_df['esi_predicted_risk'].isna() | pd.isna(y_test_admission))
    y_test_admit_clean = y_test_admission[clean_mask_admit]
    esi_risk_admit_clean = test_df.loc[clean_mask_admit, 'esi_predicted_risk'].values
    esi_auc_admission = roc_auc_score(y_test_admit_clean, esi_risk_admit_clean)
    print(f"Task 2 - Hospital Admission:       {esi_auc_admission:.4f}")

# Task 3: Hosp-LOS >7 days
try:
    y_test_hosp = pd.read_pickle(os.path.join(OUTPUT_PATH, 'y_test_hosp.pkl'))
    y_test_hosp_los = y_test_hosp['TARGET_Hosp_LOS_over7d'].values
    
    # Clean mask for Hosp-LOS (need to match indices)
    clean_mask_hosp = ~(test_df['esi_predicted_risk'].isna() | pd.isna(y_test_hosp_los))
    y_test_hosp_clean = y_test_hosp_los[clean_mask_hosp]
    esi_risk_hosp_clean = test_df.loc[clean_mask_hosp, 'esi_predicted_risk'].values
    
    esi_auc_hosp_los = roc_auc_score(y_test_hosp_clean, esi_risk_hosp_clean)
    print(f"Task 3 - Hosp-LOS >7d:            {esi_auc_hosp_los:.4f}")
except Exception as e:
    print(f"Task 3 - Hosp-LOS: Not available ({e})")

print("=" * 60)

# =============================================================================
# COMPARE ESI TO YOUR MODELS
# =============================================================================

# Load your model predictions for comparison
try:
    predictions_df = pd.read_csv(os.path.join(OUTPUT_PATH, 'task1_ed_los', 'all_predictions.csv'))
    
    print("\n" + "=" * 60)
    print("📊 ESI vs Model Performance Comparison (ED-LOS ≥8h)")
    print("=" * 60)
    print(f"ESI (Triage Acuity):              {esi_auc_ed_los:.4f}")
    
    for model in ['XGBoost', 'LightGBM', 'RandomForest', 'TabNet']:
        if model in predictions_df.columns:
            model_auc = roc_auc_score(y_test_clean, predictions_df[model].values[clean_mask])
            print(f"{model}:                          {model_auc:.4f}")
    
    print("=" * 60)
except Exception as e:
    print(f"Model predictions not found for comparison: {e}")

# =============================================================================
# ADDITIONAL: ESI PERFORMANCE BREAKDOWN BY LEVEL
# =============================================================================

print("\n" + "=" * 60)
print("📊 ESI Level Distribution and Performance")
print("=" * 60)

# Show distribution of ESI levels
esi_dist = test_df['esi_level'].value_counts().sort_index()
print("\nESI Level Distribution:")
for level, count in esi_dist.items():
    if not pd.isna(level):
        print(f"  ESI {level:.0f}: {count} patients ({count/len(test_df)*100:.1f}%)")

# Show performance by ESI level
print("\nED-LOS ≥8h Rate by ESI Level:")
for level in sorted(test_df['esi_level'].dropna().unique()):
    if not pd.isna(level):
        mask = test_df['esi_level'] == level
        ed_los_rate = y_test_ed_los[mask].mean() if mask.sum() > 0 else np.nan
        print(f"  ESI {level:.0f}: {ed_los_rate:.2%} ({mask.sum():.0f} patients)")

print("=" * 60)

# =============================================================================
# SAVE ESI PREDICTIONS FOR FUTURE USE
# =============================================================================

# Save ESI predictions to CSV
esi_df = pd.DataFrame({
    'esi_level': test_df['esi_level'],
    'esi_predicted_risk': test_df['esi_predicted_risk'],
    'esi_predicted_risk_imputed': test_df['esi_predicted_risk_imputed']
})
esi_df.to_csv(os.path.join(OUTPUT_PATH, 'esi_predictions.csv'), index=False)
print(f"\n✅ ESI predictions saved to {OUTPUT_PATH}/esi_predictions.csv")

ESI AUC for ED-LOS ≥8h prediction: 0.5571
Removed 33 rows with missing ESI values
ESI AUC (median imputed) for ED-LOS ≥8h: 0.5572
ESI AUC (mode imputed) for ED-LOS ≥8h: 0.5572

📊 ESI AUC for All Tasks (drop NaN method)
Task 1 - ED-LOS ≥8h:              0.5571
Task 2 - Hospital Admission:       0.6786
Task 3 - Hosp-LOS: Not available (operands could not be broadcast together with shapes (12020,) (4765,) )

📊 ESI vs Model Performance Comparison (ED-LOS ≥8h)
ESI (Triage Acuity):              0.5571
XGBoost:                          0.6588
LightGBM:                          0.6477
RandomForest:                          0.6601
TabNet:                          0.6385

📊 ESI Level Distribution and Performance

ESI Level Distribution:
  ESI 1: 95 patients (0.8%)
  ESI 2: 3180 patients (26.5%)
  ESI 3: 7477 patients (62.2%)
  ESI 4: 1177 patients (9.8%)
  ESI 5: 58 patients (0.5%)

ED-LOS ≥8h Rate by ESI Level:
  ESI 1: 9.47% (95 patients)
  ESI 2: 25.94% (3180 patients)
  ESI 3: 23.70% (7477 p